# BigQuery Conversational Analytics (BQCA) Prompt & Response Logging
### Customer Analytics, FinOps & Evaluation Starter Notebook (`bigquery-agent-analytics` v0.5.4)

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/blob/main/examples/bqca_prompt_response_logging_customer_notebook.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/colab-logo.png" alt="Colab logo"> Run in Colab
    </a>
  </td>
  <td>
    <a href="https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/blob/main/examples/bqca_prompt_response_logging_customer_notebook.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/github-logo.png" width="32" alt="GitHub logo"> View Notebook on GitHub
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/main/examples/bqca_prompt_response_logging_customer_notebook.ipynb">
      <img src="https://www.gstatic.com/images/branding/product/1x/google_cloud_48dp.png" alt="Vertex AI logo" width="32"> Open in Vertex AI Workbench
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/blob/main/examples/bqca_prompt_response_logging_customer_notebook.ipynb">
      <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTW1gvOovVlbZAIZylUtf5Iu8-693qS1w5NJw&s" alt="BQ logo" width="35"> Open in BQ Studio
    </a>
  </td>
</table>

<br clear="all"/>

When you enable **Prompt and Response Logging** *(Preview)* in **[BigQuery Conversational Analytics (BQCA)](https://cloud.google.com/bigquery/docs/conversational-analytics)**, BigQuery streams conversation events into an `agent_events` table in your dataset using the 16 required columns of the open-source [BigQuery Agent Analytics SDK](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK) (`bigquery-agent-analytics`) plus a BQCA `event_id` column (`17` columns total).

This starter notebook is a **copy-and-run production playbook** for BQCA customers. Point the configuration parameters in **Section 1.1** at your own `PROJECT_ID`, `DATASET_ID`, `RAW_TABLE_ID` (`agent_events`), and `VERTEX_CONNECTION_ID`, and run all cells from top to bottom (this three-parameter quick start applies to the default **US multi-region** setup; EU and single-region datasets also need the `LOCATION` / connection — and, if you pin direct `google-genai` calls to a single region, `VERTEX_LOCATION` / model — edits described in the **Region guide** at the top of Cell 1.1). For IAM setup, Vertex AI Cloud Resource Connection commands, and a 12-issue troubleshooting runbook, see the companion **[Customer Self-Hosting & Self-Debugging Manual](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/blob/main/docs/guides/bqca-prompt-response-logging-manual.md)** (`docs/guides/bqca-prompt-response-logging-manual.md`). Any sample numbers, session IDs, token totals or runtimes quoted in the manual are **illustrative** values from a synthetic 25-session reference dataset, not outputs stored in this committed notebook (its cells ship with cleared outputs); your results will differ by project, dataset and model.

**What this notebook sets up and runs for you**

| Section | Customer workflow | What it provisions or runs | BQCA events used |
|---|---|---|---|
| **1** | **Setup, Compatibility Views & `doctor()`** | Installs/upgrades the SDK (`>=0.5.4`) if needed, creates two zero-copy BigQuery views (`v_bqca_customer_sdk_events` for the SDK and `v_bqca_customer_turns` for SQL/BI dashboards), runs `client.doctor()`, and pings `AI.GENERATE` (`gemini-3.5-flash`). | All 9 BQCA event types |
| **2** | **Prompt & Response Conversation Explorer** | Summarises conversation volume, answer rates, clarifying questions, fast-path hits and latency by agent and user cohort, and previews paired prompt/response turns. | `USER_MESSAGE_RECEIVED`, `AGENT_RESPONSE`, `INVOCATION_COMPLETED`, `LLM_RESPONSE` |
| **3** | **Customer Sentiment & Frustrated-Turn Triage** | Scores up to `MAX_EVAL_SESSIONS` sessions with `LLMAsJudge.sentiment()` and classifies user tone with `evaluate_categorical()` (`Positive_Satisfied`, `Neutral_Analytical`, `Clarification_Needed`, `Frustrated_Impatient`), joining scores back to the exact prompts and answers. | `USER_MESSAGE_RECEIVED`, `AGENT_RESPONSE` |
| **4** | **User Question Categorisation & Topic Taxonomy** | Classifies user questions into your business topic taxonomy using both bulk SQL classification (`AI.CLASSIFY`) and structured rationale mode (`AI.GENERATE`). | `USER_MESSAGE_RECEIVED` |
| **5** | **Semantic Drift & Golden Question Bank Coverage** | Bootstraps or loads `bqca_golden_qa`, runs `client.drift_detection()`, and runs a production-centric `AI.EMBED` + `ML.DISTANCE` query to find which user questions are outside your Golden Question Bank. | `USER_MESSAGE_RECEIVED` + Golden Q&A table |
| **6** | **Response-Quality Evaluation (Golden Q&A Key & SDK Judges)** | Grades in-domain answers against Golden Q&A pairs via `AI.GENERATE` (reusing Section 5.2's matched pairs with zero duplicate embedding calls), and runs `LLMAsJudge.correctness()`, `LLMAsJudge.hallucination()`, `GraderPipeline`, and multi-trial judge consistency (`TrialRunner` + `BQCAPerformanceEvaluator`). | `USER_MESSAGE_RECEIVED`, `AGENT_RESPONSE` + Golden Q&A table |
| **7** | **FinOps, Latency SLOs, Thinking Tokens & Fast-Path ROI** | Evaluates session SLO budgets (`SystemEvaluator`), accounts for Gemini thinking tokens (`thoughts_token_count`), and compares Verified Queries fast-path turns (`attributes.fast_path = true`) against standard NL2SQL turns. | `LLM_RESPONSE`, `INVOCATION_COMPLETED`, `AGENT_RESPONSE` |
| **8** | **Operational Health Monitor & Executive Summary** | Monitors model cut-offs (`finish_reason != 'STOP'`), sanitised `*_ERROR` rows (`bqaa.ERROR_SQL_PREDICATE`), and polite out-of-scope refusals, then prints a live **Customer Production Health Summary Table**. | `LLM_RESPONSE`, `INVOCATION_ERROR`, `AGENT_ERROR`, `LLM_ERROR`, `AGENT_RESPONSE` |

> **What BQCA logs (observed Preview schema):** BigQuery Conversational Analytics Prompt & Response Logging *(Preview)* emits 9 event types (`INVOCATION_STARTING`, `USER_MESSAGE_RECEIVED`, `AGENT_RESPONSE`, `INVOCATION_COMPLETED`, `LLM_RESPONSE`, `EMBEDDING_SUGGESTION`, `INVOCATION_ERROR`, `AGENT_ERROR`, `LLM_ERROR`) across the 16 required SDK columns plus `event_id` (`17` columns total; `v_bqca_customer_sdk_events` also works on 16-column tables without `event_id`), and omits low-level ADK tool/agent spans (`AGENT_STARTING`, `AGENT_COMPLETED`, `LLM_REQUEST`, `TOOL_STARTING`, `TOOL_COMPLETED`, `TOOL_ERROR`). Everything in this notebook is designed for that 9-event structure.

> **Costs, quotas & resource cleanup:**
> - **Billed services:** Running this notebook ("Runtime -> Run all") executes billable **BigQuery SQL queries** (scoped to `EVAL_LOOKBACK_DAYS`, default `90` days), **BigQuery ML remote model calls** (`AI.GENERATE`, `AI.CLASSIFY`, and `AI.EMBED` billed to Vertex AI via `VERTEX_CONNECTION_ID`), and **direct Vertex AI Gemini calls** (`google-genai` in Section 6B).
> - **Approximate call volume per full run:** At the default `MAX_EVAL_SESSIONS = 100` and `MAX_GOLDEN_QUESTIONS = 200`, one full run executes `1` connection ping (Section 1.4), **`7`** batch `AI.GENERATE` / `AI.CLASSIFY` queries over `<= 100` sessions each (Sections 3.1, 3.2, 4.1×2, 6B×2, and 8.2; if a batch `AI.GENERATE` query fails at the job level, `client.evaluate` / `evaluate_categorical` can fall back to per-session direct Gemini calls via `execution_mode = "api_fallback"`), `1` `AI.GENERATE` answer-key grading query over `<= 100` in-domain sessions (Section 6A), `2` `AI.EMBED` queries (Section 5.1 SDK `client.drift_detection()` over `<= 100` prompts + all rows in `bqca_golden_qa`, and Section 5.2 SQL over `<= 100` single-turn prompts + `<= MAX_GOLDEN_QUESTIONS` (`200`) golden questions — `<= 50` on a freshly bootstrapped golden table; Section 6A reuses Section 5.2's matched pairs with zero extra embedding calls), and **`10–16`** direct Gemini judge calls in Section 6B (`2` target sessions × `2` `GraderPipeline` calls + `3` trials × `1–2` `TrialRunner` rubric calls). For a smaller, lower-cost first test run, set `MAX_EVAL_SESSIONS = 25` and `EVAL_LOOKBACK_DAYS = 7` in Cell `1.1`.
> - **Quotas & `429` retries:** Calls are subject to your project's Vertex AI API requests-per-minute (QPM) quota for `JUDGE_MODEL` (`gemini-3.5-flash`) and `EMBEDDING_MODEL` (`text-embedding-005`). The notebook's `run_sql()` helper automatically retries job-level `429 TooManyRequests`, `500`, and `503` errors with backoff, while per-row `AI.GENERATE` / `AI.EMBED` quota errors are reported in the `.status` field (`ERROR` in Section 6A).
> - **Timeouts & cancellation:** `AI_SQL_TIMEOUT_SEC` (default `180`) is a **client-side wait timeout**, not a spend cap. When an in-warehouse `AI.*` query is still running after that many seconds, the notebook stops waiting, requests a best-effort `job.cancel()`, and the judge/categorical/grading cells fall back to direct Vertex AI. BigQuery cancels asynchronously, so work done before the cancel is still billed and the fallback's Vertex AI calls are billed on top; if the cancel request cannot be confirmed, the notebook prints a `WARNING` with the BigQuery `job_id` and a `bq cancel` command. Failed or empty direct judge calls are reported as **unevaluated**, never as a measured `0.0` score.
> - **Cleanup SQL:** To remove the two views (and, if you have not added SME-curated `verified = TRUE` rows, the Golden Q&A table) created by this notebook:
>   ```sql
>   DROP VIEW IF EXISTS `your-gcp-project-id.your_bqca_logs_dataset.v_bqca_customer_sdk_events`;
>   DROP VIEW IF EXISTS `your-gcp-project-id.your_bqca_logs_dataset.v_bqca_customer_turns`;
>   -- CAUTION: Only run DROP TABLE below if you have NOT curated SME-verified (verified = TRUE) rows in bqca_golden_qa:
>   -- DROP TABLE IF EXISTS `your-gcp-project-id.your_bqca_logs_dataset.bqca_golden_qa`;
>   ```

> **Data governance, residency & privacy note:** Raw user prompts (`user_prompt`) and agent answers (`agent_response`), as well as `persona` cohort labels (which fall back to the `user_id` email prefix when `custom_labels.persona` is absent; set `ANONYMIZE_USER_ID = True` in Cell `1.1` to SHA-256 pseudonymize `user_id`, `user_id`-derived cohort fallback labels, and email-like explicit `persona` labels as `user_<8hex>` — a 32-bit display pseudonym with ~39% collision probability at 65,536 distinct users and ~50% around ~77,000 distinct users; masking covers the turns view and Section 8.1 cohort labels only, while prompts, answers, other explicit persona labels, and the compatibility view still carry raw text), may contain sensitive business questions or PII. Keep `v_bqca_customer_sdk_events` and `v_bqca_customer_turns` in the same BigQuery dataset as `agent_events` so dataset-level IAM policies and VPC Service Controls perimeter rules apply uniformly, and base-table Row-Level Security (RLS) and Column-Level Security (CLS / policy tags) on `agent_events` remain enforced when querying those views. **Important (`bqca_golden_qa` CTAS caveat):** Because `bqca_golden_qa` is a derived table materialized via `CREATE TABLE IF NOT EXISTS ... AS SELECT` (Cell `5.1`), BigQuery does **not** automatically copy source-table row access policies or column policy tags onto `bqca_golden_qa`; Cell `5.1` inspects `RAW_TABLE_ID` via the BigQuery REST `tables.rowAccessPolicies.list` endpoint and recursive `SchemaField.policy_tags` and **fails closed** by default if any row access policy or column policy tag is present **or if the caller lacks `bigquery.rowAccessPolicies.list` to verify row access policies** (pre-create `bqca_golden_qa` in `DATASET_ID` with matching row access policies / policy tags, or set `BQCA_AUTO_SEED_GOLDEN_TABLE=false` to disable automatic CTAS materialization; only set `BQCA_AUTO_SEED_GOLDEN_TABLE=force` in an isolated scratch dataset). Treat exported notebooks/HTML reports with the same access controls. Note that default `VERTEX_LOCATION = "global"` routes direct Vertex AI judge calls in Section 6B to the global endpoint without pinning a processing geography; for US or EU multi-region data residency with Gemini 3.x models (`gemini-3.5-flash`), keep your BigQuery dataset in `US` or `EU` and set `VERTEX_LOCATION = "us"` or `"eu"` (an EU dataset also needs `LOCATION = "EU"` and a connection created in the EU. For a single-region dataset such as `"us-central1"`, set `LOCATION = "us-central1"` and use a connection created in `"us-central1"`: in-warehouse BigQuery `AI.GENERATE` / `AI.CLASSIFY` automatically routes a short `gemini-3.5-flash` model name from US single regions to the `us` multi-region endpoint, from eligible EU single regions to the `eu` multi-region endpoint, and from other locations to `global` — with a fully qualified `global` endpoint URL required in `asia-south1`; see [BigQuery generative AI locations](https://cloud.google.com/bigquery/docs/generative-ai-overview#locations). Direct `google-genai` calls in Section 6B and `api_fallback` target `VERTEX_LOCATION` directly without BigQuery's multi-region routing, so either keep `VERTEX_LOCATION = "global"` / `"us"` / `"eu"` with `DIRECT_JUDGE_MODEL = "gemini-3.5-flash"`, or if you set `VERTEX_LOCATION` to a single region such as `"us-central1"`, switch `DIRECT_JUDGE_MODEL` — and optionally `JUDGE_MODEL` / `CATEGORICAL_MODEL` if you require single-region warehouse inference — to `"gemini-2.5-flash"`. See the **Region guide** in Cell 1.1 and [Vertex AI generative AI locations](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/locations)).

## 1. Customer configuration, compatibility views and `client.doctor()`

### 1.0 Install & authenticate

Run the first cell to install or upgrade to `bigquery-agent-analytics[llm]==0.5.4` if it is not already installed in your Python kernel (Google Colab, BigQuery Studio, Vertex AI Colab Enterprise, Vertex AI Workbench, or local JupyterLab) and authenticate when inside Google Colab.

In [ ]:
#@title 1.0 Install & Authenticate
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from importlib.metadata import PackageNotFoundError, version as _pkg_version
import subprocess
import sys


def _needs_sdk_install():
    try:
        parts = [int(x) for x in _pkg_version("bigquery-agent-analytics").split(".")[:3] if x.isdigit()]
        return tuple(parts) < (0, 5, 4)
    except PackageNotFoundError:
        return True


if _needs_sdk_install():
    def _pip(*args):
        return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args]).returncode

    if _pip("bigquery-agent-analytics[llm]==0.5.4") != 0:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "bigquery-agent-analytics[llm] @ git+https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK.git@v0.5.4",
            ],
            check=True,
        )
    print("Installed bigquery-agent-analytics 0.5.4. If prompted to restart the runtime, restart and re-run from the top.")
else:
    print(f"bigquery-agent-analytics {_pkg_version('bigquery-agent-analytics')} is already installed in this runtime.")

if "google.colab" in sys.modules:
    try:
        from google.colab import auth

        auth.authenticate_user()
        print("Authenticated Google Colab user.")
    except Exception as exc:
        print(f"Colab interactive auth skipped (using runtime credentials): {exc}")
else:
    print("Using active runtime Application Default Credentials (ADC).")

### 1.1 Set your project, dataset, and table parameters

Update `PROJECT_ID`, `DATASET_ID`, and `VERTEX_CONNECTION_ID` below (or set `BQCA_PROJECT_ID`, `BQCA_DATASET_ID`, `BQCA_RAW_TABLE_ID`, `BQCA_LOCATION`, `BQCA_VERTEX_LOCATION`, and `BQCA_VERTEX_CONNECTION_ID` as environment variables for headless scheduled runs). Leave `RAW_TABLE_ID = "agent_events"` when running on the default table created by BQCA Prompt & Response Logging. By default, this notebook uses **`gemini-3.5-flash`** as the BigQuery ML `AI.GENERATE` / `AI.CLASSIFY` and direct Vertex AI judge endpoint (with `VERTEX_LOCATION = "global"`, the global Vertex AI endpoint serving Gemini 3.x models).

In [ ]:
#@title 1.1 Customer Configuration Parameters
from importlib.metadata import version as _dist_version
import logging
import os
import re
import sys
import time
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module=r"google\..*")
for _noisy in ("google_genai", "google_genai.models", "google.genai", "absl", "urllib3", "httpx"):
    logging.getLogger(_noisy).setLevel(logging.ERROR)
_bqaa_logger = logging.getLogger("bigquery_agent_analytics")
_bqaa_logger.handlers = [logging.StreamHandler(sys.stdout)]
_bqaa_logger.setLevel(logging.WARNING)
_bqaa_logger.propagate = False

# ---- Customer Configuration Parameters (edit in Colab form, inline, or via selected BQCA_* env vars — see below) ----
# Region guide: distinguish the BigQuery query/connection location (LOCATION & VERTEX_CONNECTION_ID) from the Vertex AI inference endpoint:
#   * US multi-region (default, 3-edit quick start): LOCATION = "US", VERTEX_LOCATION = "global" (or "us" for US residency), connection "<project>.us.<name>".
#   * EU multi-region: ALSO set LOCATION = "EU" and use a connection created in the EU ("<project>.eu.<name>"). Keep
#     VERTEX_LOCATION = "global", or set "eu" to keep direct judge calls on the EU multi-region endpoint.
#   * Single region (for example us-central1): set LOCATION = "us-central1" and use a connection in that region ("<project>.us-central1.<name>").
#     - In-warehouse AI.GENERATE / AI.CLASSIFY (JUDGE_MODEL, CATEGORICAL_MODEL): BigQuery runs the query job in LOCATION and
#       automatically routes short gemini-3.5-flash model names from US single regions (e.g. us-central1) to the "us" multi-region
#       endpoint, from eligible EU single regions to "eu", and from other locations to "global" (in asia-south1, pass a fully
#       qualified global endpoint URL; see https://cloud.google.com/bigquery/docs/generative-ai-overview#locations). Only switch
#       JUDGE_MODEL / CATEGORICAL_MODEL to "gemini-2.5-flash" if your policy requires single-region model inference inside that
#       specific region rather than BigQuery's automatic us/eu/global multi-region routing.
#     - Direct google-genai calls (VERTEX_LOCATION & DIRECT_JUDGE_MODEL in Section 6B and api_fallback): direct SDK calls hit
#       VERTEX_LOCATION without BigQuery's multi-region routing, and gemini-3.5-flash is served only on "global", "us", and "eu".
#       Either keep VERTEX_LOCATION = "global" (or "us" / "eu") with DIRECT_JUDGE_MODEL = "gemini-3.5-flash", OR if you set
#       VERTEX_LOCATION to a single region (e.g. "us-central1"), set DIRECT_JUDGE_MODEL = "gemini-2.5-flash".
PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
DATASET_ID = "your_bqca_logs_dataset"  # @param {type:"string"}
RAW_TABLE_ID = "agent_events"  # @param {type:"string"}
COMPAT_VIEW_ID = "v_bqca_customer_sdk_events"  # @param {type:"string"}
TURNS_VIEW_ID = "v_bqca_customer_turns"  # @param {type:"string"}
GOLDEN_QA_TABLE_ID = "bqca_golden_qa"  # @param {type:"string"}
# BigQuery dataset location (must match your dataset and the connection's location; see the Region guide above):
LOCATION = "US"  # @param {type:"string"}
# Vertex AI endpoint location for direct google-genai calls ("global", or "us" / "eu" multi-region for US/EU residency with gemini-3.x):
VERTEX_LOCATION = "global"  # @param {type:"string"}
# Standard BigQuery connection format PROJECT_ID.LOCATION.CONNECTION_ID (LOCATION = your dataset's location, e.g. us, eu or us-central1):
VERTEX_CONNECTION_ID = "your-gcp-project-id.us.bqca_vertex_connection"  # @param {type:"string"}
JUDGE_MODEL = "gemini-3.5-flash"  # @param {type:"string"}
CATEGORICAL_MODEL = "gemini-3.5-flash"  # @param {type:"string"}
EMBEDDING_MODEL = "text-embedding-005"  # @param {type:"string"}
DIRECT_JUDGE_MODEL = "gemini-3.5-flash"  # @param {type:"string"}
JUDGE_THRESHOLD = 0.6  # @param {type:"number"}
# Cosine distance threshold for Section 5.2 production session semantic drift flagging (question_distance > DRIFT_THRESHOLD)
# and Section 6A in-domain golden answer pairing (GOLDEN_MATCH_MAX_DISTANCE = DRIFT_THRESHOLD). Note: Section 5.1 headline
# Golden Question Bank coverage (client.drift_detection()) uses the SDK's built-in similarity_threshold = 0.30 (SDK_DRIFT_THRESHOLD).
DRIFT_THRESHOLD = 0.15  # @param {type:"number"}
EVAL_LOOKBACK_DAYS = 90  # @param {type:"integer"}
MAX_EVAL_SESSIONS = 100  # @param {type:"integer"}
MAX_GOLDEN_QUESTIONS = 200  # @param {type:"integer"}
ANONYMIZE_USER_ID = False  # @param {type:"boolean"}
# Conservative per-query scan cap (bytes billed, 1 GB default) applied to direct notebook SQL
# queries (run_sql(), the Cell 5.1 CTAS bootstrap, and Cell 6B SESSION_SUMMARY_QUERY; SDK
# evaluators manage their own query jobs): a query estimated above this cap fails instead of
# scanning, protecting production-scale agent_events tables from an accidental full scan.
MAX_BYTES_BILLED = 1000000000  # @param {type:"integer"}
# Client-side wait timeout (seconds, 180s default; or set BQCA_AI_SQL_TIMEOUT_SEC) applied per in-warehouse BigQuery
# AI.GENERATE / AI.CLASSIFY / AI.EMBED / ML.GENERATE_TEXT / ML.GENERATE_EMBEDDING SQL query. When a warehouse AI query is
# still running after this many seconds, the notebook stops waiting and requests a best-effort job.cancel(). Note on SDK
# fallback tiers: in client.evaluate(LLMAsJudge.*), if AI.GENERATE fails or times out, the SDK first checks for a legacy BQML
# model `<project>.<dataset>.gemini_text_model` (ML.GENERATE_TEXT, execution_mode='ml_generate_text') before falling back to
# direct Vertex AI (execution_mode='api_fallback'); in client.evaluate_categorical(..., include_justification=False), if
# AI.CLASSIFY fails or times out, the SDK tries in-warehouse AI.GENERATE (execution_mode='ai_generate') before falling
# back to direct Vertex AI (execution_mode='api_fallback'; categorical evaluation does not use ML.GENERATE_TEXT, so at
# most 2 warehouse queries run before api_fallback — AI.CLASSIFY -> AI.GENERATE in evaluate_categorical(), or AI.GENERATE -> legacy ML.GENERATE_TEXT per criterion in evaluate()). If an intermediate warehouse tier
# succeeds, execution finishes in that warehouse mode; if multiple warehouse tiers stall and time out sequentially, each
# attempted warehouse query can wait up to AI_SQL_TIMEOUT_SEC before reaching api_fallback. This is NOT a spend cap: BigQuery
# cancellation is asynchronous, work done before the cancel is still billed, and any fallback calls are billed on top. If the
# cancel request cannot be confirmed, the notebook prints a WARNING with the BigQuery job_id and a `bq cancel` command.
AI_SQL_TIMEOUT_SEC = 180  # @param {type:"integer"}

# Illustrative placeholder token rates (USD per 1K tokens; not Gemini Flash list prices — replace with your organization's rate card):
COST_RATES = {"input_per_1k": 0.00125, "output_per_1k": 0.01}

# Support environment variable overrides for headless CI/CD or Cloud Run Jobs:
PROJECT_ID = os.environ.get("BQCA_PROJECT_ID", PROJECT_ID)
DATASET_ID = os.environ.get("BQCA_DATASET_ID", DATASET_ID)
RAW_TABLE_ID = os.environ.get("BQCA_RAW_TABLE_ID", RAW_TABLE_ID)
LOCATION = os.environ.get("BQCA_LOCATION", LOCATION)
VERTEX_LOCATION = os.environ.get("BQCA_VERTEX_LOCATION", VERTEX_LOCATION)
VERTEX_CONNECTION_ID = os.environ.get("BQCA_VERTEX_CONNECTION_ID", VERTEX_CONNECTION_ID)
if "BQCA_ANONYMIZE_USER_ID" in os.environ:
    ANONYMIZE_USER_ID = os.environ["BQCA_ANONYMIZE_USER_ID"].strip().lower() in ("1", "true", "yes")
if "BQCA_MAX_BYTES_BILLED" in os.environ:
    MAX_BYTES_BILLED = int(os.environ["BQCA_MAX_BYTES_BILLED"])
if "BQCA_AI_SQL_TIMEOUT_SEC" in os.environ:
    AI_SQL_TIMEOUT_SEC = int(os.environ["BQCA_AI_SQL_TIMEOUT_SEC"])

if (
    PROJECT_ID == "your-gcp-project-id"
    or DATASET_ID == "your_bqca_logs_dataset"
    or "your-gcp-project-id" in VERTEX_CONNECTION_ID
):
    raise ValueError(
        "Stop! Please update PROJECT_ID, DATASET_ID, and VERTEX_CONNECTION_ID in Cell 1.1 "
        "(or set BQCA_PROJECT_ID, BQCA_DATASET_ID, and BQCA_VERTEX_CONNECTION_ID) before running the notebook. "
        "See Section 2 of docs/guides/bqca-prompt-response-logging-manual.md for setup commands."
    )

# Configure google-genai for direct Vertex AI judge calls (GraderPipeline and TrialRunner).
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = VERTEX_LOCATION

from google.api_core import exceptions as gcp_exceptions
from google.cloud import bigquery
from IPython.display import Markdown, display
import pandas as pd

import bigquery_agent_analytics as bqaa
from bigquery_agent_analytics import (
    CategoricalEvaluationConfig,
    CategoricalMetricCategory,
    CategoricalMetricDefinition,
    Client,
    GraderPipeline,
    LLMAsJudge,
    PerformanceEvaluator,
    SystemEvaluator,
    TraceFilter,
    TrialRunner,
)
from bigquery_agent_analytics.aggregate_grader import WeightedStrategy
from bigquery_agent_analytics.system_evaluator import SESSION_SUMMARY_QUERY

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

# The 9 standard event types emitted by BigQuery Conversational Analytics Prompt & Response Logging.
BQCA_EVENT_TYPES = [
    "INVOCATION_STARTING",
    "USER_MESSAGE_RECEIVED",
    "AGENT_RESPONSE",
    "INVOCATION_COMPLETED",
    "LLM_RESPONSE",
    "EMBEDDING_SUGGESTION",
    "INVOCATION_ERROR",
    "AGENT_ERROR",
    "LLM_ERROR",
]

print(f"bigquery-agent-analytics {_dist_version('bigquery-agent-analytics')} | pandas {pd.__version__}")
print(f"Project={PROJECT_ID} | Dataset={DATASET_ID} | BQ Location={LOCATION} | Vertex Location={VERTEX_LOCATION}")
print(f"Source table={RAW_TABLE_ID} -> Compatibility view={COMPAT_VIEW_ID} | Turns view={TURNS_VIEW_ID}")
print(f"Golden Q&A table={GOLDEN_QA_TABLE_ID} | Judge/BQML AI endpoint={JUDGE_MODEL} | Embeddings={EMBEDDING_MODEL}")

In [ ]:
#@title 1.2 Shared BigQuery Helpers & Pre-Flight Check
import asyncio
import concurrent.futures
import contextvars
import gc
import inspect
import json
import math
from typing import Optional
import bigquery_agent_analytics.categorical_evaluator as _sdk_cat_eval_mod
import bigquery_agent_analytics.client as _sdk_client_mod
import bigquery_agent_analytics.evaluators as _sdk_evaluators_mod
import bigquery_agent_analytics.feedback as _sdk_feedback_mod
from bigquery_agent_analytics.trace import Trace as _SDKTrace
import bigquery_agent_analytics.performance_evaluator as _sdk_perf_mod
from bigquery_agent_analytics.aggregate_grader import AggregateGrader, GraderResult
from bigquery_agent_analytics.multi_trial_performance_evaluator import compute_pass_at_k, compute_pass_pow_k
from bigquery_agent_analytics.system_evaluator import EvaluationReport

_PROJECT_ID_RE = re.compile(r"^[A-Za-z0-9_.:\-]+\Z")
_IDENTIFIER_RE = re.compile(r"^[A-Za-z0-9_\-]+\Z")
if not _PROJECT_ID_RE.fullmatch(PROJECT_ID):
    raise ValueError(f"Invalid GCP PROJECT_ID: {PROJECT_ID!r}")
for _name, _val in [
    ("DATASET_ID", DATASET_ID),
    ("RAW_TABLE_ID", RAW_TABLE_ID),
    ("COMPAT_VIEW_ID", COMPAT_VIEW_ID),
    ("TURNS_VIEW_ID", TURNS_VIEW_ID),
    ("GOLDEN_QA_TABLE_ID", GOLDEN_QA_TABLE_ID),
]:
    if not _IDENTIFIER_RE.fullmatch(_val):
        raise ValueError(f"Invalid BigQuery identifier for {_name}: {_val!r}")

_CONFIG_LITERAL_RE = re.compile(r"^[A-Za-z0-9_.\-/:]+\Z")
for _name, _val in [
    ("LOCATION", LOCATION),
    ("VERTEX_LOCATION", VERTEX_LOCATION),
    ("VERTEX_CONNECTION_ID", VERTEX_CONNECTION_ID),
    ("JUDGE_MODEL", JUDGE_MODEL),
    ("CATEGORICAL_MODEL", CATEGORICAL_MODEL),
    ("EMBEDDING_MODEL", EMBEDDING_MODEL),
    ("DIRECT_JUDGE_MODEL", DIRECT_JUDGE_MODEL),
]:
    if not _CONFIG_LITERAL_RE.fullmatch(_val):
        raise ValueError(f"Invalid configuration literal for {_name}: {_val!r}")
if int(AI_SQL_TIMEOUT_SEC) <= 0:
    raise ValueError(f"AI_SQL_TIMEOUT_SEC must be positive, got {AI_SQL_TIMEOUT_SEC!r}")

bq = bqaa.make_bq_client(PROJECT_ID, location=LOCATION)
RESULTS = {}
PERSONA_BY_SESSION = {}
EVAL_SESSION_IDS = None
EVAL_FILTER = TraceFilter(limit=int(MAX_EVAL_SESSIONS))

_TRANSIENT_BQ_EXCEPTIONS = (
    gcp_exceptions.ServiceUnavailable,
    gcp_exceptions.TooManyRequests,
    gcp_exceptions.InternalServerError,
)
_AI_SQL_PATTERN = re.compile(r"\b(?:AI\.(?:GENERATE|CLASSIFY|EMBED)|ML\.GENERATE_(?:TEXT|EMBEDDING))\b", re.IGNORECASE)


def _cancel_bq_job_safely(job, reason="timeout"):
    """Requests a best-effort cancel of a BigQuery job and returns (accepted, message).

    BigQuery cancellation is asynchronous: accepted=True only means the cancel request was sent. The job may keep running
    for a moment and work done before it stops is still billed. If the request fails, the failure is NOT swallowed: a
    WARNING with the job_id and a manual `bq cancel` command is printed and (False, message) is returned so callers can
    surface it (a timed-out warehouse job may still be RUNNING and billing).
    """
    if job is None:
        return True, "no BigQuery job handle was created, so there is nothing to cancel"
    job_id = getattr(job, "job_id", None) or "unknown"
    detail = None
    try:
        accepted = bool(job.cancel())
        if not accepted:
            detail = "job.cancel() returned False"
    except Exception as cancel_exc:
        accepted = False
        detail = f"{type(cancel_exc).__name__}: {cancel_exc}"
    try:
        job._cancel_confirmed = accepted
    except Exception:
        pass
    if accepted:
        return True, f"best-effort job.cancel() requested for job_id={job_id}"
    print(
        f"⚠️ WARNING: Could not confirm cancellation of BigQuery job_id={job_id} ({reason}): {detail}. "
        "The warehouse job may still be RUNNING and billing in BigQuery. Inspect or cancel it manually: "
        f"bq --project_id={PROJECT_ID} --location={LOCATION} cancel {job_id}"
    )
    return False, (
        f"WARNING: job.cancel() could not be confirmed for job_id={job_id} ({detail}); "
        "the warehouse job may still be running and billing"
    )


def _tag_cancel_outcome(exc, cancel_ok, job, effective_timeout=None, cause=None):
    """Attaches cancel_confirmed / job_id / effective_timeout to a timeout exception so fallback cells can warn precisely."""
    for target in (exc, cause):
        if target is None:
            continue
        for attr, value in (
            ("cancel_confirmed", cancel_ok),
            ("job_id", getattr(job, "job_id", None) or "unknown"),
            ("effective_timeout", effective_timeout),
        ):
            try:
                setattr(target, attr, value)
            except Exception:
                pass
    return exc


def _wrap_ai_query_job(job, default_timeout_sec):
    """Wraps a BigQuery QueryJob so .result() and .to_dataframe() enforce a client-side wait timeout (default_timeout_sec) and request a best-effort job.cancel() when it expires."""
    orig_result = job.result
    orig_to_dataframe = getattr(job, "to_dataframe", None)

    def _timed_result(*args, **kwargs):
        if kwargs.get("timeout") is None and not args:
            kwargs["timeout"] = float(default_timeout_sec)
        effective_timeout = float(kwargs.get("timeout") or default_timeout_sec)
        try:
            return orig_result(*args, **kwargs)
        except (
            TimeoutError,
            concurrent.futures.TimeoutError,
            gcp_exceptions.DeadlineExceeded,
            gcp_exceptions.Cancelled,
            KeyboardInterrupt,
        ) as exc:
            cancel_ok, cancel_msg = _cancel_bq_job_safely(job, reason=type(exc).__name__)
            if isinstance(exc, (TimeoutError, concurrent.futures.TimeoutError)):
                raise _tag_cancel_outcome(
                    gcp_exceptions.DeadlineExceeded(
                        "BigQuery AI query exceeded the AI_SQL_TIMEOUT_SEC client-side wait timeout "
                        f"({effective_timeout:g}s); {cancel_msg}."
                    ),
                    cancel_ok,
                    job,
                    effective_timeout,
                    cause=exc,
                ) from exc
            raise _tag_cancel_outcome(exc, cancel_ok, job, effective_timeout)

    job.result = _timed_result
    if callable(orig_to_dataframe):
        def _timed_to_dataframe(*args, **kwargs):
            _timed_result()
            return orig_to_dataframe(*args, **kwargs)

        job.to_dataframe = _timed_to_dataframe
    return job


if not hasattr(bq, "_orig_query"):
    bq._orig_query = bq.query


def _bq_query_with_ai_timeout(query, *args, **kwargs):
    job = bq._orig_query(query, *args, **kwargs)
    if isinstance(query, str) and _AI_SQL_PATTERN.search(query):
        return _wrap_ai_query_job(job, float(AI_SQL_TIMEOUT_SEC))
    return job


bq.query = _bq_query_with_ai_timeout


# Drain google-genai httpx AsyncClient cleanup tasks before _run_sync closes its temporary event loop
# (prevents 'Task exception was never retrieved ... RuntimeError: Event loop is closed' on Python 3.11+):
if not hasattr(_sdk_client_mod, "_orig_run_sync"):
    _sdk_client_mod._orig_run_sync = _sdk_client_mod._run_sync


def _run_sync_with_genai_cleanup(coro):
    async def _wrapped():
        try:
            return await coro
        finally:
            gc.collect()
            await asyncio.sleep(0.05)

    return _sdk_client_mod._orig_run_sync(_wrapped())


_sdk_client_mod._run_sync = _run_sync_with_genai_cleanup


def _extract_bqca_span_text(span_or_event):
    """Extracts full text from a BQCA Span or AgentEvent across all 6 BQCA content shapes."""
    c = getattr(span_or_event, "content", None)
    if isinstance(c, str):
        s = c.strip()
        if s.startswith("{"):
            try:
                c = json.loads(s)
            except Exception:
                return s
        else:
            return s
    if not isinstance(c, dict):
        return ""
    for key in ("text_summary", "text", "message", "summary"):
        val = c.get(key)
        if isinstance(val, str) and val.strip():
            return val.strip()
    for resp_key in ("response", "raw_response"):
        resp = c.get(resp_key)
        if isinstance(resp, str) and resp.strip():
            return resp.strip()
        if isinstance(resp, dict):
            parts = resp.get("parts")
            if isinstance(parts, list):
                md = "\n\n".join(
                    str(p.get("markdown") or p.get("text") or "").strip()
                    for p in parts
                    if isinstance(p, dict) and str(p.get("markdown") or p.get("text") or "").strip()
                )
                if md:
                    return md
            cq = resp.get("clarifying_question")
            if isinstance(cq, str) and cq.strip():
                return cq.strip()
    parts = c.get("parts")
    if isinstance(parts, list):
        pt = "\n\n".join(
            str(p.get("text") or p.get("markdown") or "").strip()
            for p in parts
            if isinstance(p, dict) and str(p.get("text") or p.get("markdown") or "").strip()
        )
        if pt:
            return pt
    return ""


def _is_bqca_clarifying_only(span_or_event):
    """Returns True when an AGENT_RESPONSE event only carries a clarifying_question without answer markdown."""
    c = getattr(span_or_event, "content", None)
    if isinstance(c, str):
        s = c.strip()
        if s.startswith("{"):
            try:
                c = json.loads(s)
            except Exception:
                return False
        else:
            return False
    if not isinstance(c, dict):
        return False
    raw_resp = (
        c.get("raw_response")
        if isinstance(c.get("raw_response"), dict)
        else (c.get("response") if isinstance(c.get("response"), dict) else None)
    )
    if not isinstance(raw_resp, dict):
        return False
    cq = raw_resp.get("clarifying_question")
    if not (isinstance(cq, str) and cq.strip()):
        return False
    parts = raw_resp.get("parts")
    has_md = isinstance(parts, list) and any(
        isinstance(p, dict) and str(p.get("markdown") or p.get("text") or "").strip()
        for p in parts
    )
    return not has_md


def _extract_bqca_final_response(spans_or_events):
    """Returns the final answer text from a sequence of SDK Spans or AgentEvents, including BQCA AGENT_RESPONSE."""
    items = list(spans_or_events or ())
    for item in reversed(items):
        if getattr(item, "event_type", None) in ("AGENT_RESPONSE", "AGENT_COMPLETED", "LLM_RESPONSE"):
            if getattr(item, "event_type", None) == "AGENT_RESPONSE" and _is_bqca_clarifying_only(item):
                continue
            txt = _extract_bqca_span_text(item)
            if txt:
                return txt
    for item in reversed(items):
        if getattr(item, "event_type", None) == "AGENT_RESPONSE":
            txt = _extract_bqca_span_text(item)
            if txt:
                return txt
    return None


# In bigquery-agent-analytics 0.5.4, Trace.final_response only inspects LLM_RESPONSE and
# AGENT_COMPLETED spans, returning None on BQCA logs (where LLM_RESPONSE.content is NULL by
# design and answers are logged on AGENT_RESPONSE). When client.evaluate(LLMAsJudge.*) falls
# back from in-warehouse AI.GENERATE to Client._run_api_judge (execution_mode='api_fallback'),
# the unpatched SDK passes final="" and 120-char-truncated span.summary lines to the judge.
if not hasattr(_SDKTrace, "_orig_final_response_fget"):
    _SDKTrace._orig_final_response_fget = getattr(_SDKTrace.final_response, "fget", None)


def _bqca_trace_final_response(self):
    orig_fget = getattr(_SDKTrace, "_orig_final_response_fget", None)
    orig = orig_fget(self) if callable(orig_fget) else None
    return orig or _extract_bqca_final_response(self.spans)


_SDKTrace.final_response = property(_bqca_trace_final_response)

class _ValidatedJudgeScore(float):
    """A measured numeric judge score (0.0..1.0) parsed from a valid rubric JSON response."""

    _bqca_validated = True


class _ErroredJudgeScore(float):
    """A placeholder 0.0 returned when the judge call or JSON score extraction failed."""

    _bqca_judge_error = True


_BQCA_JUDGE_STATE = contextvars.ContextVar("_BQCA_JUDGE_STATE", default=None)

# Start-anchored signatures of judge failures (API / transport / parse / malformed output). Every reply rejected by
# _bqca_judge_criterion starts with "JUDGE_ERROR:". The other start-anchored prefixes classify raw exception/error
# strings when stock _judge_criterion is called directly, without matching mid-sentence prose inside a valid 0.0
# justification (for example, "The agent repeated 429 RESOURCE_EXHAUSTED instead of answering.").
_JUDGE_ERROR_PATTERNS = re.compile(
    r"^\s*(?:"
    r"JUDGE_ERROR:"
    r"|(?:429|500|502|503|504|401|403|404)\s+(?:RESOURCE_EXHAUSTED|INTERNAL|BAD_GATEWAY|UNAVAILABLE|SERVICE_UNAVAILABLE|DEADLINE_EXCEEDED|UNAUTHENTICATED|PERMISSION_DENIED|NOT_FOUND)(?:\s*$|(?=\s*[:.\[{(\-]))"
    r"|(?:RESOURCE_EXHAUSTED|DEADLINE_EXCEEDED|PERMISSION_DENIED|UNAUTHENTICATED|SERVICE_UNAVAILABLE|ClientError|ServerError|APIError|ConnectError|ReadTimeout|ConnectTimeout|PoolTimeout|RemoteProtocolError|TooManyRequests|ServiceUnavailable|InternalServerError|DeadlineExceeded|ValueError|TypeError|JSONDecodeError)\s*[:(]"
    r"|\{\s*[\"']error[\"']\s*:"
    r"|Error during evaluation:"
    r"|Could not parse structured output"
    r"|Invalid judge score:"
    r"|Empty response from judge"
    r"|Malformed judge output"
    r"|google-genai not installed"
    r"|could not convert string to float"
    r"|invalid literal for (?:int|float)\(\)"
    r"|Expecting value:"
    r"|Unterminated string"
    r"|Event loop is closed"
    r"|Job execution was cancelled"
    r"|'[A-Za-z0-9_]+' object has no attribute 'strip'"
    r"|(?:The )?read operation timed out"
    r"|connecterror\b"
    r"|pool timeout\b"
    r"|(?:[A-Za-z0-9_.]*Error|Exception):\s*(?:429|500|502|503|504|401|403|404|RESOURCE_EXHAUSTED|UNAVAILABLE|DEADLINE_EXCEEDED|PERMISSION_DENIED|Event loop is closed|read operation timed out|pool timeout)"
    r")",
    re.IGNORECASE,
)


def _is_judge_error_feedback(feedback_str):
    """Returns True when a 0.0-scored judge feedback string has a start-anchored judge API/transport/parse error signature."""
    if not feedback_str:
        return False
    s = str(feedback_str).strip()
    if _JUDGE_ERROR_PATTERNS.search(s):
        return True
    m = re.match(r"^[A-Za-z0-9_]+:\s*(.+)$", s, re.DOTALL)
    if m:
        rest = m.group(1).strip()
        if _JUDGE_ERROR_PATTERNS.search(rest):
            return True
    return False


def _validate_direct_ping_response(resp, model_name="", location=""):
    """Validates that a direct Vertex AI health-ping response contains non-empty text (never invents 'OK')."""
    raw_text = getattr(resp, "text", None) if resp is not None else None
    cleaned = str(raw_text or "").strip()
    if not cleaned:
        raise RuntimeError(
            f"Direct Vertex AI health-ping returned an empty response from {model_name or 'model'}"
            f"{f' ({location})' if location else ''}."
        )
    return cleaned


def _criterion_feedback_text(llm_feedback, criterion_name):
    """Returns the feedback text recorded for one criterion (the part after '<criterion>:'), or None when it has no line."""
    if not llm_feedback or not criterion_name:
        return None
    prefix = f"{criterion_name}:"
    for line in re.split(r"[\n;]+", str(llm_feedback)):
        stripped = line.strip()
        if stripped.startswith(prefix):
            return stripped[len(prefix) :].strip()
    return None


def _effective_criterion_feedback(llm_feedback, criterion_name=None):
    """Returns the feedback string for criterion_name, handling both '<criterion>: ...' (evaluate_session) and unprefixed feedback (warehouse AI.GENERATE)."""
    if not llm_feedback:
        return ""
    if criterion_name:
        prefixed = _criterion_feedback_text(llm_feedback, criterion_name)
        if prefixed is not None:
            return prefixed
    return str(llm_feedback).strip()


def _criterion_has_judge_error(llm_feedback, criterion_name=None):
    """Checks whether a specific criterion's feedback line (or overall feedback) indicates a judge API/parse error."""
    if not llm_feedback:
        return False
    if criterion_name:
        text = _criterion_feedback_text(llm_feedback, criterion_name)
        if text is not None:
            return _is_judge_error_feedback(text)
        # If other criteria have prefixed JUDGE_ERROR lines in a multi-criterion call, this criterion had no error line:
        if re.search(r"^[A-Za-z0-9_]+:\s*JUDGE_ERROR:", str(llm_feedback).strip(), re.M | re.I):
            return False
        s = str(llm_feedback).strip()
        return bool(_JUDGE_ERROR_PATTERNS.search(s))
    return _is_judge_error_feedback(llm_feedback)


def _judge_error_keys(score, strict=False):
    """Returns criterion names whose 0.0 score came from an API/transport/parse failure rather than a valid rubric score.

    Validated numeric scores from _bqca_judge_criterion (_ValidatedJudgeScore / details['validated_criteria']) and valid
    warehouse AI.GENERATE 0.0 scores (even with empty justification '' or prose mentioning '429 RESOURCE_EXHAUSTED' or
    'Verdict: ...') are never flagged as judge errors.
    """
    if score is None:
        return []
    details = getattr(score, "details", None) or {}
    validated = set(details.get("validated_criteria") or ())
    errored = set(details.get("errored_criteria") or ())
    feedback = getattr(score, "llm_feedback", None)
    keys = []
    for k, v in list((getattr(score, "scores", None) or {}).items()):
        if v is None or float(v) != 0.0:
            continue
        if getattr(v, "_bqca_validated", False) or k in validated:
            continue
        if getattr(v, "_bqca_judge_error", False) or k in errored:
            keys.append(k)
        elif _criterion_has_judge_error(feedback, k):
            keys.append(k)
        elif strict and not _effective_criterion_feedback(feedback, k):
            keys.append(k)
    return keys


def _mark_judge_errors_unevaluated(score, strict=False):
    """Pops errored criteria from score.scores (so they surface as NaN, not 0.0), fails the session, and records details['evaluation_error']."""
    if score is None:
        return score
    err_keys = _judge_error_keys(score, strict=strict)
    if err_keys:
        for k in err_keys:
            score.scores.pop(k, None)
        score.passed = False
        if score.details is None:
            score.details = {}
        score.details.setdefault(
            "evaluation_error",
            score.llm_feedback or f"empty_or_invalid_judge_output ({', '.join(err_keys)})",
        )
    return score


def _mark_warehouse_numeric_scores_validated(report):
    """Marks parsed numeric warehouse scores (ai_generate / ml_generate_text) as validated so valid 0.0 scores are never stripped by explanation text."""
    if report is None:
        return report
    r_details = getattr(report, "details", None)
    exec_mode = str((r_details or {}).get("execution_mode") or "").strip().lower() if isinstance(r_details, dict) else ""
    if exec_mode not in ("ai_generate", "ml_generate_text"):
        return report
    for s in list(getattr(report, "session_scores", None) or []):
        if s is None:
            continue
        if s.details is None:
            s.details = {}
        if s.details.get("evaluation_error"):
            continue
        errored = set(s.details.get("errored_criteria") or ())
        validated = list(s.details.get("validated_criteria") or ())
        for k, v in list((getattr(s, "scores", None) or {}).items()):
            if v is None or k in errored or getattr(v, "_bqca_judge_error", False):
                continue
            if not getattr(v, "_bqca_validated", False):
                try:
                    s.scores[k] = _ValidatedJudgeScore(float(v))
                except (TypeError, ValueError):
                    continue
            if k not in validated:
                validated.append(k)
        if validated:
            s.details["validated_criteria"] = validated
    return report


def _recompute_eval_report_counts(report):
    """Re-derives EvaluationReport counts over the EVALUATED denominator after judge errors are marked unevaluated.

    Stock EvaluationReport counts every session that returned ANY SessionScore, so an errored judge call (a direct-Vertex
    429, an empty/malformed reply, or a warehouse AI.GENERATE NULL that leaves scores={}) would still count as a failed
    session (e.g. 25 sessions, 18 passed = 72%, failed = 7). After this call: total_sessions = evaluated sessions only,
    failed = measured failures only, and details['raw_total_sessions'] / details['not_evaluated_sessions'] keep the full
    picture. Applies to ANY EvaluationReport that carries session_scores (warehouse ai_generate and api_fallback alike);
    sessions with empty scores={} or details['evaluation_error'] are the unevaluated ones. Reports without session_scores
    are returned untouched. Idempotent.
    """
    if report is None:
        return report
    _mark_warehouse_numeric_scores_validated(report)
    session_scores = list(getattr(report, "session_scores", None) or [])
    if not session_scores:
        return report
    for s in session_scores:
        _mark_judge_errors_unevaluated(s)

    def _is_unevaluated(s):
        return (not s.scores) or bool((s.details or {}).get("evaluation_error"))

    raw_total = len(session_scores)
    not_eval = sum(1 for s in session_scores if _is_unevaluated(s))
    evaluated = raw_total - not_eval
    passed = sum(1 for s in session_scores if (not _is_unevaluated(s)) and bool(s.passed))
    failed = evaluated - passed
    report.total_sessions = evaluated
    report.passed_sessions = passed
    report.failed_sessions = failed
    valid_scores = [s.scores for s in session_scores if not _is_unevaluated(s)]
    aggregate = {}
    for key in sorted({k for sc in valid_scores for k in sc}):
        vals = [float(sc[key]) for sc in valid_scores if sc.get(key) is not None]
        if vals:
            aggregate[key] = sum(vals) / len(vals)
    report.aggregate_scores = aggregate
    if report.details is None:
        report.details = {}
    report.details["raw_total_sessions"] = raw_total
    report.details["not_evaluated_sessions"] = not_eval
    return report


# Make EvaluationReport.summary() self-correcting: recompute counts first and append an explicit
# "Unevaluated (judge error)" line so a quota error is never printed as a failed session. Idempotent.
if not getattr(EvaluationReport.summary, "_bqca_wrapped", False):
    _orig_eval_report_summary = EvaluationReport.summary

    def _bqca_eval_report_summary(self):
        _recompute_eval_report_counts(self)
        text = _orig_eval_report_summary(self)
        details = self.details or {}
        not_eval = int(details.get("not_evaluated_sessions", 0) or 0)
        if not_eval > 0:
            raw_total = int(details.get("raw_total_sessions", self.total_sessions + not_eval))
            lines = text.split("\n")
            insert_at = next((i + 1 for i, line in enumerate(lines) if line.startswith("  Failed:")), len(lines))
            lines.insert(insert_at, f"  Unevaluated (judge error): {not_eval} (of {raw_total} total sessions)")
            text = "\n".join(lines)
        return text

    _bqca_eval_report_summary._bqca_wrapped = True
    _bqca_eval_report_summary._bqca_orig = _orig_eval_report_summary
    EvaluationReport.summary = _bqca_eval_report_summary


def _tag_warehouse_criterion_report(report, criterion=None, mode="ai_generate"):
    """Tags parsed numeric scores from warehouse AI.GENERATE / ML.GENERATE_TEXT as _ValidatedJudgeScore."""
    if report is None:
        return report
    if report.details is None:
        report.details = {}
    report.details.setdefault("execution_mode", mode)
    cname = getattr(criterion, "name", None) if criterion is not None else None
    for s in list(getattr(report, "session_scores", None) or []):
        if s is None:
            continue
        if s.details is None:
            s.details = {}
        validated = list(s.details.get("validated_criteria") or ())
        errored = list(s.details.get("errored_criteria") or ())
        scores = getattr(s, "scores", None) or {}
        if not scores:
            if cname and cname not in errored:
                errored.append(cname)
            s.details.setdefault("evaluation_error", s.llm_feedback or "empty_or_null_warehouse_judge_score")
        else:
            for k, v in list(scores.items()):
                if v is None or isinstance(v, bool) or getattr(v, "_bqca_judge_error", False):
                    scores.pop(k, None)
                    if k not in errored:
                        errored.append(k)
                    s.details.setdefault("evaluation_error", f"invalid_warehouse_judge_score ({k})")
                else:
                    try:
                        fv = float(v)
                    except (TypeError, ValueError):
                        scores.pop(k, None)
                        if k not in errored:
                            errored.append(k)
                        s.details.setdefault("evaluation_error", f"non_numeric_warehouse_judge_score ({k})")
                        continue
                    if not math.isfinite(fv) or fv < 0.0 or fv > 1.0:
                        scores.pop(k, None)
                        if k not in errored:
                            errored.append(k)
                        s.details.setdefault("evaluation_error", f"out_of_range_warehouse_judge_score ({k}={fv})")
                        continue
                    scores[k] = _ValidatedJudgeScore(fv)
                    if k not in validated:
                        validated.append(k)
        if validated:
            s.details["validated_criteria"] = validated
        if errored:
            s.details["errored_criteria"] = errored
            s.passed = False
    return report


class _WarehouseJudgeJobProxy:
    def __init__(self, job, row_transform):
        self._job = job
        self._row_transform = row_transform

    def __getattr__(self, name):
        return getattr(self._job, name)

    def result(self, *args, **kwargs):
        raw_rows = list(self._job.result(*args, **kwargs))
        return [self._row_transform(r) for r in raw_rows]


class _WarehouseJudgeBQClientProxy:
    def __init__(self, bq_client, row_transform):
        self._bq = bq_client
        self._row_transform = row_transform

    def __getattr__(self, name):
        return getattr(self._bq, name)

    def query(self, *args, **kwargs):
        return _WarehouseJudgeJobProxy(self._bq.query(*args, **kwargs), self._row_transform)


if hasattr(Client, "_ai_generate_judge") and not getattr(Client._ai_generate_judge, "_bqca_wrapped", False):
    _orig_ai_generate_judge = Client._ai_generate_judge

    def _bqca_ai_generate_judge(self, evaluator, criterion, table, where, params):
        cname = getattr(criterion, "name", None)
        row_errors = {}

        def _check_ai_gen_row(row):
            r = dict(row) if hasattr(row, "keys") else {"session_id": getattr(row, "session_id", "unknown"), "score": getattr(row, "score", None), "justification": getattr(row, "justification", ""), "gen_status": getattr(row, "gen_status", "")}
            sid = r.get("session_id", "unknown")
            gen_status = str(r.get("gen_status") or r.get("status") or r.get("ai_generate_status") or "").strip()
            if gen_status:
                row_errors[sid] = f"JUDGE_ERROR: {gen_status}"
                r["score"] = None
                return r
            raw_s = r.get("score")
            if raw_s is None:
                row_errors[sid] = "JUDGE_ERROR: NULL warehouse score"
                return r
            if isinstance(raw_s, bool):
                raise ValueError(f"Invalid boolean score in AI.GENERATE: {raw_s!r}")
            fv = float(raw_s)
            if not math.isfinite(fv) or fv < 0.0 or fv > 10.0:
                raise ValueError(f"Out-of-range score in AI.GENERATE: {raw_s!r}")
            return r

        orig_bq = self.bq_client
        orig_priv = getattr(self, "_bq_client", orig_bq)
        self._bq_client = _WarehouseJudgeBQClientProxy(orig_bq, _check_ai_gen_row)
        try:
            rep = _orig_ai_generate_judge(self, evaluator, criterion, table, where, params)
        finally:
            self._bq_client = orig_priv
        rep = _tag_warehouse_criterion_report(rep, criterion=criterion, mode="ai_generate")
        for s in list(getattr(rep, "session_scores", None) or []):
            if s.session_id in row_errors and not s.scores:
                if s.details is None:
                    s.details = {}
                s.details["evaluation_error"] = row_errors[s.session_id]
                if cname:
                    err_list = list(s.details.get("errored_criteria") or ())
                    if cname not in err_list:
                        err_list.append(cname)
                    s.details["errored_criteria"] = err_list
        return rep

    _bqca_ai_generate_judge._bqca_wrapped = True
    _bqca_ai_generate_judge._bqca_orig = _orig_ai_generate_judge
    Client._ai_generate_judge = _bqca_ai_generate_judge


if hasattr(Client, "_bqml_judge") and not getattr(Client._bqml_judge, "_bqca_wrapped", False):
    _orig_bqml_judge = Client._bqml_judge

    def _bqca_bqml_judge(self, evaluator, criterion, table, where, params, text_model):
        cname = getattr(criterion, "name", None)
        skey = getattr(criterion, "score_key", cname)
        row_errors = {}

        def _check_bqml_row(row):
            r = dict(row) if hasattr(row, "keys") else {"session_id": getattr(row, "session_id", "unknown"), "evaluation": getattr(row, "evaluation", "")}
            sid = r.get("session_id", "unknown")
            gen_status = str(r.get("ml_generate_text_status") or r.get("gen_status") or r.get("status") or "").strip()
            eval_text = r.get("evaluation")
            if gen_status:
                row_errors[sid] = f"JUDGE_ERROR: {gen_status}"
                r["evaluation"] = ""
                return r
            if eval_text is None or not str(eval_text).strip():
                row_errors[sid] = "JUDGE_ERROR: empty ML.GENERATE_TEXT evaluation"
                r["evaluation"] = ""
                return r
            parsed = _sdk_evaluators_mod._parse_json_from_text(str(eval_text))
            if not isinstance(parsed, dict) or skey not in parsed:
                row_errors[sid] = f"JUDGE_ERROR: {str(eval_text).strip()[:240]}"
                r["evaluation"] = ""
                return r
            raw_s = parsed[skey]
            if raw_s is None or isinstance(raw_s, bool):
                raise ValueError(f"Invalid ML.GENERATE_TEXT score for '{skey}': {raw_s!r}")
            fv = float(raw_s)
            if not math.isfinite(fv) or fv < 0.0 or fv > 10.0:
                raise ValueError(f"Out-of-range ML.GENERATE_TEXT score for '{skey}': {raw_s!r}")
            return r

        orig_bq = self.bq_client
        orig_priv = getattr(self, "_bq_client", orig_bq)
        self._bq_client = _WarehouseJudgeBQClientProxy(orig_bq, _check_bqml_row)
        try:
            rep = _orig_bqml_judge(self, evaluator, criterion, table, where, params, text_model)
        finally:
            self._bq_client = orig_priv
        rep = _tag_warehouse_criterion_report(rep, criterion=criterion, mode="ml_generate_text")
        for s in list(getattr(rep, "session_scores", None) or []):
            if s.session_id in row_errors and not s.scores:
                if s.details is None:
                    s.details = {}
                s.details["evaluation_error"] = row_errors[s.session_id]
                if cname:
                    err_list = list(s.details.get("errored_criteria") or ())
                    if cname not in err_list:
                        err_list.append(cname)
                    s.details["errored_criteria"] = err_list
        return rep

    _bqca_bqml_judge._bqca_wrapped = True
    _bqca_bqml_judge._bqca_orig = _orig_bqml_judge
    Client._bqml_judge = _bqca_bqml_judge


if hasattr(_sdk_client_mod, "_merge_criterion_reports") and not getattr(
    _sdk_client_mod._merge_criterion_reports, "_bqca_wrapped", False
):
    _orig_merge_criterion_reports = _sdk_client_mod._merge_criterion_reports

    def _bqca_merge_criterion_reports(evaluator_name, dataset, criteria, criterion_reports):
        merged = _orig_merge_criterion_reports(evaluator_name, dataset, criteria, criterion_reports)
        by_key = {}
        for crit, rep in criterion_reports or ():
            cname = getattr(crit, "name", None)
            for ss in list(getattr(rep, "session_scores", None) or []):
                key = (
                    ss.session_id,
                    (ss.details or {}).get("user_id"),
                    (ss.details or {}).get("root_agent_name"),
                    (ss.details or {}).get("scope_signature"),
                )
                entry = by_key.setdefault(key, {"validated": [], "errored": [], "errors": []})
                sd = ss.details or {}
                for k in sd.get("validated_criteria") or ():
                    if k not in entry["validated"]:
                        entry["validated"].append(k)
                for k in sd.get("errored_criteria") or ():
                    if k not in entry["errored"]:
                        entry["errored"].append(k)
                for k, v in (ss.scores or {}).items():
                    if getattr(v, "_bqca_validated", False) and k not in entry["validated"]:
                        entry["validated"].append(k)
                    elif getattr(v, "_bqca_judge_error", False) and k not in entry["errored"]:
                        entry["errored"].append(k)
                if not ss.scores and cname and cname not in entry["errored"]:
                    entry["errored"].append(cname)
                if sd.get("evaluation_error"):
                    entry["errors"].append(str(sd["evaluation_error"]))
        all_cnames = [getattr(c, "name", None) for c in (criteria or ()) if getattr(c, "name", None)]
        for ms in list(getattr(merged, "session_scores", None) or []):
            key = (
                ms.session_id,
                (ms.details or {}).get("user_id"),
                (ms.details or {}).get("root_agent_name"),
                (ms.details or {}).get("scope_signature"),
            )
            info = by_key.get(key, {"validated": [], "errored": [], "errors": []})
            if ms.details is None:
                ms.details = {}
            val_list = list(info["validated"])
            err_list = list(info["errored"])
            for cname in all_cnames:
                if cname not in (ms.scores or {}) and cname not in err_list:
                    err_list.append(cname)
            for k in val_list:
                if k in (ms.scores or {}) and ms.scores[k] is not None and not getattr(ms.scores[k], "_bqca_validated", False):
                    try:
                        ms.scores[k] = _ValidatedJudgeScore(float(ms.scores[k]))
                    except (TypeError, ValueError):
                        pass
            if val_list:
                ms.details["validated_criteria"] = val_list
            if err_list:
                ms.details["errored_criteria"] = err_list
                ms.passed = False
                ms.details["evaluation_error"] = (
                    "; ".join(info["errors"])
                    if info["errors"]
                    else f"missing_or_errored_criteria ({', '.join(err_list)})"
                )
        return merged

    _bqca_merge_criterion_reports._bqca_wrapped = True
    _bqca_merge_criterion_reports._bqca_orig = _orig_merge_criterion_reports
    _sdk_client_mod._merge_criterion_reports = _bqca_merge_criterion_reports


if not getattr(Client.evaluate, "_bqca_wrapped", False):
    _orig_client_evaluate = Client.evaluate

    def _bqca_client_evaluate(self, *args, **kwargs):
        report = _orig_client_evaluate(self, *args, **kwargs)
        return _recompute_eval_report_counts(report)

    _bqca_client_evaluate._bqca_wrapped = True
    _bqca_client_evaluate._bqca_orig = _orig_client_evaluate
    Client.evaluate = _bqca_client_evaluate


# Direct-judge reliability for LLMAsJudge.evaluate_session, which serves client.evaluate(api_fallback) and GraderPipeline
# (TrialRunner goes through PerformanceEvaluator.llm_judge_evaluate instead; see the wrapper further below). Three
# layers, all idempotent across cell re-runs:
#   1. _judge_criterion: SDK 0.5.4 turns an EMPTY judge reply into (0.0, ''), a non-JSON or missing-key reply into
#      (0.0, <raw text>) and a non-numeric score such as {"sentiment": "N/A"} into (0.0, "could not convert ...") --
#      all of which look like a MEASURED 0.0. The wrapper keeps the same prompt, model and call but returns an
#      explicit "JUDGE_ERROR: <reason>" failure for every empty / malformed / non-numeric / out-of-range reply.
#   2. evaluate_session: retries only TRANSIENT failures (429 / 5xx / timeouts: up to 2 more attempts after 2s and 4s) and
#      reports whatever still failed as unevaluated (score popped, details['evaluation_error']) instead of a 0.0.
#   3. AggregateGrader._run_grader (GraderPipeline): carries that failure forward as a structured _BQCAGraderResult
#      (empty scores + evaluation_error) so Cell 6B never publishes a latency-only composite as a quality score.
_JUDGE_RETRY_BACKOFF_SEC = (2.0, 4.0)
_JUDGE_PERMANENT_ERROR_RE = re.compile(
    r"\b(?:INVALID_ARGUMENT|PERMISSION_DENIED|UNAUTHENTICATED|NOT_FOUND|FAILED_PRECONDITION)\b"
    r"|API key not valid"
    r"|Empty response from judge|Malformed judge output|Invalid judge score|google-genai not installed",
    re.IGNORECASE,
)
_JUDGE_TRANSIENT_ERROR_RE = re.compile(
    r"\b(?:429|500|502|503|504)\b|RESOURCE_EXHAUSTED|\bUNAVAILABLE\b|SERVICE_UNAVAILABLE|DEADLINE_EXCEEDED|\bINTERNAL\b"
    r"|\b(?:ReadTimeout|ConnectTimeout|PoolTimeout|ConnectError|RemoteProtocolError|TooManyRequests|ServiceUnavailable|InternalServerError|DeadlineExceeded)\b"
    r"|timed out|Event loop is closed",
    re.IGNORECASE,
)


def _is_transient_judge_error(message):
    """True for failures worth retrying (rate limits, 5xx, timeouts, dropped connections); False for permission/auth/not-found/invalid-argument errors and for empty or malformed judge output.

    Every retry is another billed Vertex AI call, so permanent failures fail fast instead of being retried.
    """
    text = str(message or "")
    if _JUDGE_PERMANENT_ERROR_RE.search(text):
        return False
    return bool(_JUDGE_TRANSIENT_ERROR_RE.search(text))


def _judge_error_result(message, criterion_name=None):
    """Returns the explicit (_ErroredJudgeScore(0.0), 'JUDGE_ERROR: ...') failure tuple used by the _judge_criterion wrapper."""
    err_score = _ErroredJudgeScore(0.0)
    state = _BQCA_JUDGE_STATE.get()
    if isinstance(state, dict) and criterion_name:
        if criterion_name not in state["errored"]:
            state["errored"].append(criterion_name)
        state["raw_scores"][criterion_name] = err_score
    return err_score, "JUDGE_ERROR: " + " ".join(str(message).split())[:300]


_JUDGE_CRITERION_PARAMS = ("self", "criterion", "trace_text", "final_response")
if not getattr(LLMAsJudge._judge_criterion, "_bqca_wrapped", False):
    if (
        tuple(inspect.signature(LLMAsJudge._judge_criterion).parameters) == _JUDGE_CRITERION_PARAMS
        and callable(getattr(_sdk_evaluators_mod, "_parse_json_from_text", None))
    ):
        _orig_judge_criterion = LLMAsJudge._judge_criterion

        async def _bqca_judge_criterion(self, criterion, trace_text, final_response):
            prompt = criterion.prompt_template.format(
                trace_text=trace_text,
                final_response=final_response or "No response.",
            )
            key = criterion.score_key
            cname = getattr(criterion, "name", key)
            try:
                from google import genai
                from google.genai import types
            except ImportError:
                return _judge_error_result("google-genai not installed", cname)
            try:
                response = await genai.Client().aio.models.generate_content(
                    model=self.model,
                    contents=prompt,
                    config=types.GenerateContentConfig(temperature=0.1, max_output_tokens=2048),
                )
                text = (getattr(response, "text", None) or "").strip()
                if not text:
                    return _judge_error_result("Empty response from judge model", cname)
                result = _sdk_evaluators_mod._parse_json_from_text(text)
                if not isinstance(result, dict) or key not in result:
                    return _judge_error_result(f"Malformed judge output (missing '{key}'): {text}", cname)
                value = result[key]
                try:
                    if isinstance(value, bool) or value is None:
                        raise ValueError("not a number")
                    raw = float(value)
                except (TypeError, ValueError):
                    raw = float("nan")
                if not math.isfinite(raw) or not 0.0 <= raw <= 10.0:
                    return _judge_error_result(
                        f"Invalid judge score for '{key}': expected a number from 0 to 10, got {value!r}",
                        cname,
                    )
                justification = str(result.get("justification") or "").strip()
                val_score = _ValidatedJudgeScore(raw / 10.0)
                state = _BQCA_JUDGE_STATE.get()
                if isinstance(state, dict) and cname:
                    if cname not in state["validated"]:
                        state["validated"].append(cname)
                    state["raw_scores"][cname] = val_score
                return val_score, justification
            except Exception as exc:
                return _judge_error_result(exc, cname)

        _bqca_judge_criterion._bqca_wrapped = True
        _bqca_judge_criterion._bqca_orig = _orig_judge_criterion
        LLMAsJudge._judge_criterion = _bqca_judge_criterion
    else:
        print(
            "⚠️ LLMAsJudge._judge_criterion has an unexpected signature in this SDK version; relying on start-anchored "
            "feedback-signature checks only."
        )

if not getattr(LLMAsJudge.evaluate_session, "_bqca_wrapped", False):
    _orig_llm_judge_evaluate_session = LLMAsJudge.evaluate_session

    async def _bqca_llm_judge_evaluate_session(self, trace_text, final_response):
        score = None
        for attempt in range(len(_JUDGE_RETRY_BACKOFF_SEC) + 1):
            call_state = {"validated": [], "errored": [], "raw_scores": {}}
            token = _BQCA_JUDGE_STATE.set(call_state)
            try:
                score = await _orig_llm_judge_evaluate_session(self, trace_text, final_response)
            finally:
                _BQCA_JUDGE_STATE.reset(token)
            if score is not None:
                for _ck, _raw_v in call_state["raw_scores"].items():
                    if _ck in (score.scores or {}):
                        score.scores[_ck] = _raw_v
                for _ck, _cv in (score.scores or {}).items():
                    if getattr(_cv, "_bqca_validated", False) and _ck not in call_state["validated"]:
                        call_state["validated"].append(_ck)
                    elif getattr(_cv, "_bqca_judge_error", False) and _ck not in call_state["errored"]:
                        call_state["errored"].append(_ck)
                if call_state["validated"]:
                    if score.details is None:
                        score.details = {}
                    score.details["validated_criteria"] = list(call_state["validated"])
                if call_state["errored"]:
                    if score.details is None:
                        score.details = {}
                    score.details["errored_criteria"] = list(call_state["errored"])
            err_keys = _judge_error_keys(score)
            if not err_keys:
                break
            retryable = any(
                _is_transient_judge_error(_criterion_feedback_text(score.llm_feedback, k) or score.llm_feedback or "")
                for k in err_keys
            )
            if attempt < len(_JUDGE_RETRY_BACKOFF_SEC) and retryable:
                await asyncio.sleep(_JUDGE_RETRY_BACKOFF_SEC[attempt])
                continue
            break
        return _mark_judge_errors_unevaluated(score)

    _bqca_llm_judge_evaluate_session._bqca_wrapped = True
    _bqca_llm_judge_evaluate_session._bqca_orig = _orig_llm_judge_evaluate_session
    LLMAsJudge.evaluate_session = _bqca_llm_judge_evaluate_session


class _BQCAGraderResult(GraderResult):
    """GraderResult that also records WHY an LLM grader produced no score (evaluation_error is None when it produced real scores)."""

    evaluation_error: Optional[str] = None


_RUN_GRADER_PARAMS = ("self", "entry", "session_summary", "trace_text", "final_response", "session_id")
if not getattr(AggregateGrader._run_grader, "_bqca_wrapped", False):
    if tuple(inspect.signature(AggregateGrader._run_grader).parameters) == _RUN_GRADER_PARAMS:
        _orig_run_grader = AggregateGrader._run_grader

        async def _bqca_run_grader(self, entry, session_summary, trace_text, final_response, session_id=None):
            evaluator = entry.evaluate_fn
            if not isinstance(evaluator, LLMAsJudge):
                return await _orig_run_grader(self, entry, session_summary, trace_text, final_response, session_id)
            score = await evaluator.evaluate_session(trace_text=trace_text, final_response=final_response)
            score = _mark_judge_errors_unevaluated(score)
            error = (score.details or {}).get("evaluation_error")
            if error or not score.scores:
                return _BQCAGraderResult(
                    grader_name=entry.name,
                    scores={},
                    passed=False,
                    evaluation_error=str(error or "empty_or_invalid_judge_output"),
                )
            return GraderResult(grader_name=entry.name, scores=score.scores, passed=score.passed)

        _bqca_run_grader._bqca_wrapped = True
        _bqca_run_grader._bqca_orig = _orig_run_grader
        AggregateGrader._run_grader = _bqca_run_grader
    else:
        print(
            "⚠️ AggregateGrader._run_grader has an unexpected signature in this SDK version; GraderPipeline cannot "
            "carry judge errors forward, so Cell 6B falls back to its empty-scores guard only."
        )


# TrialRunner / BQCAPerformanceEvaluator do NOT go through LLMAsJudge: they call
# PerformanceEvaluator.llm_judge_evaluate, whose stock prompt loop makes exactly one
# genai generate_content call per rubric and turns any 429 / 503 / timeout into
# "<label> LLM evaluation failed: ..." feedback with no scores (so evaluate_session reports
# "LLM judge missing required metrics" and the trial counts as a 0.0 failure in pass@k).
# Re-implement the same prompt loop with 2s/4s per-call retries for TRANSIENT failures only (429 / 5xx / timeouts;
# permanent ones such as 403 PERMISSION_DENIED, 404 NOT_FOUND or a malformed reply fail fast with a single call per
# rubric), and skip the side-by-side rubric when the required one-sided rubric is unavailable (a persistent 429
# with num_trials=3 therefore costs exactly 3 judge calls per trial, 9 total). Idempotent.
if not getattr(PerformanceEvaluator.llm_judge_evaluate, "_bqca_wrapped", False):
    _orig_perf_llm_judge_evaluate = PerformanceEvaluator.llm_judge_evaluate

    async def _bqca_perf_llm_judge_evaluate(self, trace, task_description, expected_trajectory, golden_response=None):
        try:
            from google import genai
            from google.genai import types
        except ImportError:
            return await _orig_perf_llm_judge_evaluate(self, trace, task_description, expected_trajectory, golden_response)

        trajectory_data = [
            {"tool": tc.tool_name, "args": tc.args, "status": tc.status} for tc in (trace.tool_calls or [])
        ]
        trajectory_json = json.dumps(trajectory_data, indent=2)
        final_response = trace.final_response or "No response captured"
        prompts = [
            (
                "One-sided",
                self._ONE_SIDED_JUDGE_PROMPT.format(
                    task_description=task_description,
                    trajectory_json=trajectory_json,
                    final_response=final_response,
                ),
                ("sentiment", "hallucination"),
                10.0,
            )
        ]
        if golden_response is not None:
            prompts.append(
                (
                    "Side-by-side",
                    self._SIDE_BY_SIDE_JUDGE_PROMPT.format(
                        task_description=task_description,
                        trajectory_json=trajectory_json,
                        expected_trajectory=(
                            json.dumps(expected_trajectory, indent=2) if expected_trajectory is not None else "Not provided"
                        ),
                        golden_response=golden_response,
                        final_response=final_response,
                    ),
                    ("final_answer_correct", "tool_usage_correct", "sound_reasoning", "efficiency"),
                    1.0,
                )
            )
        scores = {}
        feedback_parts = []
        for label, prompt, keys, maximum in prompts:
            rubric_ok = False
            for attempt in range(len(_JUDGE_RETRY_BACKOFF_SEC) + 1):
                try:
                    genai_client = genai.Client()
                    response = await genai_client.aio.models.generate_content(
                        model=self.llm_judge_model,
                        contents=prompt,
                        config=types.GenerateContentConfig(temperature=0.1, max_output_tokens=1024),
                    )
                    text = (response.text or "").strip()
                    parsed_scores = self._judge_scores(text, keys, maximum)
                    if label == "Side-by-side":
                        parsed_scores["llm_judge_correctness"] = float(
                            all(
                                parsed_scores["llm_judge_" + key] == 1.0
                                for key in ("final_answer_correct", "tool_usage_correct", "sound_reasoning")
                            )
                        )
                    scores.update(parsed_scores)
                    result = _sdk_perf_mod._parse_json_from_text(text)
                    feedback_parts.append(str(result.get("justification", text)) if isinstance(result, dict) else text)
                    rubric_ok = True
                    break
                except Exception as exc:
                    message = f"{label} LLM evaluation failed: {exc}"
                    if attempt < len(_JUDGE_RETRY_BACKOFF_SEC) and _is_transient_judge_error(str(exc)):
                        await asyncio.sleep(_JUDGE_RETRY_BACKOFF_SEC[attempt])
                        continue
                    _sdk_perf_mod.logger.warning("%s", message)
                    feedback_parts.append(message)
                    break
            if not rubric_ok and label == "One-sided":
                # Required rubric unavailable -> the trial is unevaluated regardless; do not spend quota on side-by-side.
                break
        return scores, "\n".join(feedback_parts)

    _bqca_perf_llm_judge_evaluate._bqca_wrapped = True
    _bqca_perf_llm_judge_evaluate._bqca_orig = _orig_perf_llm_judge_evaluate
    PerformanceEvaluator.llm_judge_evaluate = _bqca_perf_llm_judge_evaluate


class BQCAClient(Client):
    """BQCA-aware SDK Client: preserves full AGENT_RESPONSE text in api_fallback and relies on the LLMAsJudge.evaluate_session wrapper above for transient-error retries, explicit JUDGE_ERROR handling and NaN (unevaluated) marking."""

    async def _run_api_judge(self, evaluator, traces, max_concurrency=5):
        async def evaluate_trace(trace):
            trace_lines = [
                f"{span.event_type}: {_extract_bqca_span_text(span) or getattr(span, 'summary', '') or ''}"
                for span in trace.spans
            ]
            trace_text = "\n".join(trace_lines)
            final = trace.final_response or _extract_bqca_final_response(trace.spans) or ""
            # Transient 429/5xx/deadline retries and judge-error -> unevaluated (NaN) handling live in the
            # LLMAsJudge.evaluate_session wrapper above, so the callers that go through LLMAsJudge (this
            # api_fallback path and GraderPipeline) behave identically; TrialRunner uses the
            # PerformanceEvaluator.llm_judge_evaluate wrapper instead. _mark_judge_errors_unevaluated is
            # idempotent and only acts if the wrapper was bypassed.
            score = await evaluator.evaluate_session(trace_text, final)
            return _mark_judge_errors_unevaluated(score)

        try:
            return await self._run_trace_evaluations(traces, evaluate_trace, max_concurrency=max_concurrency)
        finally:
            gc.collect()
            await asyncio.sleep(0.05)


Client._run_api_judge = BQCAClient._run_api_judge


# Golden Question Bank coverage: one golden question = one LOWER(TRIM(question)) key.
# SDK 0.5.4's semantic drift query returns one row per EXACT golden string, so a curated bank that holds both "Revenue?"
# and "revenue?" lists both as covered while total_golden collapses them under LOWER(TRIM()) -> 2 covered / 1 total =
# 200%. _normalize_drift_report() collapses covered / uncovered / semantic_matches under the same key and recomputes
# total_golden and coverage_percentage from the collapsed lists (a golden question counts as covered when any of its
# case/whitespace variants is covered). It is applied at the SDK source (_semantic_drift and Client.drift_detection) and
# again at the Cell 5.1 / 5.2 call sites. Idempotent; the keyword-overlap fallback (already de-duplicated by the SDK)
# passes through unchanged.
def _golden_key(question):
    return str(question or "").strip().lower()


_BQCA_LAST_S51_DIAG_BOX = {"diag": None}


def _is_valid_cosine_distance(val):
    if val is None or isinstance(val, bool):
        return False
    try:
        d = float(val)
    except (TypeError, ValueError):
        return False
    return math.isfinite(d) and d >= 0.0


def _normalize_drift_report(report):
    """Collapses case/whitespace-duplicate golden questions (coverage <= 100%) and excludes/counts invalid cosine distances."""
    if report is None:
        return report
    details = getattr(report, "details", None)
    if not isinstance(details, dict):
        details = {}
        try:
            report.details = details
        except Exception:
            pass
    method = str(details.get("method") or "semantic_embedding")
    is_semantic = method in ("semantic_embedding", "semantic")
    invalid_keys = set(details.get("_invalid_golden_keys") or ())
    excluded_questions = list(details.get("excluded_golden_questions") or ())
    matches = details.get("semantic_matches")
    if isinstance(matches, list):
        valid_best = {}
        raw_invalid_by_key = {}
        for m in matches:
            if not isinstance(m, dict):
                continue
            g_str = m.get("golden", "")
            key = _golden_key(g_str)
            if not key:
                continue
            raw_d = m.get("cosine_distance")
            if _is_valid_cosine_distance(raw_d):
                d_val = float(raw_d)
                m_clean = dict(m)
                m_clean["cosine_distance"] = d_val
                if key not in valid_best or d_val < float(valid_best[key]["cosine_distance"]):
                    valid_best[key] = m_clean
            else:
                raw_invalid_by_key.setdefault(key, g_str)
        for key, g_str in raw_invalid_by_key.items():
            if key not in valid_best:
                invalid_keys.add(key)
                if g_str and g_str not in excluded_questions:
                    excluded_questions.append(g_str)
        details["semantic_matches"] = list(valid_best.values())
    covered, covered_keys = [], set()
    for q in list(getattr(report, "covered_questions", None) or []):
        key = _golden_key(q)
        if key and key not in invalid_keys and key not in covered_keys:
            covered_keys.add(key)
            covered.append(q)
    uncovered, uncovered_keys = [], set()
    for q in list(getattr(report, "uncovered_questions", None) or []):
        key = _golden_key(q)
        if key and key not in invalid_keys and key not in covered_keys and key not in uncovered_keys:
            uncovered_keys.add(key)
            uncovered.append(q)
    unique_golden = len(covered) + len(uncovered)
    if unique_golden == 0 and not invalid_keys:
        unique_golden = int(getattr(report, "total_golden", 0) or 0)
    details["unique_golden_count"] = unique_golden
    if is_semantic:
        prev_failed = int(details.get("failed_semantic_rows", 0) or 0)
        failed_count = max(prev_failed, len(invalid_keys))
        details["failed_semantic_rows"] = failed_count
        details["excluded_golden_questions"] = excluded_questions
        details["semantic_available"] = bool(unique_golden > 0 or failed_count == 0)
        if failed_count > 0:
            details["failed_rows_note"] = (
                f"{failed_count} golden question row(s) returned NULL or invalid cosine distance and were excluded from the semantic coverage denominator"
            )
    else:
        diag = _BQCA_LAST_S51_DIAG_BOX.get("diag")
        details["semantic_available"] = False
        if isinstance(diag, dict) and diag.get("failed_semantic_rows") is not None:
            details.setdefault("failed_semantic_rows", int(diag["failed_semantic_rows"]))
            if diag.get("excluded_golden_questions"):
                details.setdefault("excluded_golden_questions", list(diag["excluded_golden_questions"]))
            details.setdefault(
                "failed_rows_note",
                f"semantic evaluation unavailable ({diag.get('reason', 'all semantic drift rows returned NULL/invalid distance')}); "
                f"{diag['failed_semantic_rows']} failed row(s) recorded before keyword_overlap fallback",
            )
        else:
            details.setdefault("failed_semantic_rows", None)
            reason_sfx = f" ({diag['reason']})" if isinstance(diag, dict) and diag.get("reason") else ""
            details.setdefault(
                "failed_rows_note",
                f"semantic evaluation unavailable{reason_sfx}; per-row failed embedding count is unknown on the SDK Section 5.1 route "
                "(feedback.py selects AI.EMBED(...).result without .status and falls back to keyword_overlap)",
            )
    details.pop("_invalid_golden_keys", None)
    report.covered_questions = covered
    report.uncovered_questions = uncovered
    report.total_golden = unique_golden
    report.coverage_percentage = (len(covered) / unique_golden * 100.0) if unique_golden else 0.0
    return report


_sdk_semantic_drift = getattr(_sdk_feedback_mod, "_semantic_drift", None)
if (
    _sdk_semantic_drift is not None
    and inspect.iscoroutinefunction(_sdk_semantic_drift)
    and not getattr(_sdk_semantic_drift, "_bqca_wrapped", False)
):
    _orig_semantic_drift = _sdk_semantic_drift

    class _FilteringDriftJobProxy:
        def __init__(self, job, state):
            self._job = job
            self._state = state

        def __getattr__(self, name):
            return getattr(self._job, name)

        def result(self, *args, **kwargs):
            raw_rows = list(self._job.result(*args, **kwargs))
            valid_rows = []
            for r in raw_rows:
                d = r.get("distance") if hasattr(r, "get") else getattr(r, "distance", None)
                gq = r.get("golden_question", "") if hasattr(r, "get") else getattr(r, "golden_question", "")
                key = _golden_key(gq)
                if _is_valid_cosine_distance(d):
                    valid_rows.append(r)
                    if key:
                        self._state["valid_keys"].add(key)
                else:
                    if key:
                        self._state["invalid_keys"].add(key)
                    if gq and gq not in self._state["excluded_questions"]:
                        self._state["excluded_questions"].append(gq)
            for k in list(self._state["invalid_keys"]):
                if k in self._state["valid_keys"]:
                    self._state["invalid_keys"].discard(k)
            if raw_rows and not valid_rows:
                failed_n = max(len(self._state["invalid_keys"]), len(raw_rows))
                diag_entry = {
                    "failed_semantic_rows": failed_n,
                    "excluded_golden_questions": list(self._state["excluded_questions"]),
                    "reason": f"all {failed_n} semantic drift row(s) returned NULL or invalid cosine distance",
                }
                self._state["diag"] = diag_entry
                _BQCA_LAST_S51_DIAG_BOX["diag"] = diag_entry
                raise RuntimeError(f"All {failed_n} semantic drift row(s) returned NULL or invalid cosine distance")
            return valid_rows

    class _FilteringDriftBQClientProxy:
        def __init__(self, bq_client, state):
            self._bq = bq_client
            self._state = state

        def __getattr__(self, name):
            return getattr(self._bq, name)

        def query(self, *args, **kwargs):
            return _FilteringDriftJobProxy(self._bq.query(*args, **kwargs), self._state)

    async def _bqca_semantic_drift(bq_client, *args, **kwargs):
        state = {"valid_keys": set(), "invalid_keys": set(), "excluded_questions": [], "diag": None}
        proxy_bq = _FilteringDriftBQClientProxy(bq_client, state) if bq_client is not None else bq_client
        try:
            rep = await _orig_semantic_drift(proxy_bq, *args, **kwargs)
        except Exception as exc:
            if state.get("diag") is not None:
                _BQCA_LAST_S51_DIAG_BOX["diag"] = state["diag"]
            elif _BQCA_LAST_S51_DIAG_BOX.get("diag") is None:
                _BQCA_LAST_S51_DIAG_BOX["diag"] = {
                    "failed_semantic_rows": None,
                    "excluded_golden_questions": [],
                    "reason": f"{type(exc).__name__}: {exc}",
                }
            raise
        if rep is not None and state["invalid_keys"]:
            if rep.details is None:
                rep.details = {}
            rep.details["_invalid_golden_keys"] = set(state["invalid_keys"])
            rep.details["failed_semantic_rows"] = len(state["invalid_keys"])
            rep.details["excluded_golden_questions"] = [
                q for q in state["excluded_questions"] if _golden_key(q) in state["invalid_keys"]
            ]
        return _normalize_drift_report(rep)

    _bqca_semantic_drift._bqca_wrapped = True
    _bqca_semantic_drift._bqca_orig = _orig_semantic_drift
    _sdk_feedback_mod._semantic_drift = _bqca_semantic_drift

if not getattr(Client.drift_detection, "_bqca_wrapped", False):
    _orig_client_drift_detection = Client.drift_detection

    def _bqca_client_drift_detection(self, *args, **kwargs):
        _BQCA_LAST_S51_DIAG_BOX["diag"] = None
        try:
            return _normalize_drift_report(_orig_client_drift_detection(self, *args, **kwargs))
        finally:
            _BQCA_LAST_S51_DIAG_BOX["diag"] = None

    _bqca_client_drift_detection._bqca_wrapped = True
    _bqca_client_drift_detection._bqca_orig = _orig_client_drift_detection
    Client.drift_detection = _bqca_client_drift_detection


def T(table_id):
    """Returns a back-ticked fully qualified BigQuery table or view identifier."""
    if not _IDENTIFIER_RE.fullmatch(table_id):
        raise ValueError(f"Invalid table_id: {table_id!r}")
    return f"`{PROJECT_ID}.{DATASET_ID}.{table_id}`"


def run_sql(sql, params=None, retries=2, timeout_sec=None):
    """Executes a BigQuery SQL query (capped at MAX_BYTES_BILLED bytes billed) and returns a pandas DataFrame with retry on transient 429/500/503 errors and a best-effort job.cancel() when the client-side wait timeout expires (an unconfirmed cancel prints a WARNING with the job_id)."""
    kinds = {bool: "BOOL", int: "INT64", float: "FLOAT64", str: "STRING"}
    query_params = [
        bigquery.ArrayQueryParameter(k, "STRING", [str(x) for x in v])
        if isinstance(v, (list, tuple))
        else bigquery.ScalarQueryParameter(k, kinds[type(v)], v)
        for k, v in (params or {}).items()
    ]
    job_config = bigquery.QueryJobConfig(
        query_parameters=query_params,
        maximum_bytes_billed=int(MAX_BYTES_BILLED),
    )
    eff_timeout = (
        float(timeout_sec)
        if timeout_sec is not None
        else (float(AI_SQL_TIMEOUT_SEC) if _AI_SQL_PATTERN.search(str(sql)) else None)
    )
    for attempt in range(retries + 1):
        query_job = None
        try:
            query_job = (
                bq._orig_query(sql, job_config=job_config)
                if hasattr(bq, "_orig_query")
                else bq.query(sql, job_config=job_config)
            )
            if eff_timeout is not None:
                query_job.result(timeout=eff_timeout)
            return query_job.to_dataframe()
        except (
            TimeoutError,
            concurrent.futures.TimeoutError,
            gcp_exceptions.DeadlineExceeded,
            gcp_exceptions.Cancelled,
            KeyboardInterrupt,
        ) as exc:
            cancel_ok, cancel_msg = _cancel_bq_job_safely(query_job, reason=type(exc).__name__)
            effective_timeout = eff_timeout if eff_timeout is not None else float(AI_SQL_TIMEOUT_SEC)
            if isinstance(exc, (TimeoutError, concurrent.futures.TimeoutError)):
                raise _tag_cancel_outcome(
                    gcp_exceptions.DeadlineExceeded(
                        f"BigQuery query exceeded the client-side wait timeout ({effective_timeout:g}s); {cancel_msg}."
                    ),
                    cancel_ok,
                    query_job,
                    effective_timeout,
                    cause=exc,
                ) from exc
            raise _tag_cancel_outcome(exc, cancel_ok, query_job, effective_timeout)
        except gcp_exceptions.GoogleAPICallError as exc:
            _msg_lower = str(exc).lower()
            if (
                "bytesbilledlimitexceeded" in _msg_lower
                or "bytes billed" in _msg_lower
                or "maximumbytesbilled" in _msg_lower
            ):
                raise RuntimeError(
                    f"Query exceeded MAX_BYTES_BILLED ({MAX_BYTES_BILLED:,} bytes). "
                    "Raise MAX_BYTES_BILLED in Cell 1.1 (or set BQCA_MAX_BYTES_BILLED) or lower EVAL_LOOKBACK_DAYS."
                ) from exc
            if isinstance(exc, _TRANSIENT_BQ_EXCEPTIONS):
                if attempt == retries:
                    raise
                time.sleep(5 * (attempt + 1))
            else:
                raise


def md_table(df, max_chars=160):
    """Formats a pandas DataFrame as a Markdown table with truncated cells."""
    def cell(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return ""
        s = str(v).replace("|", "\\|").replace("\n", " ")
        return s if len(s) <= max_chars else s[: max_chars - 1] + "…"

    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    lines += ["| " + " | ".join(cell(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


def show(text):
    display(Markdown(text))


def short(text, n=140):
    s = " ".join(str(text or "").split())
    return s if len(s) <= n else s[: n - 1] + "…"


def fmt(x, spec=".2f"):
    return "n/a" if x is None or pd.isna(v := x) else format(v, spec)


class timed:
    """Context manager that prints elapsed wall time for a step."""
    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        print(f"[{self.label}] {time.time() - self.t0:.1f}s")


# Validate AI.CLASSIFY category labels against the configured taxonomy (stock parse_classify_row marks any non-NULL
# classify_N string as passed_validation=True, even if empty or outside metric.categories). Idempotent.
if callable(getattr(_sdk_cat_eval_mod, "parse_classify_row", None)) and not getattr(
    _sdk_cat_eval_mod.parse_classify_row, "_bqca_wrapped", False
):
    _orig_parse_classify_row = _sdk_cat_eval_mod.parse_classify_row

    def _bqca_parse_classify_row(session_id, row, config):
        sr, null_count = _orig_parse_classify_row(session_id, row, config)
        lookup = {
            m.name: {c.name.lower().strip(): c.name for c in m.categories}
            for m in (getattr(config, "metrics", None) or ())
        }
        for mr in getattr(sr, "metrics", None) or ():
            if mr.category is not None:
                canonical = lookup.get(mr.metric_name, {}).get(str(mr.category).lower().strip())
                if canonical is not None:
                    mr.category = canonical
                    mr.passed_validation = True
                else:
                    mr.passed_validation = False
                    mr.parse_error = True
        return sr, null_count

    _bqca_parse_classify_row._bqca_wrapped = True
    _bqca_parse_classify_row._bqca_orig = _orig_parse_classify_row
    _sdk_cat_eval_mod.parse_classify_row = _bqca_parse_classify_row
    if hasattr(_sdk_client_mod, "parse_classify_row"):
        _sdk_client_mod.parse_classify_row = _bqca_parse_classify_row


def _normalize_categorical_report(report):
    """Excludes unvalidated / parse-errored categories (e.g. 'N/A') from report.category_distributions and summary."""
    if report is None:
        return report
    session_results = list(getattr(report, "session_results", None) or [])
    if not session_results:
        return report
    existing_dists = getattr(report, "category_distributions", None) or {}
    metric_names = list(existing_dists.keys())
    valid_counts = {m: {} for m in metric_names}
    total_slots = 0
    bad_slots = 0
    for sr in session_results:
        for mr in getattr(sr, "metrics", None) or ():
            mname = getattr(mr, "metric_name", None)
            if not mname:
                continue
            if mname not in valid_counts:
                metric_names.append(mname)
                valid_counts[mname] = {}
            total_slots += 1
            cat = getattr(mr, "category", None)
            is_valid = (
                bool(getattr(mr, "passed_validation", True))
                and not bool(getattr(mr, "parse_error", False))
                and cat is not None
                and bool(str(cat).strip())
            )
            if is_valid:
                cat_str = str(cat)
                valid_counts[mname][cat_str] = valid_counts[mname].get(cat_str, 0) + 1
            else:
                bad_slots += 1
    report.category_distributions = {
        m: dict(sorted(valid_counts.get(m, {}).items(), key=lambda x: -x[1]))
        for m in metric_names
    }
    details = getattr(report, "details", None)
    if isinstance(details, dict) and total_slots > 0:
        prev_errs = int(details.get("parse_errors", 0) or 0)
        err_count = max(prev_errs, bad_slots)
        details["parse_errors"] = err_count
        details["parse_error_rate"] = err_count / total_slots
    return report


if callable(getattr(_sdk_cat_eval_mod, "build_categorical_report", None)) and not getattr(
    _sdk_cat_eval_mod.build_categorical_report, "_bqca_wrapped", False
):
    _orig_build_categorical_report = _sdk_cat_eval_mod.build_categorical_report

    def _bqca_build_categorical_report(*args, **kwargs):
        return _normalize_categorical_report(_orig_build_categorical_report(*args, **kwargs))

    _bqca_build_categorical_report._bqca_wrapped = True
    _bqca_build_categorical_report._bqca_orig = _orig_build_categorical_report
    _sdk_cat_eval_mod.build_categorical_report = _bqca_build_categorical_report
    if hasattr(_sdk_client_mod, "build_categorical_report"):
        _sdk_client_mod.build_categorical_report = _bqca_build_categorical_report


if hasattr(_sdk_cat_eval_mod, "CategoricalEvaluationReport") and not getattr(
    _sdk_cat_eval_mod.CategoricalEvaluationReport.summary, "_bqca_wrapped", False
):
    _orig_cat_report_summary = _sdk_cat_eval_mod.CategoricalEvaluationReport.summary

    def _bqca_cat_report_summary(self):
        _normalize_categorical_report(self)
        return _orig_cat_report_summary(self)

    _bqca_cat_report_summary._bqca_wrapped = True
    _bqca_cat_report_summary._bqca_orig = _orig_cat_report_summary
    _sdk_cat_eval_mod.CategoricalEvaluationReport.summary = _bqca_cat_report_summary


if hasattr(Client, "evaluate_categorical") and not getattr(Client.evaluate_categorical, "_bqca_wrapped", False):
    _orig_client_evaluate_categorical = Client.evaluate_categorical

    def _bqca_client_evaluate_categorical(self, *args, **kwargs):
        return _normalize_categorical_report(_orig_client_evaluate_categorical(self, *args, **kwargs))

    _bqca_client_evaluate_categorical._bqca_wrapped = True
    _bqca_client_evaluate_categorical._bqca_orig = _orig_client_evaluate_categorical
    Client.evaluate_categorical = _bqca_client_evaluate_categorical


def score_frame(report):
    """Converts an SDK EvaluationReport into a per-session DataFrame, mapping per-session judge API/parse errors to None (NaN)."""
    _mark_warehouse_numeric_scores_validated(report)
    rows = []
    for s in report.session_scores:
        eval_err = bool((getattr(s, "details", None) or {}).get("evaluation_error"))
        fb_raw = s.llm_feedback or ""
        err_keys = set(_judge_error_keys(s))
        has_err = eval_err or bool(err_keys)
        row_scores = {}
        for k, v in (s.scores or {}).items():
            if v is None or eval_err or k in err_keys:
                row_scores[k] = None
                has_err = True
            else:
                row_scores[k] = round(float(v), 3)
        row = {"session_id": s.session_id, "passed": False if has_err else bool(s.passed)}
        row.update(row_scores)
        row["feedback"] = (
            fb_raw or str((getattr(s, "details", None) or {}).get("evaluation_error") or "")
        ).replace("\n", " ")
        rows.append(row)
    if not rows:
        return pd.DataFrame(columns=["session_id", "persona", "passed", "feedback"])
    out = pd.DataFrame(rows).sort_values("session_id").reset_index(drop=True)
    out.insert(1, "persona", out["session_id"].map(PERSONA_BY_SESSION).fillna("unattributed"))
    return out


def cat_frame(report):
    """Converts an SDK CategoricalEvaluationReport into a per-session DataFrame, mapping unvalidated categories (passed_validation=False) to None."""
    _normalize_categorical_report(report)
    rows = []
    for sr in report.session_results:
        for m in sr.metrics:
            is_valid = bool(m.passed_validation)
            rows.append({
                "session_id": sr.session_id,
                "metric": m.metric_name,
                "category": m.category if is_valid else None,
                "raw_category": m.category,
                "valid": is_valid,
                "justification": (m.justification or "").replace("\n", " "),
            })
    if not rows:
        return pd.DataFrame(columns=["session_id", "persona", "metric", "category", "raw_category", "valid", "justification"])
    out = pd.DataFrame(rows).sort_values("session_id").reset_index(drop=True)
    out.insert(1, "persona", out["session_id"].map(PERSONA_BY_SESSION).fillna("unattributed"))
    return out


def _is_row_mixed_fast_path(row):
    """Returns True when a row labeled is_fast_path also contains standard NL2SQL turn activity or LLM token usage."""
    if isinstance(row, dict):
        g = row.get
    else:
        g = lambda k, d=None: getattr(row, k, row[k] if hasattr(row, "__getitem__") and k in row else d)
    is_fp = bool(g("is_fast_path", False)) or int(g("fast_path_turns", 0) or 0) > 0
    if not is_fp:
        return False
    if bool(g("is_mixed_path", False)):
        return True
    fp_turns = g("fast_path_turns", None)
    st_turns = g("standard_turns", None)
    if fp_turns is not None and pd.notna(fp_turns) and st_turns is not None and pd.notna(st_turns):
        if int(fp_turns) > 0 and int(st_turns) > 0:
            return True
    turns_val = g("turns", None)
    if turns_val is not None and pd.notna(turns_val) and int(turns_val) > 1:
        if st_turns is None or pd.isna(st_turns) or int(st_turns) > 0:
            return True
    llm_val = g("llm_calls", None)
    if llm_val is not None and pd.notna(llm_val) and int(llm_val) > 0:
        return True
    for tok_col in ("total_tokens", "prompt_tokens", "candidate_tokens", "thinking_tokens"):
        tv = g(tok_col, None)
        if tv is not None and pd.notna(tv) and float(tv) > 0:
            return True
    return False


def _classify_session_execution_path(df):
    """Classifies session rows into 'Fast Path (Verified Queries)', 'Standard NL2SQL', or 'Mixed Session (Fast Path + Standard NL2SQL)'."""
    if df is None or df.empty:
        return pd.Series([], dtype="object")
    labels = []
    for _, row in df.iterrows():
        row_dict = row.to_dict()
        is_fp = bool(row_dict.get("is_fast_path", False)) or int(row_dict.get("fast_path_turns", 0) or 0) > 0
        if not is_fp:
            labels.append("Standard NL2SQL")
        elif _is_row_mixed_fast_path(row_dict):
            labels.append("Mixed Session (Fast Path + Standard NL2SQL)")
        else:
            labels.append("Fast Path (Verified Queries)")
    return pd.Series(labels, index=df.index, dtype="object")


def _compute_invocation_finops_df(events_or_inv_df, cost_rates=None):
    """Computes per-invocation (turn-level) Fast-Path vs. Standard NL2SQL latency, tokens, and cost from event or invocation rows."""
    rates = cost_rates or globals().get("COST_RATES") or {"input_per_1k": 0.00125, "output_per_1k": 0.01}
    r_in, r_out = float(rates["input_per_1k"]), float(rates["output_per_1k"])
    cols = [
        "session_id", "invocation_id", "is_fast_path", "execution_path",
        "llm_calls", "prompt_tokens", "candidate_tokens", "thinking_tokens",
        "cached_tokens", "total_tokens", "e2e_latency_ms",
        "cost_sdk_usd", "cost_incl_thinking_usd",
    ]
    if events_or_inv_df is None or len(events_or_inv_df) == 0:
        return pd.DataFrame(columns=cols)
    df = pd.DataFrame(events_or_inv_df).copy()
    # Case A: Raw event stream (has event_type column)
    if "event_type" in df.columns:
        groups = {}
        for idx, row in df.iterrows():
            rd = row.to_dict()
            sid = str(rd.get("session_id") or "")
            inv_raw = str(rd.get("invocation_id") or "").strip()
            inv_id = inv_raw or str(rd.get("timestamp") or idx)
            key = (sid, inv_id)
            g = groups.setdefault(key, {
                "session_id": sid,
                "invocation_id": inv_id,
                "is_fast_path": False,
                "llm_calls": 0,
                "prompt_tokens": 0,
                "candidate_tokens": 0,
                "thinking_tokens": 0,
                "cached_tokens": 0,
                "total_tokens": 0,
                "comp_latencies": [],
                "any_turn_event": False,
            })
            ev = str(rd.get("event_type") or "")
            if ev in ("USER_MESSAGE_RECEIVED", "AGENT_RESPONSE", "INVOCATION_COMPLETED", "LLM_RESPONSE"):
                g["any_turn_event"] = True
            attrs = rd.get("attributes")
            if isinstance(attrs, str) and attrs.strip().startswith("{"):
                try:
                    attrs = json.loads(attrs)
                except Exception:
                    attrs = {}
            elif not isinstance(attrs, dict):
                attrs = {}
            fp_attr = attrs.get("fast_path") if isinstance(attrs, dict) else None
            if str(fp_attr).lower() == "true" or bool(rd.get("is_fast_path", False)) or bool(rd.get("fast_path", False)):
                g["is_fast_path"] = True
            if ev == "INVOCATION_COMPLETED":
                lat = rd.get("latency_ms")
                if isinstance(lat, str) and lat.strip().startswith("{"):
                    try:
                        lat = json.loads(lat)
                    except Exception:
                        lat = {}
                tot_ms = lat.get("total_ms") if isinstance(lat, dict) else (lat if isinstance(lat, (int, float)) else rd.get("total_ms", rd.get("e2e_latency_ms")))
                if tot_ms is not None and pd.notna(tot_ms):
                    try:
                        g["comp_latencies"].append(float(tot_ms))
                    except (TypeError, ValueError):
                        pass
            elif ev == "LLM_RESPONSE":
                g["llm_calls"] += 1
                um = (attrs.get("usage_metadata") if isinstance(attrs, dict) else None) or {}
                p_tok = int(um.get("prompt_token_count", rd.get("prompt_tokens", 0)) or 0)
                c_tok = int(um.get("candidates_token_count", rd.get("candidate_tokens", 0)) or 0)
                t_tok = int(um.get("thoughts_token_count", rd.get("thinking_tokens", 0)) or 0)
                ca_tok = int(um.get("cached_content_token_count", rd.get("cached_tokens", 0)) or 0)
                tot_tok = int(um.get("total_token_count", rd.get("total_tokens", p_tok + c_tok + t_tok)) or 0)
                g["prompt_tokens"] += p_tok
                g["candidate_tokens"] += c_tok
                g["thinking_tokens"] += t_tok
                g["cached_tokens"] += ca_tok
                g["total_tokens"] += tot_tok
        out_rows = []
        for g in groups.values():
            if not g["any_turn_event"]:
                continue
            e2e_ms = max(g["comp_latencies"]) if g["comp_latencies"] else None
            c_sdk = round(g["prompt_tokens"] / 1000.0 * r_in + g["candidate_tokens"] / 1000.0 * r_out, 6)
            c_true = round(c_sdk + g["thinking_tokens"] / 1000.0 * r_out, 6)
            is_fp = bool(g["is_fast_path"])
            out_rows.append({
                "session_id": g["session_id"],
                "invocation_id": g["invocation_id"],
                "is_fast_path": is_fp,
                "execution_path": "Fast Path (Verified Queries)" if is_fp else "Standard NL2SQL",
                "llm_calls": g["llm_calls"],
                "prompt_tokens": g["prompt_tokens"],
                "candidate_tokens": g["candidate_tokens"],
                "thinking_tokens": g["thinking_tokens"],
                "cached_tokens": g["cached_tokens"],
                "total_tokens": g["total_tokens"],
                "e2e_latency_ms": e2e_ms,
                "cost_sdk_usd": c_sdk,
                "cost_incl_thinking_usd": c_true,
            })
        return pd.DataFrame(out_rows, columns=cols)
    # Case B: Already-grouped per-invocation rows (or SQL query output)
    out_rows = []
    for idx, row in df.iterrows():
        rd = row.to_dict()
        sid = str(rd.get("session_id") or f"s_{idx}")
        inv_id = str(rd.get("invocation_id") or rd.get("inv_key") or f"inv_{idx}")
        ex_path_raw = str(rd.get("execution_path") or "").strip()
        if "fast" in ex_path_raw.lower() and "mixed" not in ex_path_raw.lower():
            is_fp = True
        elif "standard" in ex_path_raw.lower():
            is_fp = False
        else:
            is_fp = bool(rd.get("is_fast_path", rd.get("fast_path", False)))
        p_tok = int(rd.get("prompt_tokens", 0) or 0)
        c_tok = int(rd.get("candidate_tokens", 0) or 0)
        t_tok = int(rd.get("thinking_tokens", 0) or 0)
        ca_tok = int(rd.get("cached_tokens", 0) or 0)
        tot_tok = int(rd.get("total_tokens", p_tok + c_tok + t_tok) or 0)
        llm_c = int(rd.get("llm_calls", 1 if tot_tok > 0 and not is_fp else 0) or 0)
        lat_raw = rd.get("e2e_latency_ms", rd.get("latency_ms", rd.get("total_ms", None)))
        e2e_ms = float(lat_raw) if (lat_raw is not None and pd.notna(lat_raw)) else None
        c_sdk = float(rd["cost_sdk_usd"]) if ("cost_sdk_usd" in rd and rd["cost_sdk_usd"] is not None and pd.notna(rd["cost_sdk_usd"])) else round(p_tok / 1000.0 * r_in + c_tok / 1000.0 * r_out, 6)
        c_true = float(rd["cost_incl_thinking_usd"]) if ("cost_incl_thinking_usd" in rd and rd["cost_incl_thinking_usd"] is not None and pd.notna(rd["cost_incl_thinking_usd"])) else round(c_sdk + t_tok / 1000.0 * r_out, 6)
        out_rows.append({
            "session_id": sid,
            "invocation_id": inv_id,
            "is_fast_path": is_fp,
            "execution_path": "Fast Path (Verified Queries)" if is_fp else "Standard NL2SQL",
            "llm_calls": llm_c,
            "prompt_tokens": p_tok,
            "candidate_tokens": c_tok,
            "thinking_tokens": t_tok,
            "cached_tokens": ca_tok,
            "total_tokens": tot_tok,
            "e2e_latency_ms": e2e_ms,
            "cost_sdk_usd": c_sdk,
            "cost_incl_thinking_usd": c_true,
        })
    return pd.DataFrame(out_rows, columns=cols)


def _prepare_fast_path_roi_df(finops_df, inv_df=None, cost_rates=None):
    """Prepares turn/session rows for Fast-Path vs. Standard NL2SQL comparison, separating or decomposing mixed sessions."""
    rates = cost_rates or globals().get("COST_RATES") or {"input_per_1k": 0.00125, "output_per_1k": 0.01}
    r_in, r_out = float(rates["input_per_1k"]), float(rates["output_per_1k"])
    if inv_df is not None and isinstance(inv_df, pd.DataFrame) and not inv_df.empty:
        return _compute_invocation_finops_df(inv_df, cost_rates=rates)
    if finops_df is None or len(finops_df) == 0:
        return pd.DataFrame(columns=["session_id", "execution_path", "llm_calls", "e2e_latency_ms", "total_tokens", "cost_incl_thinking_usd"])
    df = pd.DataFrame(finops_df).copy()
    # If df is already invocation-level (duplicate session_ids or explicit invocation_id with 1 turn per row), classify each row directly:
    if "event_type" in df.columns:
        return _compute_invocation_finops_df(df, cost_rates=rates)
    if "cost_sdk_usd" not in df.columns:
        p_s = df["prompt_tokens"] if "prompt_tokens" in df.columns else 0.0
        c_s = df["candidate_tokens"] if "candidate_tokens" in df.columns else 0.0
        df["cost_sdk_usd"] = p_s / 1000.0 * r_in + c_s / 1000.0 * r_out
    if "cost_incl_thinking_usd" not in df.columns:
        t_s = df["thinking_tokens"] if "thinking_tokens" in df.columns else 0.0
        df["cost_incl_thinking_usd"] = df["cost_sdk_usd"] + t_s / 1000.0 * r_out
    df["execution_path"] = _classify_session_execution_path(df)
    expanded_rows = []
    for _, row in df.iterrows():
        rd = row.to_dict()
        if rd.get("execution_path") == "Mixed Session (Fast Path + Standard NL2SQL)":
            fp_t = rd.get("fast_path_turns")
            st_t = rd.get("standard_turns")
            fp_lat = rd.get("fast_path_latency_ms")
            st_lat = rd.get("standard_latency_ms")
            # When v_bqca_customer_turns provides exact 1-turn fast-path & 1-turn standard latencies, decompose cleanly:
            if (
                fp_t is not None and pd.notna(fp_t) and int(fp_t) == 1
                and st_t is not None and pd.notna(st_t) and int(st_t) == 1
                and ((fp_lat is not None and pd.notna(fp_lat)) or (st_lat is not None and pd.notna(st_lat)))
            ):
                fp_row = dict(rd)
                fp_row["execution_path"] = "Fast Path (Verified Queries)"
                fp_row["is_fast_path"] = True
                fp_row["e2e_latency_ms"] = float(fp_lat) if (fp_lat is not None and pd.notna(fp_lat)) else None
                fp_row["llm_calls"] = 0
                fp_row["prompt_tokens"] = 0
                fp_row["candidate_tokens"] = 0
                fp_row["thinking_tokens"] = 0
                fp_row["cached_tokens"] = 0
                fp_row["total_tokens"] = 0
                fp_row["cost_sdk_usd"] = 0.0
                fp_row["cost_incl_thinking_usd"] = 0.0
                expanded_rows.append(fp_row)

                st_row = dict(rd)
                st_row["execution_path"] = "Standard NL2SQL"
                st_row["is_fast_path"] = False
                st_row["e2e_latency_ms"] = float(st_lat) if (st_lat is not None and pd.notna(st_lat)) else None
                expanded_rows.append(st_row)
                continue
        expanded_rows.append(rd)
    return pd.DataFrame(expanded_rows)


def _summarize_fast_path_roi(finops_or_inv_df, inv_df=None, cost_rates=None):
    """Builds (fast_path_cmp_df, s7_metrics_dict) from session-level finops_df and/or per-invocation inv_df."""
    roi_df = _prepare_fast_path_roi_df(finops_or_inv_df, inv_df=inv_df, cost_rates=cost_rates)
    if roi_df.empty:
        cmp_df = pd.DataFrame(
            columns=["execution_path", "sessions", "llm_calls", "e2e_min_ms", "e2e_p50_ms", "e2e_p90_ms", "median_total_tokens", "median_cost_incl_thinking_usd"]
        )
    else:
        cmp_df = (
            roi_df.groupby("execution_path")
            .agg(
                sessions=("session_id", "count"),
                llm_calls=("llm_calls", "sum"),
                e2e_min_ms=("e2e_latency_ms", "min"),
                e2e_p50_ms=("e2e_latency_ms", "median"),
                e2e_p90_ms=("e2e_latency_ms", lambda s: s.quantile(0.9)),
                median_total_tokens=("total_tokens", "median"),
                median_cost_incl_thinking_usd=("cost_incl_thinking_usd", "median"),
            )
            .round({"e2e_min_ms": 0, "e2e_p50_ms": 0, "e2e_p90_ms": 0, "median_total_tokens": 0, "median_cost_incl_thinking_usd": 4})
            .reset_index()
        )
    fc_idx = cmp_df.set_index("execution_path") if not cmp_df.empty else pd.DataFrame()
    fast_n = int(fc_idx.loc["Fast Path (Verified Queries)", "sessions"]) if "Fast Path (Verified Queries)" in fc_idx.index else 0
    std_n = int(fc_idx.loc["Standard NL2SQL", "sessions"]) if "Standard NL2SQL" in fc_idx.index else 0
    mixed_n = int(fc_idx.loc["Mixed Session (Fast Path + Standard NL2SQL)", "sessions"]) if "Mixed Session (Fast Path + Standard NL2SQL)" in fc_idx.index else 0
    if mixed_n == 0 and finops_or_inv_df is not None and isinstance(finops_or_inv_df, pd.DataFrame) and not finops_or_inv_df.empty and "event_type" not in finops_or_inv_df.columns:
        sess_paths = _classify_session_execution_path(finops_or_inv_df)
        mixed_n = int((sess_paths == "Mixed Session (Fast Path + Standard NL2SQL)").sum())
    _fp_raw = fc_idx.loc["Fast Path (Verified Queries)", "e2e_p50_ms"] if (fast_n > 0 and "Fast Path (Verified Queries)" in fc_idx.index) else None
    _st_raw = fc_idx.loc["Standard NL2SQL", "e2e_p50_ms"] if (std_n > 0 and "Standard NL2SQL" in fc_idx.index) else None
    fast_p50_ms = float(_fp_raw) if (_fp_raw is not None and pd.notna(_fp_raw)) else None
    std_p50_ms = float(_st_raw) if (_st_raw is not None and pd.notna(_st_raw)) else None
    _ftok_raw = fc_idx.loc["Fast Path (Verified Queries)", "median_total_tokens"] if (fast_n > 0 and "Fast Path (Verified Queries)" in fc_idx.index) else None
    fast_med_tokens = int(round(float(_ftok_raw))) if (_ftok_raw is not None and pd.notna(_ftok_raw)) else None
    _stok_raw = fc_idx.loc["Standard NL2SQL", "median_total_tokens"] if (std_n > 0 and "Standard NL2SQL" in fc_idx.index) else None
    std_med_tokens = int(round(float(_stok_raw))) if (_stok_raw is not None and pd.notna(_stok_raw)) else None
    s7_summary = {
        "fast_n": fast_n,
        "std_n": std_n,
        "mixed_n": mixed_n,
        "fast_p50_ms": fast_p50_ms,
        "std_p50_ms": std_p50_ms,
        "fast_med_tokens": fast_med_tokens,
        "std_med_tokens": std_med_tokens,
    }
    return cmp_df, s7_summary


def _format_fast_path_roi(s7):
    """Formats Section 7.2 takeaway line and Section 8.3 Row 7 fast-path summary without inventing a 0 ms baseline or attributing mixed sessions to Fast Path."""
    if not isinstance(s7, dict):
        return (
            "- **Verified Queries Fast-Path ROI**: fast-path vs. standard NL2SQL comparison **unavailable**.",
            "Fast-Path comparison **unavailable**",
        )
    f_n = int(s7.get("fast_n", 0) or 0)
    st_n = int(s7.get("std_n", 0) or 0)
    mx_n = int(s7.get("mixed_n", 0) or 0)
    f_p50 = s7.get("fast_p50_ms")
    st_p50 = s7.get("std_p50_ms")
    f_tok = s7.get("fast_med_tokens")
    f_ok = bool(f_n > 0 and f_p50 is not None and pd.notna(f_p50))
    st_ok = bool(st_n > 0 and st_p50 is not None and pd.notna(st_p50))
    tok_clause = f" with **{int(f_tok):,}** median LLM tokens" if (f_tok is not None and pd.notna(f_tok)) else ""
    if f_ok and st_ok:
        s7_line = (
            f"- **Verified Queries Fast-Path ROI**: Fast-path turns (`n={f_n}`) achieved a median end-to-end "
            f"latency of **{float(f_p50):,.0f} ms**{tok_clause}, compared with **{float(st_p50):,.0f} ms** "
            f"across standard NL2SQL turns (`n={st_n}`)."
        )
        s8_str = (
            f"Fast-Path median latency **{float(f_p50):,.0f} ms** (`n={f_n}`) vs. "
            f"**{float(st_p50):,.0f} ms** standard NL2SQL (`n={st_n}`)"
        )
    elif f_ok and not st_ok:
        if st_n == 0 and mx_n > 0:
            std_reason = f"no pure standard NL2SQL turns isolated (`n=0`; `{mx_n}` mixed fast-path + standard NL2SQL session(s) excluded from single-path latency comparison)"
            std_short = f"0 pure standard NL2SQL sessions, {mx_n} mixed session(s) excluded"
        elif st_n == 0:
            std_reason = "no standard NL2SQL turns observed (`n=0`)"
            std_short = "0 standard NL2SQL sessions"
        else:
            std_reason = f"standard NL2SQL latency telemetry unavailable (`n={st_n}`, all-NULL latency_ms)"
            std_short = f"standard NL2SQL latency unavailable, n={st_n}"
        s7_line = (
            f"- **Verified Queries Fast-Path ROI**: Fast-path turns (`n={f_n}`) achieved a median end-to-end "
            f"latency of **{float(f_p50):,.0f} ms**{tok_clause}; fast-path vs. standard NL2SQL comparison "
            f"**unavailable** ({std_reason})."
        )
        s8_str = (
            f"Fast-Path median latency **{float(f_p50):,.0f} ms** (`n={f_n}`); "
            f"standard NL2SQL comparison **unavailable** ({std_short})"
        )
    elif not f_ok and st_ok:
        if f_n == 0 and mx_n > 0:
            fast_reason = f"no pure fast-path turns isolated (`n=0`; `{mx_n}` mixed fast-path + standard NL2SQL session(s) excluded from single-path latency comparison)"
            fast_short = f"0 pure fast-path sessions, {mx_n} mixed session(s) excluded"
        elif f_n == 0:
            fast_reason = "no fast-path turns observed in this dataset (`n=0`)"
            fast_short = "0 fast-path sessions"
        else:
            fast_reason = f"fast-path latency telemetry unavailable (`n={f_n}`, all-NULL latency_ms)"
            fast_short = f"fast-path latency unavailable, n={f_n}"
        s7_line = (
            f"- **Verified Queries Fast-Path ROI**: {fast_reason}; fast-path vs. standard NL2SQL comparison "
            f"**unavailable** (standard NL2SQL median latency **{float(st_p50):,.0f} ms** across `n={st_n}` turns)."
        )
        s8_str = (
            f"Fast-Path comparison **unavailable** ({fast_short}; "
            f"standard NL2SQL median **{float(st_p50):,.0f} ms**, `n={st_n}`)"
        )
    else:
        mx_clause = f", `mixed_n={mx_n}`" if mx_n > 0 else ""
        s7_line = (
            f"- **Verified Queries Fast-Path ROI**: fast-path vs. standard NL2SQL latency comparison "
            f"**unavailable** (`fast_n={f_n}`, `std_n={st_n}`{mx_clause}; no measured end-to-end latency telemetry)."
        )
        s8_str = (
            f"Fast-Path vs. standard NL2SQL latency comparison **unavailable** "
            f"(`fast_n={f_n}`, `std_n={st_n}`{mx_clause}; no measured latency telemetry)"
        )
    return s7_line, s8_str


print(
    "Notebook helpers loaded:",
    ", ".join(["T", "run_sql", "md_table", "show", "short", "fmt", "timed", "score_frame", "cat_frame", "BQCAClient"]),
)

### 1.3 Provision the two customer views (`v_bqca_customer_sdk_events` and `v_bqca_customer_turns`)

Why create these two lightweight views directly over your raw `agent_events` table before running analytics?

1. **SDK Compatibility View (`v_bqca_customer_sdk_events`)**: BQCA logs `AGENT_RESPONSE.content.response` as a structured JSON object (`{"parts": [{"markdown": "..."}]}` or `{"clarifying_question": "..."}`), whereas `LLMAsJudge` in `bigquery-agent-analytics` v0.5.4 reads `JSON_VALUE(content, '$.response')` (which returns `NULL` on a JSON object unless flattened to a string). This view flattens `$.response` and `$.text_summary` to the scalar answer text (checking `$.text_summary`, ordered `$.response.parts[*].markdown`, `$.response.clarifying_question`, `$.parts[0].text`, and scalar `$.response`) while preserving the original nested object under `$.raw_response`, backfills a missing `attributes.root_agent_name` from the session's unique explicitly-logged root (falling back to `agent` when the session has a single agent value) so the common missing-root case resolves in `client.get_session_trace()` (sessions whose events still carry conflicting explicit roots raise `AmbiguousSessionError` — pass an explicit `root_agent_name` selector or fix the source telemetry), and uses `SELECT * EXCEPT(...) REPLACE(...)` so it works seamlessly on both 17-column BQCA tables (with `event_id`) and 16-column standard SDK tables.
2. **Conversation Turns View (`v_bqca_customer_turns`)**: Analysts and BI dashboards usually want **one row per session** pairing the initial non-empty `user_prompt` (`NULLIF(TRIM(JSON_VALUE(content, '$.text_summary')), '')`) with the earliest non-clarifying `agent_response` (a session whose only agent turns are clarifying questions shows the earliest clarifying question instead; the earliest non-empty clarifying text is also surfaced separately in `clarifying_question`) with `clarifying_question`, `is_fast_path` (plus `fast_path_turns`, `standard_turns`, `is_mixed_path`, `fast_path_latency_ms`, and `standard_latency_ms` so multi-turn sessions that mix Verified Queries Fast-Path and standard NL2SQL turns are separated before computing single-path turn latency/cost), `e2e_latency_ms` (summed across distinct `invocation_id`s using the per-invocation `MAX(INVOCATION_COMPLETED.latency_ms.total_ms)` — partitioned by `(session_id, event_type = 'INVOCATION_COMPLETED', invocation_id)` so non-completion events never compete with `INVOCATION_COMPLETED` rows while duplicate completion rows emitted for the same invocation are not double-counted), token counts (`prompt_tokens`, `candidate_tokens`, `thinking_tokens`, `cached_tokens`, `total_tokens`), estimated token costs (`cost_sdk_usd`, `cost_incl_thinking_usd` at your configured `COST_RATES`), `finish_reasons`, and `error_messages`. Defining both views directly over `RAW_TABLE_ID` (scoped by `EVAL_LOOKBACK_DAYS`) avoids nested view-over-view overhead when running `AI.GENERATE` or `AI.EMBED`.

The next cell creates or updates both views in your dataset.

In [ ]:
#@title 1.3 Provision Customer Views (v_bqca_customer_sdk_events & v_bqca_customer_turns)
event_types_sql = ", ".join(f"'{t}'" for t in BQCA_EVENT_TYPES)
lookback_days = int(EVAL_LOOKBACK_DAYS)
cost_rate_in = float(COST_RATES["input_per_1k"])
cost_rate_out = float(COST_RATES["output_per_1k"])
user_id_proj_sql = (
    "IF(NULLIF(TRIM(ANY_VALUE(user_id)), '') IS NOT NULL, CONCAT('user_', SUBSTR(TO_HEX(SHA256(CAST(TRIM(ANY_VALUE(user_id)) AS BYTES))), 1, 8)), NULL)"
    if ANONYMIZE_USER_ID
    else "ANY_VALUE(user_id)"
)
user_cohort_fallback_sql = (
    "IF(NULLIF(TRIM(ANY_VALUE(user_id)), '') IS NOT NULL, CONCAT('user_', SUBSTR(TO_HEX(SHA256(CAST(TRIM(ANY_VALUE(user_id)) AS BYTES))), 1, 8)), NULL)"
    if ANONYMIZE_USER_ID
    else "NULLIF(SPLIT(TRIM(ANY_VALUE(user_id)), '@')[SAFE_OFFSET(0)], '')"
)

_persona_raw_sql = (
    "COALESCE(\n"
    "    NULLIF(MAX(JSON_VALUE(attributes, '$.session_metadata.state.custom_labels.persona')), ''),\n"
    f"    {user_cohort_fallback_sql},\n"
    "    NULLIF(MAX(JSON_VALUE(attributes, '$.session_metadata.state.\"data-agent-id\"')), ''),\n"
    "    'unattributed'\n"
    "  )"
)
# Anonymize user_id in turns_df when ANONYMIZE_USER_ID is True (including explicit persona
# labels that contain '@', so cohort displays never print raw email addresses):
persona_sql = (
    f"IF(REGEXP_CONTAINS({_persona_raw_sql}, '@'), "
    f"CONCAT('user_', SUBSTR(TO_HEX(SHA256(CAST({_persona_raw_sql} AS BYTES))), 1, 8)), "
    f"{_persona_raw_sql})"
    if ANONYMIZE_USER_ID
    else _persona_raw_sql
)

CREATE_SDK_VIEW_SQL = f"""
CREATE OR REPLACE VIEW {T(COMPAT_VIEW_ID)} AS
WITH filtered AS (
  SELECT
    *,
    MIN(JSON_VALUE(attributes, '$.root_agent_name')) OVER (PARTITION BY session_id) AS _min_session_root,
    MAX(JSON_VALUE(attributes, '$.root_agent_name')) OVER (PARTITION BY session_id) AS _max_session_root,
    MIN(agent) OVER (PARTITION BY session_id) AS _min_session_agent,
    MAX(agent) OVER (PARTITION BY session_id) AS _max_session_agent,
    IF(
      event_type = 'AGENT_RESPONSE' AND content IS NOT NULL,
      COALESCE(
        NULLIF(JSON_VALUE(content, '$.text_summary'), ''),
        NULLIF(ARRAY_TO_STRING(ARRAY(
          SELECT JSON_VALUE(part, '$.markdown')
          FROM UNNEST(JSON_QUERY_ARRAY(content, '$.response.parts')) AS part WITH OFFSET AS part_offset
          WHERE JSON_VALUE(part, '$.markdown') IS NOT NULL
          ORDER BY part_offset
        ), '\\n\\n'), ''),
        NULLIF(TRIM(JSON_VALUE(content, '$.response.clarifying_question')), ''),
        NULLIF(JSON_VALUE(content, '$.parts[0].text'), ''),
        JSON_VALUE(content, '$.response')
      ),
      NULL
    ) AS answer_text
  FROM {T(RAW_TABLE_ID)}
  WHERE event_type IN ({event_types_sql})
    AND timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {lookback_days} DAY)
)
SELECT
  * EXCEPT(answer_text, _min_session_root, _max_session_root, _min_session_agent, _max_session_agent)
  REPLACE (
    CASE
      WHEN event_type = 'AGENT_RESPONSE' AND content IS NOT NULL THEN
        JSON_SET(
          content,
          '$.raw_response', JSON_QUERY(content, '$.response'),
          '$.text_summary', answer_text,
          '$.response', answer_text
        )
      ELSE content
    END AS content,
    IF(
      JSON_VALUE(attributes, '$.root_agent_name') IS NULL AND agent IS NOT NULL,
      JSON_SET(
        COALESCE(attributes, JSON '{{}}'),
        '$.root_agent_name',
        COALESCE(
          IF(_min_session_root = _max_session_root, _max_session_root, NULL),
          IF(_min_session_agent = _max_session_agent, agent, NULL)
        )
      ),
      attributes
    ) AS attributes
  )
FROM filtered
"""

CREATE_TURNS_VIEW_SQL = f"""
CREATE OR REPLACE VIEW {T(TURNS_VIEW_ID)} AS
WITH base AS (
  SELECT
    *,
    LOGICAL_OR(COALESCE(JSON_VALUE(attributes, '$.fast_path') = 'true', FALSE)) OVER (
      PARTITION BY
        session_id,
        IFNULL(NULLIF(TRIM(invocation_id), ''), CAST(timestamp AS STRING))
    ) AS _inv_is_fast_path,
    IF(
      event_type = 'INVOCATION_COMPLETED',
      ROW_NUMBER() OVER (
        PARTITION BY
          session_id,
          event_type = 'INVOCATION_COMPLETED',
          IFNULL(NULLIF(TRIM(invocation_id), ''), CAST(timestamp AS STRING))
        ORDER BY
          (event_type = 'INVOCATION_COMPLETED') DESC,
          SAFE_CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64) DESC NULLS LAST,
          timestamp DESC
      ),
      NULL
    ) AS _inv_complete_rn
  FROM {T(RAW_TABLE_ID)}
  WHERE event_type IN ({event_types_sql})
    AND timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {lookback_days} DAY)
)
SELECT
  session_id,
  ARRAY_AGG(invocation_id IGNORE NULLS ORDER BY timestamp ASC LIMIT 1)[SAFE_OFFSET(0)] AS invocation_id,
  {user_id_proj_sql} AS user_id,
  {persona_sql} AS persona,
  MAX(JSON_VALUE(attributes, '$.session_metadata.state.custom_labels.persona_role')) AS persona_role,
  MAX(JSON_VALUE(attributes, '$.session_metadata.state.custom_labels.scenario_tag')) AS scenario_tag,
  MAX(JSON_VALUE(attributes, '$.session_metadata.state."data-agent-id"')) AS data_agent_id,
  MAX(JSON_VALUE(attributes, '$.session_metadata.state."conversation-id"')) AS conversation_id,
  MIN(timestamp) AS started_at,
  COUNT(*) AS event_count,
  COUNTIF(event_type = 'USER_MESSAGE_RECEIVED') AS turns,
  ARRAY_AGG(
    IF(
      event_type = 'USER_MESSAGE_RECEIVED',
      NULLIF(TRIM(JSON_VALUE(content, '$.text_summary')), ''),
      NULL
    ) IGNORE NULLS
    ORDER BY timestamp ASC
    LIMIT 1
  )[SAFE_OFFSET(0)] AS user_prompt,
  ARRAY_AGG(
    IF(
      event_type = 'AGENT_RESPONSE' AND content IS NOT NULL,
      COALESCE(
        NULLIF(JSON_VALUE(content, '$.text_summary'), ''),
        NULLIF(ARRAY_TO_STRING(ARRAY(
          SELECT JSON_VALUE(part, '$.markdown')
          FROM UNNEST(JSON_QUERY_ARRAY(content, '$.response.parts')) AS part WITH OFFSET AS part_offset
          WHERE JSON_VALUE(part, '$.markdown') IS NOT NULL
          ORDER BY part_offset
        ), '\\n\\n'), ''),
        NULLIF(TRIM(JSON_VALUE(content, '$.response.clarifying_question')), ''),
        NULLIF(JSON_VALUE(content, '$.parts[0].text'), ''),
        JSON_VALUE(content, '$.response')
      ),
      NULL
    )
    IGNORE NULLS ORDER BY
      CASE
        WHEN event_type = 'AGENT_RESPONSE' AND content IS NOT NULL
          AND NULLIF(JSON_VALUE(content, '$.text_summary'), '') IS NULL
          AND NULLIF(ARRAY_TO_STRING(ARRAY(
            SELECT JSON_VALUE(part, '$.markdown')
            FROM UNNEST(JSON_QUERY_ARRAY(content, '$.response.parts')) AS part WITH OFFSET AS part_offset
            WHERE JSON_VALUE(part, '$.markdown') IS NOT NULL
            ORDER BY part_offset
          ), '\\n\\n'), '') IS NULL
          AND NULLIF(TRIM(JSON_VALUE(content, '$.response.clarifying_question')), '') IS NOT NULL
        THEN 1
        ELSE 0
      END ASC,
      timestamp ASC
    LIMIT 1
  )[SAFE_OFFSET(0)] AS agent_response,
  ARRAY_AGG(
    IF(
      event_type = 'AGENT_RESPONSE',
      NULLIF(TRIM(JSON_VALUE(content, '$.response.clarifying_question')), ''),
      NULL
    ) IGNORE NULLS
    ORDER BY timestamp ASC
    LIMIT 1
  )[SAFE_OFFSET(0)] AS clarifying_question,
  LOGICAL_OR(COALESCE(JSON_VALUE(attributes, '$.fast_path') = 'true', FALSE)) AS is_fast_path,
  COUNTIF(event_type = 'INVOCATION_COMPLETED' AND _inv_complete_rn = 1 AND _inv_is_fast_path) AS fast_path_turns,
  COUNTIF(event_type = 'INVOCATION_COMPLETED' AND _inv_complete_rn = 1 AND NOT _inv_is_fast_path) AS standard_turns,
  (
    LOGICAL_OR(_inv_is_fast_path)
    AND (
      LOGICAL_OR(IF(event_type IN ('INVOCATION_COMPLETED', 'USER_MESSAGE_RECEIVED', 'LLM_RESPONSE'), NOT _inv_is_fast_path, FALSE))
      OR COUNTIF(event_type = 'LLM_RESPONSE') > 0
      OR COUNTIF(event_type = 'USER_MESSAGE_RECEIVED') > 1
    )
  ) AS is_mixed_path,
  SUM(IF(event_type = 'INVOCATION_COMPLETED' AND _inv_complete_rn = 1 AND _inv_is_fast_path, SAFE_CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), NULL)) AS fast_path_latency_ms,
  SUM(IF(event_type = 'INVOCATION_COMPLETED' AND _inv_complete_rn = 1 AND NOT _inv_is_fast_path, SAFE_CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), NULL)) AS standard_latency_ms,
  -- Sum per-invocation MAX(total_ms) across distinct invocation_ids so duplicate completion rows within an invocation do not double-count while multi-turn sessions sum across turns:
  SUM(IF(event_type = 'INVOCATION_COMPLETED' AND _inv_complete_rn = 1, SAFE_CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), NULL)) AS e2e_latency_ms,
  AVG(SAFE_CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64)) AS avg_event_latency_ms,
  AVG(SAFE_CAST(JSON_VALUE(latency_ms, '$.time_to_first_token_ms') AS FLOAT64)) AS avg_ttft_ms,
  COUNTIF(event_type = 'LLM_RESPONSE') AS llm_calls,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.prompt_token_count') AS INT64), 0)), 0) AS prompt_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64), 0)), 0) AS candidate_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.thoughts_token_count') AS INT64), 0)), 0) AS thinking_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.cached_content_token_count') AS INT64), 0)), 0) AS cached_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.total_token_count') AS INT64), 0)), 0) AS total_tokens,
  ROUND(
    COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.prompt_token_count') AS INT64), 0)), 0) / 1000.0 * {cost_rate_in}
    + COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64), 0)), 0) / 1000.0 * {cost_rate_out},
    6
  ) AS cost_sdk_usd,
  ROUND(
    COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.prompt_token_count') AS INT64), 0)), 0) / 1000.0 * {cost_rate_in}
    + (
      COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64), 0)), 0)
      + COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.thoughts_token_count') AS INT64), 0)), 0)
    ) / 1000.0 * {cost_rate_out},
    6
  ) AS cost_incl_thinking_usd,
  STRING_AGG(DISTINCT IF(event_type = 'LLM_RESPONSE', COALESCE(JSON_VALUE(attributes, '$.finish_reason'), '(missing)'), NULL), ', ') AS finish_reasons,
  COALESCE(LOGICAL_OR(ENDS_WITH(event_type, '_ERROR') OR error_message IS NOT NULL OR status = 'ERROR'), FALSE) AS has_error,
  STRING_AGG(DISTINCT error_message, ' | ') AS error_messages
FROM base
GROUP BY session_id
"""

bq.query(CREATE_SDK_VIEW_SQL).result()
bq.query(CREATE_TURNS_VIEW_SQL).result()
print(f"Provisioned SDK compatibility view: {T(COMPAT_VIEW_ID)} (lookback={lookback_days}d)")
print(f"Provisioned flat conversation turns view: {T(TURNS_VIEW_ID)} (lookback={lookback_days}d)")

verify_norm_df = run_sql(f"""
SELECT
  (SELECT COUNT(*) FROM {T(RAW_TABLE_ID)}) AS raw_rows,
  COUNT(*) AS sdk_view_rows,
  COUNT(DISTINCT session_id) AS distinct_sessions,
  COUNTIF(event_type = 'AGENT_RESPONSE') AS agent_response_rows,
  COUNTIF(event_type = 'AGENT_RESPONSE' AND JSON_VALUE(content, '$.response') IS NOT NULL) AS scalar_response_ok,
  COUNTIF(event_type = 'AGENT_RESPONSE' AND JSON_VALUE(content, '$.text_summary') IS NOT NULL) AS text_summary_ok,
  COUNTIF(event_type = 'AGENT_RESPONSE' AND COALESCE(TO_JSON_STRING(JSON_QUERY(content, '$.raw_response')), 'null') != 'null') AS raw_response_preserved
FROM {T(COMPAT_VIEW_ID)}
""")
RESULTS["s1_total_sessions"] = int(verify_norm_df["distinct_sessions"].iloc[0]) if not verify_norm_df.empty else 0
display(verify_norm_df)

### 1.4 Initialize the SDK `Client`, run `client.doctor()` & ping Vertex AI connection

`client.doctor()` verifies the 16 required SDK schema columns (plus BQCA's 17th `event_id` column when present), confirms that `endpoint` and `connection_id` are set on the client, and reports event counts by `event_type`. Because BQCA intentionally omits low-level ADK tool and agent spans (`AGENT_STARTING`, `AGENT_COMPLETED`, `LLM_REQUEST`, `TOOL_STARTING`, `TOOL_COMPLETED`), `doctor()` will list those non-BQCA event types under `warnings` — that is expected on production BQCA logs. We then execute a live 1-row `AI.GENERATE` ping against `JUDGE_MODEL` (`gemini-3.5-flash`) and check both `.verdict` and `.status` to verify your Cloud Resource Connection IAM end-to-end.

In [ ]:
#@title 1.4 Run client.doctor() & 1-Row Vertex AI Connection Ping
# BQCAClient extends bigquery_agent_analytics.Client so that if an in-warehouse AI.GENERATE
# judge call in client.evaluate(LLMAsJudge.*) is cancelled or falls back to direct Gemini API
# execution (execution_mode='api_fallback'), _run_api_judge still receives the full BQCA
# AGENT_RESPONSE answer text rather than an empty Trace.final_response:
client = BQCAClient(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=COMPAT_VIEW_ID,
    location=LOCATION,
    bq_client=bq,
    endpoint=JUDGE_MODEL,
    connection_id=VERTEX_CONNECTION_ID,
)

doc = client.doctor()
cols = doc["schema"].get("columns") or doc["schema"].get("present", [])
n_cols = len(cols)
cov = doc.get("event_coverage", {})
if "error" in cov:
    raise RuntimeError(f"client.doctor() failed to query {COMPAT_VIEW_ID}: {cov['error']}")
total_events = sum(v for k, v in cov.items() if isinstance(v, (int, float)))
print(f"Schema status: {doc['schema']['status']} ({n_cols} columns) | Total events in {COMPAT_VIEW_ID}: {total_events:,}")
print(f"AI.GENERATE endpoint: {doc['ai_generate']['endpoint']} | Connection ID set: {bool(doc['ai_generate']['connection_id'])}")
for w in doc.get("warnings", []):
    if w.startswith("No events for types:"):
        print(f"  Expected on BQCA logs (non-BQCA ADK event types): {w}")
    else:
        print(f"  WARNING: {w}")

RESULTS["s1"] = {
    "schema_status": doc["schema"]["status"],
    "columns": n_cols,
    "total_events": int(total_events),
    "active_event_types": sum(1 for v in cov.values() if isinstance(v, (int, float)) and v > 0),
    "agent_response_rows": int(verify_norm_df.agent_response_rows.iloc[0]),
    "scalar_responses": int(verify_norm_df.scalar_response_ok.iloc[0]),
    "ping_verdict": "not_attempted",
}

# Live 1-row AI.GENERATE connection & IAM ping (checking both .verdict and .status).
# Mark ping_verdict = "failed" immediately before the query so a timeout or cancellation
# is never misreported as "not_attempted" in Section 8.3:
RESULTS["s1"]["ping_verdict"] = "failed"
_ping_timeout_sec = min(float(AI_SQL_TIMEOUT_SEC), 90.0)
_allow_ping_fallback = os.environ.get("BQCA_PING_API_FALLBACK", "true").strip().lower() not in ("0", "false", "no")
try:
    ping_df = run_sql(
        f"""
        SELECT
          gen.verdict AS ai_generate_ping,
          gen.status AS ai_generate_status
        FROM (
          SELECT AI.GENERATE(
            prompt => 'Reply with OK if Vertex AI connection is healthy.',
            connection_id => '{VERTEX_CONNECTION_ID}',
            endpoint => '{JUDGE_MODEL}',
            model_params => JSON '{{"generationConfig": {{"temperature": 0.0, "maxOutputTokens": 256}}}}',
            output_schema => 'verdict STRING'
          ) AS gen
        )
        """,
        retries=0,
        timeout_sec=_ping_timeout_sec,
    )
    ping_verdict = ping_df.ai_generate_ping.iloc[0]
    ping_status = ping_df.ai_generate_status.iloc[0] if "ai_generate_status" in ping_df.columns else ""
    if pd.isna(ping_verdict) or not str(ping_verdict).strip() or (pd.notna(ping_status) and str(ping_status).strip()):
        RESULTS["s1"]["ping_verdict"] = f"failed (status={ping_status!r})"
        raise RuntimeError(
            f"AI.GENERATE health ping failed on {JUDGE_MODEL}: verdict={ping_verdict!r}, status={ping_status!r}. "
            "Verify VERTEX_CONNECTION_ID and roles/aiplatform.user on the connection service account."
        )
    RESULTS["s1"]["ping_verdict"] = str(ping_verdict).strip()
    print(f"Live 1-row AI.GENERATE ping ({JUDGE_MODEL}): verdict={ping_verdict!r}, status={ping_status!r}")
except Exception as _ping_exc:
    if RESULTS["s1"]["ping_verdict"] == "failed":
        RESULTS["s1"]["ping_verdict"] = f"failed ({type(_ping_exc).__name__})"
    if _allow_ping_fallback and isinstance(
        _ping_exc,
        (
            TimeoutError,
            concurrent.futures.TimeoutError,
            gcp_exceptions.DeadlineExceeded,
            gcp_exceptions.Cancelled,
            gcp_exceptions.ServiceUnavailable,
            gcp_exceptions.TooManyRequests,
        ),
    ):
        if getattr(_ping_exc, "cancel_confirmed", True) is False:
            print(
                f"⚠️ WARNING: warehouse ping job_id={getattr(_ping_exc, 'job_id', None) or 'unknown'} timed out and its cancellation "
                "could not be confirmed (see the warning above), so it may still be RUNNING and billing in BigQuery while the "
                "direct Vertex AI ping below also runs."
            )
        try:
            from google import genai as _genai
            from google.genai import types as _genai_types

            _direct_client = _genai.Client(vertexai=True, project=PROJECT_ID, location=VERTEX_LOCATION)
            try:
                _direct_resp = _direct_client.models.generate_content(
                    model=DIRECT_JUDGE_MODEL,
                    contents="Reply with OK if Vertex AI connection is healthy.",
                    config=_genai_types.GenerateContentConfig(temperature=0.0, max_output_tokens=64),
                )
            finally:
                _direct_client.close()
            _direct_text = _validate_direct_ping_response(
                _direct_resp, model_name=DIRECT_JUDGE_MODEL, location=VERTEX_LOCATION
            )
            _direct_short = _direct_text.splitlines()[0][:24]
            RESULTS["s1"]["ping_verdict"] = (
                f"{_direct_short} (api_fallback; warehouse_ping={type(_ping_exc).__name__})"
            )
            print(
                f"⚠️ In-warehouse AI.GENERATE ping did not complete within {_ping_timeout_sec:g}s "
                f"({type(_ping_exc).__name__}: {_ping_exc}). Direct Vertex AI ping ({DIRECT_JUDGE_MODEL} in "
                f"{VERTEX_LOCATION}) succeeded with {_direct_short!r} — downstream judge cells will automatically "
                f"use execution_mode='api_fallback' if warehouse AI.GENERATE jobs time out."
            )
        except Exception as _direct_exc:
            RESULTS["s1"]["ping_verdict"] = (
                f"failed (warehouse={type(_ping_exc).__name__}, direct={type(_direct_exc).__name__})"
            )
            raise _ping_exc from _direct_exc
    else:
        raise

all_types = sorted(set(BQCA_EVENT_TYPES) | set(cov))
cov_df = pd.DataFrame({
    "event_type": all_types,
    "in_bqca_event_contract": ["yes" if t in BQCA_EVENT_TYPES else "no" for t in all_types],
    "event_count": [cov.get(t, 0) for t in all_types],
}).sort_values(["in_bqca_event_contract", "event_count"], ascending=[False, False]).reset_index(drop=True)
display(cov_df)

_ping_takeaway = (
    f"the 1-row `AI.GENERATE` ping on `{JUDGE_MODEL}` returned `{RESULTS['s1']['ping_verdict']}` (`status=''`)"
    if "api_fallback" not in str(RESULTS["s1"]["ping_verdict"])
    else f"in-warehouse `AI.GENERATE` ping timed out/cancelled while direct Vertex AI (`{DIRECT_JUDGE_MODEL}`) verified `{RESULTS['s1']['ping_verdict']}`"
)
show(f"""**Section 1 Takeaway (live)**

- `client.doctor()` confirmed schema `{RESULTS['s1']['schema_status']}` (**{RESULTS['s1']['columns']}** columns) with **{RESULTS['s1']['total_events']}** events across **{RESULTS['s1']['active_event_types']}** active BQCA event types, and {_ping_takeaway}.
- In `{COMPAT_VIEW_ID}`, **{RESULTS['s1']['scalar_responses']}/{RESULTS['s1']['agent_response_rows']}** `AGENT_RESPONSE` rows expose both scalar `$.response` and `$.text_summary` while keeping the original nested payload in `$.raw_response`.""")

## 2. Prompt & Response Conversation Explorer (`v_bqca_customer_turns`)

Before running LLM judges, inspect `v_bqca_customer_turns` to see what your users are asking, how your agents are responding, how often the agent asks a clarifying question (`clarifying_question`), and which turns hit the Verified Queries fast path (`is_fast_path = TRUE`).

In [ ]:
#@title 2.1 Conversation Explorer & Cohort Summary
MAX_EXPLORER_SESSIONS = int(MAX_EVAL_SESSIONS) * 10
turns_df = (
    run_sql(f"SELECT * FROM {T(TURNS_VIEW_ID)} ORDER BY started_at DESC, session_id LIMIT {MAX_EXPLORER_SESSIONS}")
    .sort_values(["persona", "session_id"])
    .reset_index(drop=True)
)
if turns_df.empty:
    raise RuntimeError(
        f"No conversation turns found in {T(TURNS_VIEW_ID)} within the last {EVAL_LOOKBACK_DAYS} days. "
        f"Verify that {T(RAW_TABLE_ID)} contains BQCA events or increase EVAL_LOOKBACK_DAYS in Cell 1.1."
    )
PERSONA_BY_SESSION.update(dict(zip(turns_df["session_id"], turns_df["persona"])))
EVAL_SESSION_IDS = (
    turns_df.sort_values(["started_at", "session_id"], ascending=[False, True])["session_id"]
    .head(int(MAX_EVAL_SESSIONS))
    .tolist()
)
EVAL_FILTER = TraceFilter(session_ids=EVAL_SESSION_IDS, limit=int(MAX_EVAL_SESSIONS))
single_turn_sessions = int((turns_df["turns"] == 1).sum())
multi_turn_sessions = int((turns_df["turns"] > 1).sum())
print(
    f"Loaded {len(turns_df)} sessions from {TURNS_VIEW_ID} (cap={MAX_EXPLORER_SESSIONS}): "
    f"{single_turn_sessions} single-turn (turns == 1), {multi_turn_sessions} multi-turn (turns > 1)."
)

turns_df["is_fast_path"] = turns_df["is_fast_path"].fillna(False).astype(bool)
turns_df["has_error"] = turns_df["has_error"].fillna(False).astype(bool)
cohort_summary_df = (
    turns_df.groupby(["data_agent_id", "persona", "persona_role"], dropna=False)
    .agg(
        sessions=("session_id", "count"),
        answered_turns=("agent_response", lambda s: int(s.notna().sum())),
        clarifying_turns=("clarifying_question", lambda s: int(s.notna().sum())),
        fast_path_turns=("is_fast_path", lambda s: int(s.fillna(False).astype(bool).astype(int).sum())),
        error_turns=("has_error", lambda s: int(s.fillna(False).astype(bool).astype(int).sum())),
        median_e2e_latency_ms=("e2e_latency_ms", "median"),
        total_tokens=("total_tokens", "sum"),
    )
    .astype({
        "sessions": int,
        "answered_turns": int,
        "clarifying_turns": int,
        "fast_path_turns": int,
        "error_turns": int,
        "total_tokens": int,
    })
    .round({"median_e2e_latency_ms": 0})
    .reset_index()
)
print("Conversation summary by data_agent_id and user cohort:")
display(cohort_summary_df)

sample_turns_df = (
    turns_df.groupby("persona").head(1)[
        ["persona", "session_id", "is_fast_path", "has_error", "e2e_latency_ms", "total_tokens", "user_prompt", "agent_response"]
    ].reset_index(drop=True)
)
show("**Sample paired `user_prompt` -> `agent_response` per user cohort**\n\n" + md_table(sample_turns_df, max_chars=150))

clarify_df = turns_df[turns_df.clarifying_question.notna()][
    ["persona", "session_id", "user_prompt", "clarifying_question"]
].reset_index(drop=True)
if not clarify_df.empty:
    print(f"Turns where the agent asked a clarifying question ({len(clarify_df)} total, showing up to 25):")
    display(clarify_df.head(25))

RESULTS["s2"] = {
    "sessions": len(turns_df),
    "cohorts": int(turns_df.persona.nunique()),
    "answered": int(turns_df.agent_response.notna().sum()),
    "clarifying": int(turns_df.clarifying_question.notna().sum()),
    "fast_path": int(turns_df.is_fast_path.astype(bool).sum()),
    "errors": int(turns_df.has_error.astype(bool).sum()),
}
show(f"""**Section 2 Takeaway (live)**

- `{TURNS_VIEW_ID}` pairs **{RESULTS['s2']['sessions']}** sessions across **{RESULTS['s2']['cohorts']}** user cohorts: **{RESULTS['s2']['answered']}** delivered an `AGENT_RESPONSE` (including **{RESULTS['s2']['clarifying']}** clarifying-question turn(s)), **{RESULTS['s2']['fast_path']}** used the Verified Queries fast path, and **{RESULTS['s2']['errors']}** ended with a sanitised error event.""")

## 3. Customer sentiment & frustrated-conversation triage

Two complementary SDK methods let you monitor user satisfaction directly inside BigQuery (`AI.GENERATE` / `AI.CLASSIFY`):

- **3.1 `LLMAsJudge.sentiment()`**: Scores overall interaction satisfaction and helpfulness on `[0.0, 1.0]`. Joining those scores to `v_bqca_customer_turns` gives your support and product teams an immediate triage queue of low-scoring conversations with the exact prompt, response, and judge explanation.
- **3.2 `evaluate_categorical()` Tone Rubric**: Classifies each conversation into your own UX buckets (`Positive_Satisfied`, `Neutral_Analytical`, `Clarification_Needed`, `Frustrated_Impatient`) with a written justification.

In [ ]:
#@title 3.1 Sentiment Evaluation (LLMAsJudge.sentiment)
with timed(f"LLMAsJudge.sentiment via client.evaluate ({JUDGE_MODEL})"):
    sentiment_report = client.evaluate(
        LLMAsJudge.sentiment(threshold=JUDGE_THRESHOLD, model=JUDGE_MODEL),
        filters=EVAL_FILTER,
    )
_recompute_eval_report_counts(sentiment_report)  # counts over the evaluated denominator; judge errors reported separately
print(sentiment_report.summary())
print("Execution details:", {k: sentiment_report.details.get(k) for k in ("execution_mode", "fallback_reason", "parse_errors") if sentiment_report.details.get(k) is not None})
sent_df = score_frame(sentiment_report)
if "sentiment" not in sent_df.columns:
    sent_df["sentiment"] = pd.NA
sent_mode = sentiment_report.details.get("execution_mode", "unknown")
sent_not_evaluated = int(sent_df["sentiment"].isna().sum())
sent_evaluated = int(sent_df["sentiment"].notna().sum())
sent_eval_passed = int((sent_df["sentiment"].notna() & sent_df["passed"]).sum())
sent_eval_failed = int((sent_df["sentiment"].notna() & (~sent_df["passed"])).sum())
print(
    f"LLMAsJudge.sentiment: {sent_eval_passed}/{sent_evaluated} evaluated sessions passed "
    f"({sent_eval_passed / max(sent_evaluated, 1):.0%} of evaluated, failed={sent_eval_failed}, "
    f"not_evaluated={sent_not_evaluated}, mode={sent_mode})"
)
if sent_not_evaluated >= len(sent_df) > 0:
    print(
        f"⚠️ No live sentiment evaluations: all {len(sent_df)} judge calls failed "
        f"(execution_mode={sent_mode!r}). Scores of NaN/None below are judge errors, not measured sentiment."
    )
elif sent_not_evaluated > 0:
    print(
        f"⚠️ {sent_not_evaluated}/{len(sent_df)} sessions returned no sentiment score (judge errors, "
        f"execution_mode={sent_mode!r}); they are excluded from the pass rate in Section 3.2."
    )

# Join sentiment scores with v_bqca_customer_turns to inspect the lowest-scoring conversations.
sent_triage_df = (
    sent_df.merge(
        turns_df[["session_id", "user_prompt", "agent_response", "has_error", "finish_reasons"]],
        on="session_id",
        how="left",
    )
    .sort_values(["sentiment", "session_id"], na_position="first")
    .reset_index(drop=True)
)
_failed_sent_df = sent_triage_df[~sent_triage_df.passed][
    ["persona", "session_id", "sentiment", "has_error", "finish_reasons", "user_prompt", "feedback"]
]
print(
    f"Triage queue — sessions that failed the {JUDGE_THRESHOLD} sentiment threshold or returned no score "
    f"({len(_failed_sent_df)} total, showing up to 25):"
)
display(_failed_sent_df.head(25))

# passed / failed count ONLY evaluated sessions; judge errors (NaN sentiment) are reported in not_evaluated.
sent_by_cohort_df = (
    sent_df.assign(
        _evaluated=sent_df["sentiment"].notna(),
        _eval_passed=sent_df["sentiment"].notna() & sent_df["passed"],
        _eval_failed=sent_df["sentiment"].notna() & (~sent_df["passed"]),
        _not_evaluated=sent_df["sentiment"].isna(),
    )
    .groupby("persona")
    .agg(
        sessions=("session_id", "count"),
        evaluated=("_evaluated", lambda s: int(s.sum())),
        passed=("_eval_passed", lambda s: int(s.sum())),
        failed=("_eval_failed", lambda s: int(s.sum())),
        not_evaluated=("_not_evaluated", lambda s: int(s.sum())),
        mean_sentiment=("sentiment", "mean"),
        min_sentiment=("sentiment", "min"),
    )
    .round(2)
    .reset_index()
)
print("Sentiment summary by user cohort:")
display(sent_by_cohort_df)

In [ ]:
#@title 3.2 UX & Tone Categorical Rubric (evaluate_categorical)
ux_tone_metric = CategoricalMetricDefinition(
    name="customer_ux_tone",
    definition="Overall user experience and tone across the session, judged from the user's prompt and how well the agent addressed their need.",
    categories=[
        CategoricalMetricCategory(
            name="Positive_Satisfied",
            definition="The user is appreciative or satisfied, and the agent clearly answered their question.",
        ),
        CategoricalMetricCategory(
            name="Neutral_Analytical",
            definition="Matter-of-fact analytical question and response with no strong emotional signal.",
        ),
        CategoricalMetricCategory(
            name="Clarification_Needed",
            definition="The user's request was underspecified or ambiguous, and the agent asked a clarifying question.",
        ),
        CategoricalMetricCategory(
            name="Frustrated_Impatient",
            definition="The user expresses frustration, impatience, or dissatisfaction with the agent or its output.",
        ),
    ],
)
ux_tone_cfg = CategoricalEvaluationConfig(
    metrics=[ux_tone_metric],
    endpoint=CATEGORICAL_MODEL,
    connection_id=VERTEX_CONNECTION_ID,
    include_justification=True,
)
with timed(f"evaluate_categorical: customer_ux_tone ({CATEGORICAL_MODEL})"):
    ux_tone_report = client.evaluate_categorical(ux_tone_cfg, filters=EVAL_FILTER)
_normalize_categorical_report(ux_tone_report)
print(ux_tone_report.summary())
print("Execution mode:", ux_tone_report.details.get("execution_mode"))
ux_tone_df = cat_frame(ux_tone_report)
_valid_tone_df = (
    ux_tone_df[ux_tone_df["valid"].fillna(False)]
    if not ux_tone_df.empty and "valid" in ux_tone_df.columns
    else ux_tone_df
)
if not _valid_tone_df.empty:
    print("User tone distribution by cohort:")
    display(pd.crosstab(_valid_tone_df["persona"], _valid_tone_df["category"]))

flagged_ux_df = _valid_tone_df[_valid_tone_df.category.isin(["Frustrated_Impatient", "Clarification_Needed"])][
    ["persona", "session_id", "category", "justification"]
].reset_index(drop=True)
print(f"Sessions flagged as Frustrated_Impatient or Clarification_Needed ({len(flagged_ux_df)} total, showing up to 25):")
display(flagged_ux_df.head(25))

tone_not_evaluated = int((~ux_tone_df["valid"].fillna(False)).sum()) if not ux_tone_df.empty else 0
tone_mode = ux_tone_report.details.get("execution_mode", "unknown")
RESULTS["s3"] = {
    "sessions": len(sent_df),
    # passed / failed are MEASURED outcomes only (sentiment score present); judge API errors
    # (e.g. 429 RESOURCE_EXHAUSTED -> NaN) are counted separately in sent_not_evaluated.
    "sent_evaluated": int(sent_df["sentiment"].notna().sum()),
    "passed": int((sent_df["sentiment"].notna() & sent_df["passed"]).sum()),
    "failed": int((sent_df["sentiment"].notna() & (~sent_df["passed"])).sum()),
    "sent_not_evaluated": int(sent_df["sentiment"].isna().sum()),
    "mean_sentiment": round(float(sent_df["sentiment"].dropna().mean()), 3) if sent_df["sentiment"].notna().any() else None,
    "sent_mode": globals().get("sent_mode", "unknown"),
    "tone_counts": _valid_tone_df.category.value_counts().to_dict(),
    "tone_not_evaluated": tone_not_evaluated,
    "tone_sessions": len(ux_tone_df),
    "tone_evaluated": len(ux_tone_df) - tone_not_evaluated,
    "tone_mode": tone_mode,
    "frustrated_sessions": _valid_tone_df[_valid_tone_df.category == "Frustrated_Impatient"].session_id.tolist(),
    "clarification_sessions": _valid_tone_df[_valid_tone_df.category == "Clarification_Needed"].session_id.tolist(),
}
_s3 = RESULTS["s3"]
_s3_sent_eval = _s3["sent_evaluated"]
if _s3["sessions"] > 0 and _s3["sent_not_evaluated"] >= _s3["sessions"]:
    _s3_sent_line = (
        f"- **Sentiment judge (`LLMAsJudge.sentiment`)**: **no live evaluations** — all **{_s3['sessions']}** "
        f"judge calls failed (`execution_mode='{_s3['sent_mode']}'`); the triage table above reflects judge errors, "
        "not measured user sentiment."
    )
else:
    _s3_sent_line = (
        f"- **Sentiment judge (`LLMAsJudge.sentiment`)**: **{_s3['passed']}/{_s3_sent_eval}** evaluated sessions "
        f"passed the `{JUDGE_THRESHOLD}` threshold (**{_s3['passed'] / max(_s3_sent_eval, 1):.0%}** of evaluated); "
        f"**{_s3['failed']}** evaluated sessions failed and were routed to the triage table above"
        + (f"; **{_s3['sent_not_evaluated']}** of **{_s3['sessions']}** sessions could not be evaluated (judge errors)" if _s3["sent_not_evaluated"] else "")
        + "."
    )
_tone_total = len(ux_tone_df)
if _tone_total > 0 and tone_not_evaluated >= _tone_total:
    _s3_tone_line = (
        f"- **UX & Tone rubric (`evaluate_categorical`)**: **no live evaluations** — all **{_tone_total}** "
        f"judge calls failed (`execution_mode='{tone_mode}'`)."
    )
else:
    _frust_str = (
        ", ".join(f"`{s}`" for s in _s3["frustrated_sessions"][:5])
        + (f", ... (+{len(_s3['frustrated_sessions']) - 5} more)" if len(_s3["frustrated_sessions"]) > 5 else "")
    ) or "none"
    _clar_str = (
        ", ".join(f"`{s}`" for s in _s3["clarification_sessions"][:5])
        + (f", ... (+{len(_s3['clarification_sessions']) - 5} more)" if len(_s3["clarification_sessions"]) > 5 else "")
    ) or "none"
    _s3_tone_line = (
        f"- **UX & Tone rubric (`evaluate_categorical`)**: {', '.join(f'**{k}** {v}' for k, v in _s3['tone_counts'].items())}. "
        f"Frustrated session(s) ({len(_s3['frustrated_sessions'])}): {_frust_str}; "
        f"clarification-needed session(s) ({len(_s3['clarification_sessions'])}): {_clar_str}"
        + (f" (**{tone_not_evaluated}** of **{_tone_total}** sessions unevaluated: judge errors)" if tone_not_evaluated else "")
        + "."
    )
show("**Section 3 Takeaway (live)**\n\n" + _s3_sent_line + "\n" + _s3_tone_line)

## 4. User question categorisation & topic taxonomy (`AI.CLASSIFY` & `AI.GENERATE`)

Understanding **what kinds of questions** users ask helps data owners prioritize new tables, column descriptions, and Verified Queries. With `client.evaluate_categorical()`, you can run your taxonomy in two modes:

- **Bulk SQL Classification Mode (`include_justification=False`)**: Uses BigQuery `AI.CLASSIFY` to return category labels without rationale tokens (wall-clock latency varies with Vertex AI concurrency).
- **Audited Mode (`include_justification=True`)**: Uses BigQuery `AI.GENERATE` with structured output to return both the category label and a 1-sentence explanation.

In [ ]:
#@title 4.1 Question Topic Taxonomy (AI.CLASSIFY vs AI.GENERATE)
# Customize these 5 example topic categories (shown here for a public education dataset) to match your own business domain.
topic_metric = CategoricalMetricDefinition(
    name="question_topic",
    definition="Primary analytical intent of the user's question in this session.",
    categories=[
        CategoricalMetricCategory(
            name="Ranking_TopN",
            definition="Top-N, bottom-N, or ranking questions (for example, top 5 entities or schools by metric).",
        ),
        CategoricalMetricCategory(
            name="Aggregation_KPI",
            definition="Overall totals, counts, averages, percentages, or headline KPIs.",
        ),
        CategoricalMetricCategory(
            name="Cohort_Comparison",
            definition="Comparing two or more regions, segments, cohorts, or entity types.",
        ),
        CategoricalMetricCategory(
            name="Schema_Metadata",
            definition="Questions about available tables, join keys, column meanings, or record counts.",
        ),
        CategoricalMetricCategory(
            name="Out_of_Scope_Drift",
            definition="Questions outside the connected dataset (other domains, unrelated industries, or unlogged PII).",
        ),
    ],
)

topic_reports = {}
for include_justification in (False, True):
    mode_label = "AI.GENERATE (with rationale)" if include_justification else "AI.CLASSIFY (fast bulk)"
    cfg = CategoricalEvaluationConfig(
        metrics=[topic_metric],
        endpoint=CATEGORICAL_MODEL,
        connection_id=VERTEX_CONNECTION_ID,
        include_justification=include_justification,
    )
    with timed(f"evaluate_categorical: question_topic [{mode_label}] ({CATEGORICAL_MODEL})"):
        topic_reports[include_justification] = client.evaluate_categorical(cfg, filters=EVAL_FILTER)
    print(f"  -> {mode_label} execution_mode={topic_reports[include_justification].details.get('execution_mode')}")

df_classify = cat_frame(topic_reports[False])
df_generate = cat_frame(topic_reports[True])
_valid_classify_df = (
    df_classify[df_classify["valid"].fillna(False)]
    if not df_classify.empty and "valid" in df_classify.columns
    else df_classify
)
_valid_generate_df = (
    df_generate[df_generate["valid"].fillna(False)]
    if not df_generate.empty and "valid" in df_generate.columns
    else df_generate
)

topic_dist_df = (
    pd.DataFrame({
        "AI.CLASSIFY (fast)": _valid_classify_df.category.value_counts(),
        "AI.GENERATE (audited)": _valid_generate_df.category.value_counts(),
    })
    .fillna(0)
    .astype(int)
    .sort_values("AI.GENERATE (audited)", ascending=False)
)
print("Question topic distribution across both execution engines:")
display(topic_dist_df)

topic_merged_df = (
    df_classify[["session_id", "persona", "category", "valid"]]
    .merge(df_generate[["session_id", "category", "valid", "justification"]], on="session_id", suffixes=("_classify", "_generate"))
    .sort_values(["persona", "session_id"])
    .reset_index(drop=True)
)
topic_both_valid = (
    topic_merged_df.valid_classify.fillna(False)
    & topic_merged_df.valid_generate.fillna(False)
    & topic_merged_df.category_classify.notna()
    & topic_merged_df.category_generate.notna()
)
topic_evaluated = int(topic_both_valid.sum())
topic_not_evaluated = len(topic_merged_df) - topic_evaluated
mode_agreement = int(((topic_merged_df.category_classify == topic_merged_df.category_generate) & topic_both_valid).sum())
if len(topic_merged_df) > 0 and topic_evaluated == 0:
    print(f"⚠️ No live topic evaluations: all {len(topic_merged_df)} sessions failed on at least one engine; agreement below is not signal.")
print(
    f"Engine agreement: AI.CLASSIFY and AI.GENERATE assigned the same category on {mode_agreement}/{topic_evaluated} evaluated sessions"
    + (f" ({topic_not_evaluated} of {len(topic_merged_df)} unevaluated: judge errors)" if topic_not_evaluated else "")
    + "."
)
if not _valid_generate_df.empty:
    print("\nQuestion topics by user cohort (AI.GENERATE):")
    display(pd.crosstab(_valid_generate_df["persona"], _valid_generate_df["category"]))

classify_mode = topic_reports[False].details.get("execution_mode", "unknown")
generate_mode = topic_reports[True].details.get("execution_mode", "unknown")
both_native_modes = (classify_mode == "ai_classify" and generate_mode == "ai_generate")
mode_pair_desc = (
    "`AI.CLASSIFY` and `AI.GENERATE`"
    if both_native_modes
    else f"fast mode (`{classify_mode}`) and rationale mode (`{generate_mode}`)"
)
RESULTS["s4"] = {
    "sessions": len(topic_merged_df),
    "evaluated": topic_evaluated,
    "not_evaluated": topic_not_evaluated,
    "agree": mode_agreement,
    "classify_mode": classify_mode,
    "generate_mode": generate_mode,
    "distribution": _valid_generate_df.category.value_counts().to_dict(),
    "out_of_scope": int((_valid_generate_df.category == "Out_of_Scope_Drift").sum()),
}
_s4 = RESULTS["s4"]
if _s4["sessions"] > 0 and _s4["evaluated"] == 0:
    _s4_agree_line = (
        f"- Across **{_s4['sessions']}** sessions, {mode_pair_desc} produced **no live evaluations** — every session "
        f"failed on at least one engine (fast mode=`{_s4['classify_mode']}`, rationale mode=`{_s4['generate_mode']}`)."
    )
else:
    _s4_agree_line = (
        f"- Across **{_s4['sessions']}** sessions, {mode_pair_desc} agreed on **{_s4['agree']}/{_s4['evaluated']}** "
        f"evaluated topic labels (**{_s4['agree'] / max(_s4['evaluated'], 1):.0%}**)"
        + (f"; **{_s4['not_evaluated']}** sessions unevaluated (judge errors)" if _s4["not_evaluated"] else "")
        + "."
    )
_s4_dist_str = ", ".join(f"**{k}** {v}" for k, v in _s4["distribution"].items()) or "none (no valid topic classifications)"
show(f"""**Section 4 Takeaway (live)**

{_s4_agree_line}
- Topic breakdown: {_s4_dist_str}. Use `AI.CLASSIFY` for daily dashboards and `AI.GENERATE` when auditing specific cohorts.""")

## 5. Semantic drift detection & Golden Question Bank coverage

### 5.1 Bootstrap or load `bqca_golden_qa` & run `client.drift_detection()`

As your users adopt Conversational Analytics, two coverage questions arise:

1. **Golden Coverage (`client.drift_detection()`)**: How much of your curated Golden Question Bank is actually being exercised in production traffic, and which golden questions have never been asked?
2. **Production Semantic Drift (`AI.EMBED` + `ML.DISTANCE`)**: Which incoming user questions are semantically far (`cosine_distance > DRIFT_THRESHOLD`) from *every* question in your Golden Question Bank — signaling new datasets or verified queries you should add?

> **How starter bootstrapping works (and RLS/CLS governance note):** If `bqca_golden_qa` does not exist yet in your dataset, Cell `5.1` runs `CREATE TABLE IF NOT EXISTS` to seed an initial template table from up to 50 distinct, clean single-turn answered sessions (`turns = 1`, deduplicated by `LOWER(TRIM(user_prompt))`) with `verified = FALSE` and `source = 'auto_bootstrap_template'`. Auto-seeded rows allow the drift and SQL grading pipelines to run on your first click, and print a reminder showing how to replace them with SME-verified ground truth (`verified = TRUE`). **Security caveat:** Because `CREATE TABLE ... AS SELECT` materializes a new physical table, BigQuery does **not** automatically copy source-table Row-Level Security (RLS) policies or Column-Level Security (CLS / policy tags) onto `bqca_golden_qa`. Cell `5.1` inspects `RAW_TABLE_ID` via `tables.rowAccessPolicies.list` and recursive `SchemaField.policy_tags` and **fails closed** by default if any row access policy or column policy tag is present **or if the caller lacks `bigquery.rowAccessPolicies.list` to verify row access policies**: pre-create `bqca_golden_qa` in `DATASET_ID` with matching row access policies / policy tags, or set `BQCA_AUTO_SEED_GOLDEN_TABLE=false` before running Cell `5.1` (only set `BQCA_AUTO_SEED_GOLDEN_TABLE=force` in an isolated scratch dataset). `CREATE TABLE IF NOT EXISTS` never refreshes an existing `bqca_golden_qa` — re-running Cell `5.1` reuses the current table until you drop or curate it.

In [ ]:
#@title 5.1 Bootstrap or Load Golden Question Bank & Run client.drift_detection()
AUTO_SEED_GOLDEN_TABLE = os.environ.get("BQCA_AUTO_SEED_GOLDEN_TABLE", "true").strip().lower() not in ("false", "0", "no")
ASSUME_BYO_GOLDEN_VERIFIED = os.environ.get("BQCA_ASSUME_BYO_GOLDEN_VERIFIED", "false").strip().lower() in ("true", "1", "yes")

# Clear any prior Section 5 state so continuing to 5.2 / 6 / 8.3 after a Section 5.1 refusal
# reports Section 5 as unavailable rather than crashing on undefined names or stale data:
drift_report = None
RESULTS.pop("s5", None)
for _stale_var in (
    "matched_df",
    "graded_df",
    "verified_graded_df",
    "template_graded_df",
    "GOLDEN_GRADING_AVAILABLE",
    "GOLDEN_GRADING_COMPLETED",
    "PROD_EMBEDDING_FAILED_COUNT",
    "GOLD_EMBEDDING_FAILED_COUNT",
    "EMBEDDING_FAILED_COUNT",
    "EMBEDDING_INCOMPLETE",
    "no_gold_emb_count",
    "blank_golden_count",
    "dup_golden_count",
    "GOLDEN_BLANK_QUESTION_SQL_FIX",
):
    globals().pop(_stale_var, None)

# NOTE (data governance): same-dataset placement gives bqca_golden_qa uniform dataset-level
# IAM and VPC-SC perimeter rules, but row access policies and column policy tags are per-table
# and are NOT copied by this CTAS, which materializes whatever rows the executing principal can
# see. Below we inspect RAW_TABLE_ID for active row access policies (via tables.rowAccessPolicies.list)
# and column policy tags and fail closed by default if either is present or cannot be verified.
# CREATE TABLE IF NOT EXISTS never refreshes an existing golden bank (drop or curate the table to re-seed it).
BOOTSTRAP_GOLDEN_SQL = f"""
CREATE TABLE IF NOT EXISTS {T(GOLDEN_QA_TABLE_ID)} AS
WITH eligible AS (
  SELECT
    session_id,
    started_at,
    persona_role,
    user_prompt,
    agent_response
  FROM {T(TURNS_VIEW_ID)}
  WHERE user_prompt IS NOT NULL
    AND agent_response IS NOT NULL
    AND NOT has_error
    AND NULLIF(TRIM(clarifying_question), '') IS NULL
    AND turns = 1
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY LOWER(TRIM(user_prompt))
    ORDER BY started_at ASC, session_id ASC
  ) = 1
)
SELECT
  FORMAT('G%02d', ROW_NUMBER() OVER (ORDER BY started_at ASC, session_id ASC)) AS golden_id,
  COALESCE(NULLIF(persona_role, ''), 'Auto-Seeded Template') AS category,
  user_prompt AS golden_question,
  agent_response AS golden_answer,
  user_prompt AS question,
  agent_response AS expected_answer,
  session_id AS source_session_id,
  'auto_bootstrap_template' AS source,
  FALSE AS verified
FROM eligible
ORDER BY started_at ASC, session_id ASC
LIMIT 50
"""


def _count_policy_tags(fields):
    total = 0
    for f in fields or ():
        pt = getattr(f, "policy_tags", None)
        if pt and getattr(pt, "names", None):
            total += len(pt.names)
        sub = getattr(f, "fields", None)
        if sub:
            total += _count_policy_tags(sub)
    return total


def _row_access_policy_count(client, project, dataset, table):
    # google-cloud-bigquery has no public helper for tables.rowAccessPolicies.list, so we call the REST
    # endpoint via client._connection.api_request (if _connection ever changes, AttributeError is caught
    # by _rls_probe_error below and the gate fails closed).
    path = f"/projects/{project}/datasets/{dataset}/tables/{table}/rowAccessPolicies"
    total = 0
    token = None
    while True:
        resp = client._connection.api_request(
            method="GET",
            path=path,
            query_params={"pageToken": token} if token else None,
        )
        total += len(resp.get("rowAccessPolicies", ()) or ())
        token = resp.get("nextPageToken")
        if not token:
            return total


try:
    _golden_table_ref = bq.get_table(f"{PROJECT_ID}.{DATASET_ID}.{GOLDEN_QA_TABLE_ID}")
except gcp_exceptions.NotFound:
    if not AUTO_SEED_GOLDEN_TABLE:
        raise RuntimeError(
            f"Table {PROJECT_ID}.{DATASET_ID}.{GOLDEN_QA_TABLE_ID} does not exist and BQCA_AUTO_SEED_GOLDEN_TABLE=false. "
            "Pre-create the table with your required RLS/CLS policies before running Section 5."
        )
    _raw_table_ref = bq.get_table(f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE_ID}")
    _cls_tag_count = _count_policy_tags(_raw_table_ref.schema)
    _rls_policy_count = None
    _rls_probe_error = None
    try:
        _rls_policy_count = _row_access_policy_count(bq, PROJECT_ID, DATASET_ID, RAW_TABLE_ID)
    except Exception as exc:
        _rls_probe_error = exc
    _force_seed = os.environ.get("BQCA_AUTO_SEED_GOLDEN_TABLE", "").strip().lower() == "force"
    if not _force_seed and (_rls_probe_error is not None or _rls_policy_count or _cls_tag_count):
        _reason = (
            f"could not verify row access policies on {RAW_TABLE_ID} via tables.rowAccessPolicies.list "
            f"({_rls_probe_error}; grant bigquery.rowAccessPolicies.list, e.g. roles/bigquery.dataOwner or roles/bigquery.admin on the source table, or a custom role)"
            if _rls_probe_error is not None
            else (
                f"source table {RAW_TABLE_ID} has active governance policies "
                f"(row_access_policies={_rls_policy_count}, column_policy_tags={_cls_tag_count}) "
                "that BigQuery CTAS does not copy"
            )
        )
        raise RuntimeError(
            f"Refusing to auto-materialize {PROJECT_ID}.{DATASET_ID}.{GOLDEN_QA_TABLE_ID} via CTAS because {_reason}. "
            "Pre-create bqca_golden_qa with matching row access policies / policy tags, "
            "or set BQCA_AUTO_SEED_GOLDEN_TABLE=force only in an isolated scratch dataset."
        )
    if _force_seed:
        print(
            f"⚠️ BQCA_AUTO_SEED_GOLDEN_TABLE=force: Bootstrapping {PROJECT_ID}.{DATASET_ID}.{GOLDEN_QA_TABLE_ID} via CTAS "
            f"(row_access_policies={_rls_policy_count if _rls_policy_count is not None else f'unverified ({_rls_probe_error})'}, "
            f"column_policy_tags={_cls_tag_count})."
        )
    else:
        print(
            f"Bootstrapping starter {PROJECT_ID}.{DATASET_ID}.{GOLDEN_QA_TABLE_ID} via CREATE TABLE IF NOT EXISTS ... AS SELECT "
            "(verified 0 row access policies via tables.rowAccessPolicies.list and 0 column policy tags on source table)..."
        )
    bq.query(
        BOOTSTRAP_GOLDEN_SQL,
        job_config=bigquery.QueryJobConfig(maximum_bytes_billed=int(MAX_BYTES_BILLED)),
    ).result()
    _golden_table_ref = bq.get_table(f"{PROJECT_ID}.{DATASET_ID}.{GOLDEN_QA_TABLE_ID}")

golden_table_cols = {field.name for field in _golden_table_ref.schema}
has_verified_col = "verified" in golden_table_cols
has_source_sid_col = "source_session_id" in golden_table_cols
verified_check_sql = "LOGICAL_AND(COALESCE(verified, FALSE))" if has_verified_col else ("TRUE" if ASSUME_BYO_GOLDEN_VERIFIED else "FALSE")

golden_overview_df = run_sql(f"""
SELECT
  category,
  {verified_check_sql} AS all_verified,
  COUNT(*) AS golden_questions
FROM {T(GOLDEN_QA_TABLE_ID)}
GROUP BY category
ORDER BY golden_questions DESC, category
""")
total_golden_rows = int(golden_overview_df["golden_questions"].sum()) if not golden_overview_df.empty else 0
if total_golden_rows == 0:
    raise RuntimeError(
        f"Golden Question Bank {T(GOLDEN_QA_TABLE_ID)} contains 0 rows! "
        "This happens if an empty table was left behind from an earlier run or no clean single-turn answered sessions "
        f"matched the bootstrap filter in {T(TURNS_VIEW_ID)}. Run `DROP TABLE {T(GOLDEN_QA_TABLE_ID)};` and re-run Cell 5.1 "
        "(after Cell 1.3), or insert SME-curated rows via INSERT INTO."
    )
print(f"Golden Question Bank ({GOLDEN_QA_TABLE_ID}) summary by category ({total_golden_rows} total rows):")
display(golden_overview_df)

if not bool(golden_overview_df["all_verified"].all()):
    if has_verified_col:
        print(
            "⚠️ [UNVERIFIED AUTO-SEEDED TEMPLATE] Your bqca_golden_qa table contains unverified rows (verified = FALSE) "
            "bootstrapped from the agent's own responses so Sections 5 & 6A can run out-of-the-box.\n"
            "   Before using Section 6A as an accuracy KPI, verify the answers with a domain SME and run:\n"
            f"   UPDATE {T(GOLDEN_QA_TABLE_ID)} SET golden_answer = '<verified answer>', expected_answer = '<verified answer>', "
            "verified = TRUE, source = 'sme_curated', source_session_id = NULL WHERE golden_id = 'G01';"
        )
    else:
        print(
            "⚠️ [UNVERIFIED BYO GOLDEN TABLE — missing 'verified' column] Your bqca_golden_qa table has no `verified` BOOL column, "
            "so rows default to unverified unless BQCA_ASSUME_BYO_GOLDEN_VERIFIED=true.\n"
            "   Either set `os.environ['BQCA_ASSUME_BYO_GOLDEN_VERIFIED'] = 'true'` before running Section 5, or add a `verified` column:\n"
            f"   ALTER TABLE {T(GOLDEN_QA_TABLE_ID)} ADD COLUMN IF NOT EXISTS verified BOOL;\n"
            f"   UPDATE {T(GOLDEN_QA_TABLE_ID)} SET verified = TRUE WHERE TRUE;"
        )

# Blank/empty golden questions make AI.EMBED return an empty vector ("INVALID_ARGUMENT: The text content
# is empty"), which breaks ML.DISTANCE inside client.drift_detection() and silently drops it to the
# keyword-overlap fallback. Detect them up front and print the exact repair statement.
# The SDK embeds the `question` column directly (it does not COALESCE to golden_question), so a NULL/blank `question`
# breaks AI.EMBED. Section 5.2 resolves COALESCE(golden_question, question), so a NULL golden_question next to a usable
# `question` is fine there, but an explicitly empty/whitespace golden_question ('') is not: flag the row when `question`
# is NULL/blank or golden_question is non-NULL and blank (a NULL golden_question alone is NOT flagged or deleted).
if {"golden_question", "question"} <= golden_table_cols:
    _blank_pred = "(TRIM(COALESCE(question, '')) = '' OR TRIM(golden_question) = '')"
elif "golden_question" in golden_table_cols:
    _blank_pred = "TRIM(COALESCE(golden_question, '')) = ''"
else:
    _blank_pred = "TRIM(COALESCE(question, '')) = ''"
blank_golden_count = int(
    run_sql(f"SELECT COUNTIF({_blank_pred}) AS n FROM {T(GOLDEN_QA_TABLE_ID)}").iloc[0]["n"]
)
GOLDEN_BLANK_QUESTION_SQL_FIX = f"DELETE FROM {T(GOLDEN_QA_TABLE_ID)} WHERE {_blank_pred};"
if blank_golden_count > 0:
    print(
        f"⚠️ Warning: {GOLDEN_QA_TABLE_ID} contains {blank_golden_count} blank/empty question row(s). "
        "Empty text causes AI.EMBED to return an empty vector (INVALID_ARGUMENT: The text content is empty) and "
        "forces client.drift_detection() into keyword fallback. Repair or delete blank rows before interpreting drift:\n"
        f"   {GOLDEN_BLANK_QUESTION_SQL_FIX}"
    )

# Case/whitespace duplicates ("Revenue?" vs "revenue?") are ONE golden question under the SDK's LOWER(TRIM()) key. The
# coverage headline below collapses them (see _normalize_drift_report in Cell 1.2), but duplicate rows still add embedding
# and Section 6A grading cost and hide how many distinct questions the bank really has, so surface them up front.
_dup_col = "question" if "question" in golden_table_cols else "golden_question"
dup_golden_count = int(
    run_sql(
        f"""
SELECT
  COUNTIF(TRIM(COALESCE({_dup_col}, '')) != '')
  - COUNT(DISTINCT IF(TRIM(COALESCE({_dup_col}, '')) != '', LOWER(TRIM({_dup_col})), NULL)) AS n
FROM {T(GOLDEN_QA_TABLE_ID)}
"""
    ).iloc[0]["n"]
)
if dup_golden_count > 0:
    print(
        f"⚠️ Warning: {GOLDEN_QA_TABLE_ID} has {dup_golden_count} duplicate golden question row(s) under LOWER(TRIM({_dup_col})) "
        "(for example 'Revenue?' and 'revenue?'). Coverage below counts each distinct question once, but the duplicates still "
        "add embedding and grading cost; list them with:\n"
        f"   SELECT LOWER(TRIM({_dup_col})) AS question_key, COUNT(*) AS copies FROM {T(GOLDEN_QA_TABLE_ID)} "
        "GROUP BY question_key HAVING copies > 1 ORDER BY copies DESC;"
    )

with timed(f"client.drift_detection (AI.EMBED + ML.DISTANCE, {EMBEDDING_MODEL})"):
    drift_report = client.drift_detection(
        golden_dataset=GOLDEN_QA_TABLE_ID,
        filters=EVAL_FILTER,
        embedding_model=EMBEDDING_MODEL,
    )
# Idempotent safety net (Cell 1.2 already normalizes at the SDK source): one golden question per LOWER(TRIM()) key, coverage <= 100%.
drift_report = _normalize_drift_report(drift_report)
print(drift_report.summary())
drift_method = drift_report.details.get("method")
DRIFT_METHOD = drift_method
_s51_failed_rows = drift_report.details.get("failed_semantic_rows")
_s51_failed_note = drift_report.details.get(
    "failed_rows_note",
    "per-row failed embedding count is unknown on the SDK Section 5.1 route (feedback.py selects AI.EMBED(...).result without .status)",
)
print("drift_detection method:", drift_method)
if drift_method in ("semantic_embedding", "semantic"):
    if _s51_failed_rows:
        print(
            f"⚠️ Warning: client.drift_detection() excluded {_s51_failed_rows} golden question row(s) that returned NULL/invalid "
            f"cosine distance from the semantic coverage denominator (evaluated {drift_report.total_golden} valid golden question(s))."
        )
else:
    _s51_count_msg = (
        f"Failed semantic rows before fallback: {_s51_failed_rows}."
        if _s51_failed_rows is not None
        else f"Failed-row count on SDK 5.1 route: unknown ({_s51_failed_note})."
    )
    if blank_golden_count > 0:
        print(
            f"⚠️ Warning: semantic drift evaluation is unavailable in Section 5.1 — client.drift_detection() used fallback "
            f"method={drift_method!r} instead of 'semantic_embedding' (AI.EMBED) because {GOLDEN_QA_TABLE_ID} contains "
            f"{blank_golden_count} empty/blank question row(s) (INVALID_ARGUMENT on empty text/vector). {_s51_count_msg} "
            f"Repair or delete blank rows in {T(GOLDEN_QA_TABLE_ID)} and re-run Section 5:\n   {GOLDEN_BLANK_QUESTION_SQL_FIX}"
        )
    else:
        print(
            f"⚠️ Warning: semantic drift evaluation is unavailable in Section 5.1 — client.drift_detection() used fallback "
            f"method={drift_method!r} instead of 'semantic_embedding' (AI.EMBED). {_s51_count_msg} "
            f"Check {T(GOLDEN_QA_TABLE_ID)} and {T(COMPAT_VIEW_ID)} for empty/invalid question text (INVALID_ARGUMENT), "
            "verify Vertex AI embedding quota, or ensure the notebook runner principal and connection service account have "
            "roles/aiplatform.user and roles/bigquery.connectionUser."
        )
# Note: client.drift_detection() uses the SDK's built-in similarity_threshold = 0.30 (SDK_DRIFT_THRESHOLD),
# whereas DRIFT_THRESHOLD (0.15 in Cell 1.1) governs Section 5.2 production drift flagging and Section 6A in-domain pairing.
SDK_DRIFT_THRESHOLD = drift_report.details.get("similarity_threshold", 0.3)

if drift_method in ("semantic_embedding", "semantic"):
    _uncovered_label = f"never asked within cosine distance {SDK_DRIFT_THRESHOLD}"
else:
    _uncovered_label = f"never asked (keyword-overlap fallback, method={drift_method!r} — not a cosine-distance measurement)"
print(f"\nUncovered Golden Questions ({_uncovered_label}):")
for q in drift_report.uncovered_questions:
    print("  -", short(q, 180))

In [ ]:
#@title 5.2 Production-Centric Semantic Drift Query (AI.EMBED + ML.DISTANCE)
# Reset downstream Section 6A grading state whenever Section 5.2 runs so jumping
# from 5.2 to 6B without re-running 6A never reuses a stale graded_df:
GOLDEN_GRADING_AVAILABLE = False
GOLDEN_GRADING_COMPLETED = False
GOLDEN_GRADING_MODE = "not_run"
graded_df = None
verified_graded_df = pd.DataFrame(columns=["session_id", "verdict"])
template_graded_df = pd.DataFrame(columns=["session_id", "verdict"])
RESULTS.pop("s6", None)

if globals().get("drift_report") is None:
    RESULTS.pop("s5", None)
    for _stale_var in (
        "matched_df",
        "graded_df",
        "verified_graded_df",
        "template_graded_df",
        "GOLDEN_GRADING_AVAILABLE",
        "GOLDEN_GRADING_COMPLETED",
        "GOLDEN_GRADING_MODE",
        "PROD_EMBEDDING_FAILED_COUNT",
        "GOLD_EMBEDDING_FAILED_COUNT",
        "EMBEDDING_FAILED_COUNT",
        "EMBEDDING_INCOMPLETE",
        "no_gold_emb_count",
    ):
        globals().pop(_stale_var, None)
    print(
        "⚠️ Skipping Section 5.2 because Section 5.1 did not complete (golden question bank unavailable). "
        "Resolve the Section 5.1 error above and re-run Sections 5.1 and 5.2."
    )
    show(
        "**Section 5 Takeaway (live)**\n\n"
        "- **Semantic Drift & Golden Question Bank coverage**: unavailable (Section 5.1 did not complete — golden question bank unavailable)."
    )
else:
    verified_expr = "COALESCE(verified, FALSE)" if has_verified_col else ("TRUE" if ASSUME_BYO_GOLDEN_VERIFIED else "FALSE")
    source_sid_expr = "source_session_id" if has_source_sid_col else "CAST(NULL AS STRING)"
    if {"golden_question", "question"} <= golden_table_cols:
        golden_q_expr = "COALESCE(golden_question, question)"
    elif "golden_question" in golden_table_cols:
        golden_q_expr = "golden_question"
    else:
        golden_q_expr = "question"

    if {"golden_answer", "expected_answer"} <= golden_table_cols:
        golden_a_expr = "COALESCE(golden_answer, expected_answer)"
    elif "golden_answer" in golden_table_cols:
        golden_a_expr = "golden_answer"
    else:
        golden_a_expr = "expected_answer"

    eval_sid_clause = "\n    AND session_id IN UNNEST(@eval_session_ids)" if EVAL_SESSION_IDS else ""
    eval_sid_params = {"eval_session_ids": EVAL_SESSION_IDS} if EVAL_SESSION_IDS else {}

    MATCH_CTE = f"""
prod AS (
  SELECT
    session_id,
    user_id,
    persona,
    user_prompt AS prod_question,
    agent_response AS prod_answer
  FROM {T(TURNS_VIEW_ID)}
  WHERE user_prompt IS NOT NULL
    AND turns = 1{eval_sid_clause}
  ORDER BY started_at DESC, session_id
  LIMIT {int(MAX_EVAL_SESSIONS)}
),
prod_emb_raw AS (
  SELECT
    *,
    AI.EMBED(
      prod_question,
      connection_id => '{VERTEX_CONNECTION_ID}',
      endpoint => '{EMBEDDING_MODEL}'
    ) AS emb
  FROM prod
),
gold_source AS (
  SELECT
    golden_id,
    category AS golden_category,
    {golden_q_expr} AS golden_question,
    {golden_a_expr} AS golden_answer,
    {verified_expr} AS golden_verified,
    {source_sid_expr} AS source_session_id
  FROM {T(GOLDEN_QA_TABLE_ID)}
  WHERE {golden_q_expr} IS NOT NULL
  ORDER BY golden_verified DESC, golden_id ASC
  LIMIT {int(MAX_GOLDEN_QUESTIONS)}
),
gold_emb_raw AS (
  SELECT
    *,
    AI.EMBED(
      golden_question,
      connection_id => '{VERTEX_CONNECTION_ID}',
      endpoint => '{EMBEDDING_MODEL}'
    ) AS emb
  FROM gold_source
),
gold_emb AS (
  SELECT
    * EXCEPT(emb),
    emb.result AS q_emb,
    emb.status AS emb_status,
    ARRAY_LENGTH(emb.result) > 0 AND IFNULL(emb.status, '') = '' AS gold_emb_ok,
    COUNT(*) OVER () AS gold_emb_attempted,
    COUNTIF(ARRAY_LENGTH(emb.result) = 0 OR emb.result IS NULL OR IFNULL(emb.status, '') != '') OVER () AS gold_emb_failed,
    STRING_AGG(
      IF(
        ARRAY_LENGTH(emb.result) = 0 OR emb.result IS NULL OR IFNULL(emb.status, '') != '',
        CONCAT(golden_id, ':', IFNULL(NULLIF(emb.status, ''), 'empty_vector')),
        NULL
      ),
      ' | '
    ) OVER () AS failed_golden_summary
  FROM gold_emb_raw
),
matched AS (
  SELECT
    p.session_id,
    p.user_id,
    p.persona,
    p.prod_question,
    p.prod_answer,
    ARRAY_LENGTH(p.emb.result) = 0 OR p.emb.result IS NULL OR IFNULL(p.emb.status, '') != '' AS prod_emb_failed,
    p.emb.status AS prod_emb_status,
    IF(g.gold_emb_ok, g.golden_id, NULL) AS golden_id,
    IF(g.gold_emb_ok, g.golden_category, NULL) AS golden_category,
    IF(g.gold_emb_ok, g.golden_question, NULL) AS golden_question,
    IF(g.gold_emb_ok, g.golden_answer, NULL) AS golden_answer,
    IF(g.gold_emb_ok, g.golden_verified, NULL) AS golden_verified,
    IF(g.gold_emb_ok, g.source_session_id, NULL) AS source_session_id,
    g.gold_emb_attempted,
    g.gold_emb_failed,
    g.failed_golden_summary,
    IF(
      ARRAY_LENGTH(p.emb.result) > 0 AND IFNULL(p.emb.status, '') = '' AND g.gold_emb_ok,
      ML.DISTANCE(p.emb.result, g.q_emb, 'COSINE'),
      NULL
    ) AS question_distance
  FROM prod_emb_raw p
  LEFT JOIN gold_emb g
    ON TRUE
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.session_id
    ORDER BY question_distance ASC NULLS LAST, g.gold_emb_ok DESC, g.golden_id ASC
  ) = 1
)"""

    with timed(f"Production-centric semantic drift query (AI.EMBED + ML.DISTANCE, {EMBEDDING_MODEL})"):
        raw_matched_df = run_sql(
            "WITH " + MATCH_CTE + "\nSELECT * FROM matched ORDER BY question_distance DESC NULLS LAST, session_id",
            eval_sid_params,
        )

    prod_emb_attempted = int(len(raw_matched_df))
    prod_emb_failed_count = int(raw_matched_df["prod_emb_failed"].fillna(True).sum()) if not raw_matched_df.empty else 0
    no_gold_emb_count = (
        int((raw_matched_df["golden_id"].isna() & ~raw_matched_df["prod_emb_failed"].fillna(True)).sum())
        if not raw_matched_df.empty
        else 0
    )
    gold_emb_attempted = (
        int(raw_matched_df["gold_emb_attempted"].dropna().iloc[0])
        if not raw_matched_df.empty and not raw_matched_df["gold_emb_attempted"].dropna().empty
        else 0
    )
    gold_emb_failed_count = (
        int(raw_matched_df["gold_emb_failed"].dropna().iloc[0])
        if not raw_matched_df.empty and not raw_matched_df["gold_emb_failed"].dropna().empty
        else 0
    )
    failed_golden_summary = (
        "; ".join(str(raw_matched_df["failed_golden_summary"].dropna().iloc[0]).split(" | ")[:5])
        if not raw_matched_df.empty and not raw_matched_df["failed_golden_summary"].dropna().empty
        else ""
    )
    PROD_EMBEDDING_FAILED_COUNT = prod_emb_failed_count
    GOLD_EMBEDDING_FAILED_COUNT = gold_emb_failed_count
    # Session-level count of single-turn sessions without a valid nearest-golden distance:
    EMBEDDING_FAILED_COUNT = prod_emb_failed_count + no_gold_emb_count
    EMBEDDING_INCOMPLETE = bool(prod_emb_failed_count > 0 or gold_emb_failed_count > 0 or no_gold_emb_count > 0)
    print(
        f"Embedding status: production prompts {prod_emb_attempted - prod_emb_failed_count}/{prod_emb_attempted} succeeded; "
        f"golden questions {max(0, gold_emb_attempted - gold_emb_failed_count)}/{gold_emb_attempted} succeeded."
    )

    def _embedding_remediation_hint(status_text, table_label, is_golden=False):
        s_low = str(status_text or "").lower()
        has_invalid = any(k in s_low for k in ("invalid_argument", "empty", "empty_vector"))
        has_quota = any(
            k in s_low for k in ("resource_exhausted", "429", "quota", "rate_limit", "deadline_exceeded", "unavailable", "503")
        )
        if has_invalid and not has_quota:
            if is_golden:
                # Reuse the exact repair statement computed by the Cell 5.1 pre-flight check when available.
                _fix_sql = globals().get("GOLDEN_BLANK_QUESTION_SQL_FIX") or (
                    f"DELETE FROM {T(GOLDEN_QA_TABLE_ID)} WHERE TRIM(COALESCE({golden_q_expr}, '')) = '';"
                )
                return (
                    f"repair or delete empty/invalid question rows in {table_label} "
                    f"(e.g. `{_fix_sql}`) and re-run Section 5"
                )
            return f"filter or repair empty/invalid prompt rows in {table_label} and re-run Section 5"
        if has_quota and not has_invalid:
            return "retry after Vertex AI embedding quota/service recovery"
        return (
            f"repair or delete empty/invalid question rows in {table_label} "
            "(if status shows INVALID_ARGUMENT/empty_vector) or retry after Vertex AI quota recovery"
        )

    sample_statuses = []
    if prod_emb_failed_count > 0:
        sample_statuses = (
            raw_matched_df.loc[raw_matched_df["prod_emb_failed"].fillna(True), "prod_emb_status"]
            .dropna()
            .unique()
            .tolist()[:3]
        )
        _prod_hint = _embedding_remediation_hint(sample_statuses, TURNS_VIEW_ID, is_golden=False)
        print(
            f"⚠️ Warning: {prod_emb_failed_count}/{prod_emb_attempted} single-turn sessions had failed/empty AI.EMBED vectors "
            f"(status={sample_statuses}) — {_prod_hint} before interpreting as semantic drift."
        )
    if gold_emb_failed_count > 0 or no_gold_emb_count > 0:
        _gold_hint = _embedding_remediation_hint(failed_golden_summary, GOLDEN_QA_TABLE_ID, is_golden=True)
        if gold_emb_failed_count > 0:
            print(
                f"⚠️ Warning: {gold_emb_failed_count}/{gold_emb_attempted or '?'} golden questions had failed/empty AI.EMBED vectors "
                f"({failed_golden_summary or 'no valid golden embeddings returned'}). "
                f"Nearest-neighbor distances and downstream Section 6A matching are INCOMPLETE — {_gold_hint} before interpreting as semantic drift."
            )
        else:
            print(
                f"⚠️ Warning: 0 embeddable golden questions returned from {GOLDEN_QA_TABLE_ID} "
                f"({no_gold_emb_count} production session(s) could not be matched to a golden embedding). "
                f"Nearest-neighbor distances and downstream Section 6A matching are INCOMPLETE — {_gold_hint} before interpreting as semantic drift."
            )
    matched_df = (
        raw_matched_df[~raw_matched_df["prod_emb_failed"].fillna(True) & raw_matched_df["question_distance"].notna()]
        .drop(
            columns=["prod_emb_failed", "prod_emb_status", "gold_emb_attempted", "gold_emb_failed", "failed_golden_summary"],
            errors="ignore",
        )
        .reset_index(drop=True)
    )

    drift_by_persona_df = (
        matched_df.groupby("persona")
        .agg(
            sessions=("session_id", "count"),
            min_distance=("question_distance", "min"),
            median_distance=("question_distance", "median"),
            max_distance=("question_distance", "max"),
            drifted_beyond_threshold=("question_distance", lambda s: int((s > DRIFT_THRESHOLD).sum())),
            drifted_beyond_sdk_030=("question_distance", lambda s: int((s > SDK_DRIFT_THRESHOLD).sum())),
        )
        .round(3)
        .sort_values("median_distance", ascending=False)
        .reset_index()
    )
    print(f"Semantic distance to nearest Golden Question by user cohort (DRIFT_THRESHOLD={DRIFT_THRESHOLD}):")
    display(drift_by_persona_df)

    drifted_prompts_df = (
        matched_df[matched_df.question_distance > DRIFT_THRESHOLD]
        .assign(
            cosine_distance=lambda d: d.question_distance.round(3),
            user_question=lambda d: d.prod_question.map(lambda q: short(q, 110)),
        )[["persona", "session_id", "cosine_distance", "golden_id", "user_question"]]
        .reset_index(drop=True)
    )
    print(
        f"Drifted / novel production questions (cosine_distance > {DRIFT_THRESHOLD}; "
        f"{len(drifted_prompts_df)} total, showing up to 25):"
    )
    display(drifted_prompts_df.head(25))

    if not drift_by_persona_df.empty:
        top_drift_row = drift_by_persona_df.iloc[0]
        top_drift_persona = str(top_drift_row.persona)
        top_drift_median = float(top_drift_row.median_distance)
    else:
        top_drift_persona = "n/a"
        top_drift_median = 0.0

    _combined_emb_err = f"{sample_statuses} {failed_golden_summary}".lower()
    if not EMBEDDING_INCOMPLETE:
        drift_status = "COMPLETE"
    elif any(k in _combined_emb_err for k in ("invalid_argument", "empty", "empty_vector")) and not any(
        k in _combined_emb_err for k in ("resource_exhausted", "429", "quota", "rate_limit", "deadline_exceeded", "unavailable", "503")
    ):
        drift_status = "INCOMPLETE (partial AI.EMBED failure — repair empty/invalid question rows before interpreting as semantic drift)"
    elif any(k in _combined_emb_err for k in ("resource_exhausted", "429", "quota", "rate_limit", "deadline_exceeded", "unavailable", "503")) and not any(
        k in _combined_emb_err for k in ("invalid_argument", "empty", "empty_vector")
    ):
        drift_status = "INCOMPLETE (partial AI.EMBED failure — retry after quota recovery before interpreting as semantic drift)"
    else:
        drift_status = "INCOMPLETE (partial AI.EMBED failure — repair invalid/empty rows or retry before interpreting as semantic drift)"
    coverage_method = globals().get("DRIFT_METHOD", globals().get("drift_method", "unknown"))
    coverage_is_semantic = coverage_method in ("semantic_embedding", "semantic")
    semantic_distances_ok = not matched_df.empty
    # Defensive + idempotent: one golden question per LOWER(TRIM()) key so the headline can never exceed 100%.
    drift_report = _normalize_drift_report(drift_report)
    _s51_details = drift_report.details if isinstance(getattr(drift_report, "details", None), dict) else {}
    _s51_failed_rows = _s51_details.get("failed_semantic_rows")
    _s51_failed_note = _s51_details.get(
        "failed_rows_note",
        "per-row failed embedding count is unknown on the SDK Section 5.1 route (feedback.py selects AI.EMBED(...).result without .status)"
        if not coverage_is_semantic
        else "",
    )
    RESULTS["s5"] = {
        "available": True,
        "coverage_pct": float(drift_report.coverage_percentage),
        "covered_golden": len(drift_report.covered_questions),
        "total_golden": int(drift_report.total_golden),
        "uncovered_golden": len(drift_report.uncovered_questions),
        "coverage_method": coverage_method,
        "coverage_is_semantic": coverage_is_semantic,
        "s51_semantic_available": bool(_s51_details.get("semantic_available", coverage_is_semantic)),
        "s51_failed_semantic_rows": _s51_failed_rows,
        "s51_failed_rows_note": _s51_failed_note,
        "s51_excluded_golden_questions": list(_s51_details.get("excluded_golden_questions") or ()),
        "semantic_distances_ok": semantic_distances_ok,
        "drifted_over_015": len(drifted_prompts_df),
        "top_drift_persona": top_drift_persona,
        "top_drift_median": top_drift_median,
        "drift_status": drift_status,
        "embedding_failed": EMBEDDING_FAILED_COUNT,
        "prod_emb_failed": prod_emb_failed_count,
        "gold_emb_failed": gold_emb_failed_count,
    }
    drift_incomplete_note = (
        f" *(status: `{drift_status}`; prod_emb_failed={prod_emb_failed_count}, gold_emb_failed={gold_emb_failed_count})*"
        if EMBEDDING_INCOMPLETE
        else ""
    )
    _s5 = RESULTS["s5"]
    if coverage_is_semantic:
        _s51_excl_sfx = (
            f"; **{_s51_failed_rows}** golden row(s) with NULL/invalid distance excluded"
            if _s51_failed_rows
            else ""
        )
        _s5_coverage_line = (
            f"- **Golden Question Bank coverage**: **{_s5['coverage_pct']:.1f}%** (**{_s5['covered_golden']}/{_s5['total_golden']}** "
            f"golden questions covered within SDK cosine distance `{SDK_DRIFT_THRESHOLD}`; **{_s5['uncovered_golden']}** uncovered{_s51_excl_sfx})."
        )
    else:
        _s51_fb_count_sfx = (
            f"SDK 5.1 failed semantic rows: **{_s51_failed_rows}**"
            if _s51_failed_rows is not None
            else "SDK 5.1 per-row failed embedding count is **unknown** because `feedback.py` selects `AI.EMBED(...).result` without `.status`"
        )
        _s5_coverage_line = (
            f"- **Golden Question Bank semantic coverage (`UNAVAILABLE` — keyword-fallback coverage **{_s5['coverage_pct']:.1f}%**)**: "
            f"**{_s5['covered_golden']}/{_s5['total_golden']}** golden questions covered by keyword overlap (`client.drift_detection()` "
            f"fell back to `method='{coverage_method}'`, not a cosine-distance measurement; **{_s5['uncovered_golden']}** uncovered; {_s51_fb_count_sfx})."
        )
    if semantic_distances_ok:
        _cohort_drift_clause = (
            f"The most drifted cohort is `{_s5['top_drift_persona']}` (median cosine distance **{_s5['top_drift_median']:.3f}**)."
            if _s5["top_drift_median"] > 0.0
            else "No cohort showed median drift > 0.000 (all cohort median distances = **0.000**)."
        )
        _s5_drift_line = (
            f"- **Emerging / drifted user prompts**: **{_s5['drifted_over_015']}** production sessions had `cosine_distance > {DRIFT_THRESHOLD}` "
            f"from the nearest golden question. {_cohort_drift_clause}"
        )
    else:
        _s5_drift_line = (
            "- **Emerging / drifted user prompts**: no valid semantic distances (all production or golden embeddings failed, "
            "or 0 single-turn sessions in turns view); drift ranking unavailable — see the warning above."
        )
    show("**Section 5 Takeaway (live)**\n\n" + _s5_coverage_line + "\n" + _s5_drift_line + drift_incomplete_note)

## 6. Response-quality evaluation (Golden Q&A key grading & SDK judges)

### 6A. Automated Golden Q&A Answer Key Grading (`AI.GENERATE` over Section 5.2 Matched Pairs)

Because BQCA logs the user's question (`USER_MESSAGE_RECEIVED`) and the agent's final response (`AGENT_RESPONSE`) — and omits intermediate `TOOL_*` SQL result rows — there are two complementary ways to evaluate response quality:

- **6A: Automated Golden Q&A Answer Key Grading (Recommended for accuracy audits)**: Filter the in-domain production questions already matched in Section 5.2 (`question_distance <= DRIFT_THRESHOLD` — avoiding any duplicate `AI.EMBED` calls) and use `AI.GENERATE` (`gemini-3.5-flash`) to grade whether the agent's numbers and entities match the answer key (`PASS` / `FAIL`). When `bqca_golden_qa` contains SME-verified rows (`verified = TRUE`), this catches fluent factual discrepancies by comparing each in-domain answer's entities and numbers against your curated answer key (flagging, for example, the single seeded hallucination in the synthetic 25-session reference dataset described in the manual; note that LLM answer-key grading is bounded by the coverage and accuracy of your Golden Q&A rows).
- **6B: Reference-Free SDK Judges, `GraderPipeline`, and Multi-Trial Judge Consistency (`TrialRunner`)**: Run `LLMAsJudge.correctness()`, `LLMAsJudge.hallucination()`, a weighted `GraderPipeline`, and multi-trial `TrialRunner` judge consistency checks (`pass@3` and `pass^3`).

In [ ]:
#@title 6A. Golden Q&A Answer Key Grading (AI.GENERATE over Matched In-Domain Pairs)
GOLDEN_MATCH_MAX_DISTANCE = DRIFT_THRESHOLD
# Optional benchmark comparison session IDs (falls back automatically to your top/bottom scored sessions if absent):
HALLUCINATED_SESSION = "session_hallucinated_retrieval_demo"
REFERENCE_SESSION = "session_lena_top_math_01"

# Optional on Gemini 3.x: add "thinkingConfig": {"thinkingLevel": "LOW"} inside generationConfig to cap thinking tokens
GENERATE_PARAMS = """JSON '{"generationConfig": {"temperature": 0.0, "maxOutputTokens": 1024}}'"""
GRADER_INSTRUCTIONS = """You are grading an analytics agent answer against a verified golden answer.
Decide whether the AGENT ANSWER is factually consistent with the GOLDEN ANSWER.
verdict = PASS when every entity and number stated in the golden answer appears in the agent answer with the same value, and the agent answer does not contradict the golden answer. Additional correct detail, different formatting or wording, and a different ordering of the same facts are allowed.
verdict = FAIL when the agent answer omits a golden entity or number, states a different value or entity for the same quantity, invents entities or numbers that conflict with the golden answer, or gives no answer.
justification = one sentence naming the first mismatching value, or confirming the match."""

matched_df_local = globals().get("matched_df")
_s5_upstream_ok = bool(
    isinstance(matched_df_local, pd.DataFrame)
    and ("s5" in RESULTS and RESULTS["s5"].get("available", True))
)
# Reset Section 6A completion flags before running GRADE_SQL so a cancelled (499) or failed
# grading job never leaves GOLDEN_GRADING_AVAILABLE=True with an empty/stale graded_df:
GOLDEN_GRADING_AVAILABLE = False
GOLDEN_GRADING_COMPLETED = False
GOLDEN_GRADING_MODE = "not_run"
graded_df = None
verified_graded_df = pd.DataFrame(columns=["session_id", "verdict"])
template_graded_df = pd.DataFrame(columns=["session_id", "verdict"])
RESULTS.pop("s6", None)

if not _s5_upstream_ok:
    print("Skipping Golden Q&A grading: Section 5 did not complete (matched_df unavailable).")
    in_domain_matches_df = pd.DataFrame()
elif matched_df_local.empty or "question_distance" not in matched_df_local.columns:
    print("Skipping Golden Q&A grading: no matched pairs from Section 5.2 (matched_df empty or missing question_distance).")
    in_domain_matches_df = pd.DataFrame()
else:
    in_domain_matches_df = matched_df_local[matched_df_local.question_distance <= GOLDEN_MATCH_MAX_DISTANCE].copy()

GRADE_SQL = f"""
WITH in_domain AS (
  SELECT
    JSON_VALUE(item, '$.persona') AS persona,
    JSON_VALUE(item, '$.session_id') AS session_id,
    JSON_VALUE(item, '$.golden_id') AS golden_id,
    COALESCE(SAFE_CAST(JSON_VALUE(item, '$.golden_verified') AS BOOL), FALSE) AS golden_verified,
    JSON_VALUE(item, '$.source_session_id') AS source_session_id,
    SAFE_CAST(JSON_VALUE(item, '$.question_distance') AS FLOAT64) AS question_distance,
    JSON_VALUE(item, '$.prod_question') AS prod_question,
    JSON_VALUE(item, '$.golden_answer') AS golden_answer,
    JSON_VALUE(item, '$.prod_answer') AS prod_answer
  FROM UNNEST(JSON_QUERY_ARRAY(PARSE_JSON(@matched_json))) AS item
)
SELECT
  m.persona,
  m.session_id,
  m.golden_id,
  m.golden_verified AS verified,
  COALESCE(m.session_id = m.source_session_id, FALSE) AS is_self_seeded,
  ROUND(m.question_distance, 3) AS question_distance,
  CASE
    WHEN NULLIF(TRIM(gen.status), '') IS NOT NULL THEN 'ERROR'
    WHEN UPPER(TRIM(gen.verdict)) IN ('PASS', 'FAIL') THEN UPPER(TRIM(gen.verdict))
    ELSE 'ERROR'
  END AS verdict,
  COALESCE(NULLIF(gen.justification, ''), NULLIF(gen.status, ''), 'No justification returned') AS justification,
  gen.status AS judge_status
FROM (
  SELECT
    m.*,
    AI.GENERATE(
      prompt => CONCAT(
        @instructions, CHR(10), CHR(10),
        'Question: ', m.prod_question, CHR(10), CHR(10),
        'GOLDEN ANSWER (ground truth):', CHR(10), m.golden_answer, CHR(10), CHR(10),
        'AGENT ANSWER:', CHR(10), COALESCE(m.prod_answer, 'N/A')),
      endpoint => '{JUDGE_MODEL}',
      connection_id => '{VERTEX_CONNECTION_ID}',
      model_params => {GENERATE_PARAMS},
      output_schema => 'verdict STRING, justification STRING'
    ) AS gen
  FROM in_domain m
) AS m
ORDER BY verdict ASC, persona, session_id
"""


def _grade_in_domain_via_api(matches_df):
    """Grades pre-matched in-domain pairs directly via Vertex AI (google-genai) when in-warehouse AI.GENERATE times out or is cancelled."""
    from google import genai as _genai
    from google.genai import types as _genai_types

    _client = _genai.Client(vertexai=True, project=PROJECT_ID, location=VERTEX_LOCATION)
    rows = []
    try:
        for row in matches_df.itertuples(index=False):
            prod_ans = (
                str(row.prod_answer).strip()
                if pd.notna(row.prod_answer) and str(row.prod_answer).strip()
                else "N/A"
            )
            prompt = (
                f"{GRADER_INSTRUCTIONS}\n\n"
                f"Question: {row.prod_question}\n\n"
                f"GOLDEN ANSWER (ground truth):\n{row.golden_answer}\n\n"
                f"AGENT ANSWER:\n{prod_ans}\n\n"
                'Return a JSON object with keys "verdict" ("PASS" or "FAIL") and "justification" (one sentence).'
            )
            verdict = "ERROR"
            justification = "No justification returned"
            judge_status = ""
            for attempt in range(3):
                try:
                    resp = _client.models.generate_content(
                        model=DIRECT_JUDGE_MODEL,
                        contents=prompt,
                        config=_genai_types.GenerateContentConfig(
                            temperature=0.0,
                            max_output_tokens=1024,
                            response_mime_type="application/json",
                        ),
                    )
                    raw_txt = (getattr(resp, "text", None) or "").strip()
                    if raw_txt.startswith("```"):
                        raw_txt = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_txt, flags=re.IGNORECASE | re.DOTALL).strip()
                    parsed = json.loads(raw_txt)
                    if isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):
                        parsed = parsed[0]
                    v_str = str((parsed or {}).get("verdict") or "").strip().upper()
                    j_str = str((parsed or {}).get("justification") or "").strip()
                    if v_str in ("PASS", "FAIL"):
                        verdict = v_str
                        justification = j_str or "Confirmed by direct Vertex AI grader."
                        judge_status = ""
                    else:
                        verdict = "ERROR"
                        justification = j_str or raw_txt or "Invalid verdict in JSON response"
                        judge_status = "invalid_verdict"
                    break
                except Exception as exc:
                    err_low = str(exc).lower()
                    if attempt < 2 and any(k in err_low for k in ("429", "resource_exhausted", "503", "unavailable", "deadline_exceeded")):
                        time.sleep(1.0 * (attempt + 1))
                        continue
                    verdict = "ERROR"
                    justification = str(exc)
                    judge_status = str(exc)
                    break
            source_sid = getattr(row, "source_session_id", None)
            rows.append({
                "persona": row.persona,
                "session_id": row.session_id,
                "golden_id": row.golden_id,
                "verified": bool(getattr(row, "golden_verified", False)),
                "is_self_seeded": bool(pd.notna(source_sid) and str(row.session_id) == str(source_sid)),
                "question_distance": round(float(row.question_distance), 3),
                "verdict": verdict,
                "justification": justification,
                "judge_status": judge_status,
            })
    finally:
        _client.close()
    if not rows:
        return pd.DataFrame(
            columns=["persona", "session_id", "golden_id", "verified", "is_self_seeded", "question_distance", "verdict", "justification", "judge_status"]
        )
    return pd.DataFrame(rows).sort_values(["verdict", "persona", "session_id"]).reset_index(drop=True)


if not _s5_upstream_ok:
    graded_df = pd.DataFrame(
        columns=["persona", "session_id", "golden_id", "verified", "is_self_seeded", "question_distance", "verdict", "justification", "judge_status"]
    )
elif in_domain_matches_df.empty:
    graded_df = pd.DataFrame(
        columns=["persona", "session_id", "golden_id", "verified", "is_self_seeded", "question_distance", "verdict", "justification", "judge_status"]
    )
    GOLDEN_GRADING_MODE = "ai_generate"
    GOLDEN_GRADING_COMPLETED = True
    GOLDEN_GRADING_AVAILABLE = True
else:
    matched_payload_cols = [
        "persona", "session_id", "golden_id", "golden_verified", "source_session_id",
        "question_distance", "prod_question", "golden_answer", "prod_answer",
    ]
    _allow_golden_api_fallback = os.environ.get("BQCA_GOLDEN_GRADING_API_FALLBACK", "true").strip().lower() not in (
        "0",
        "false",
        "no",
    )
    try:
        with timed(f"Golden Q&A Answer Key grading via AI.GENERATE ({JUDGE_MODEL}, n={len(in_domain_matches_df)} pre-matched pairs)"):
            graded_df = run_sql(
                GRADE_SQL,
                {
                    "instructions": GRADER_INSTRUCTIONS,
                    "matched_json": in_domain_matches_df[matched_payload_cols].to_json(orient="records"),
                },
                retries=0,
                timeout_sec=float(AI_SQL_TIMEOUT_SEC),
            )
        graded_df["verdict"] = graded_df["verdict"].fillna("ERROR").astype("string").str.strip().str.upper()
        if "judge_status" in graded_df.columns:
            _has_status_err = graded_df["judge_status"].fillna("").astype("string").str.strip() != ""
            graded_df.loc[_has_status_err, "verdict"] = "ERROR"
        _invalid_verdict = ~graded_df["verdict"].isin(["PASS", "FAIL", "ERROR"])
        if bool(_invalid_verdict.any()):
            if "judge_status" in graded_df.columns:
                _empty_status = graded_df["judge_status"].fillna("").astype("string").str.strip() == ""
                graded_df.loc[_invalid_verdict & _empty_status, "judge_status"] = "invalid_verdict"
            graded_df.loc[_invalid_verdict, "verdict"] = "ERROR"
        GOLDEN_GRADING_MODE = "ai_generate"
        GOLDEN_GRADING_COMPLETED = True
        GOLDEN_GRADING_AVAILABLE = True
    except Exception as _grade_exc:
        if _allow_golden_api_fallback and isinstance(
            _grade_exc,
            (
                TimeoutError,
                concurrent.futures.TimeoutError,
                gcp_exceptions.DeadlineExceeded,
                gcp_exceptions.Cancelled,
                gcp_exceptions.ServiceUnavailable,
                gcp_exceptions.TooManyRequests,
                gcp_exceptions.InternalServerError,
            ),
        ):
            if getattr(_grade_exc, "cancel_confirmed", True) is False:
                print(
                    f"⚠️ WARNING: warehouse grading job_id={getattr(_grade_exc, 'job_id', None) or 'unknown'} timed out and its "
                    "cancellation could not be confirmed (see the warning above), so it may still be RUNNING and billing in "
                    "BigQuery while the direct Vertex AI fallback grading below also runs."
                )
            print(
                f"⚠️ In-warehouse AI.GENERATE grading did not complete ({type(_grade_exc).__name__}: {_grade_exc}). "
                f"Falling back to direct Vertex AI grading (execution_mode='api_fallback', model={DIRECT_JUDGE_MODEL}, n={len(in_domain_matches_df)})..."
            )
            with timed(f"Golden Q&A Answer Key grading via direct Vertex AI fallback ({DIRECT_JUDGE_MODEL}, n={len(in_domain_matches_df)})"):
                _fb_df = _grade_in_domain_via_api(in_domain_matches_df)
            if not _fb_df.empty and bool(_fb_df["verdict"].isin(["PASS", "FAIL"]).any()):
                graded_df = _fb_df
                GOLDEN_GRADING_MODE = "api_fallback"
                GOLDEN_GRADING_COMPLETED = True
                GOLDEN_GRADING_AVAILABLE = True
            else:
                GOLDEN_GRADING_MODE = "failed"
                GOLDEN_GRADING_COMPLETED = False
                GOLDEN_GRADING_AVAILABLE = False
                graded_df = _fb_df if isinstance(_fb_df, pd.DataFrame) and not _fb_df.empty else None
                if isinstance(graded_df, pd.DataFrame) and not graded_df.empty:
                    verified_graded_df = graded_df[graded_df["verified"] & ~graded_df["is_self_seeded"]]
                    template_graded_df = graded_df[~graded_df["verified"] | graded_df["is_self_seeded"]]
                else:
                    verified_graded_df = pd.DataFrame(columns=["session_id", "verdict"])
                    template_graded_df = pd.DataFrame(columns=["session_id", "verdict"])
                raise _grade_exc
        else:
            GOLDEN_GRADING_MODE = "failed"
            GOLDEN_GRADING_COMPLETED = False
            GOLDEN_GRADING_AVAILABLE = False
            graded_df = None
            raise

if isinstance(graded_df, pd.DataFrame) and not graded_df.empty:
    verified_graded_df = graded_df[graded_df["verified"] & ~graded_df["is_self_seeded"]]
    template_graded_df = graded_df[~graded_df["verified"] | graded_df["is_self_seeded"]]
else:
    verified_graded_df = pd.DataFrame(columns=["session_id", "verdict"])
    template_graded_df = pd.DataFrame(columns=["session_id", "verdict"])

if GOLDEN_GRADING_COMPLETED and isinstance(graded_df, pd.DataFrame):
    if not template_graded_df.empty:
        print(
            f"⚠️ Note: {len(template_graded_df)}/{len(graded_df)} graded sessions matched unverified auto-seeded template rows "
            "(pipeline smoke test only). Verify rows in bqca_golden_qa (SET verified = TRUE) for ground-truth accuracy."
        )
    if in_domain_matches_df.empty and matched_df_local is not None and not matched_df_local.empty:
        _min_dist = float(matched_df_local["question_distance"].min())
        print(
            f"💡 Tip: 0 sessions matched within GOLDEN_MATCH_MAX_DISTANCE ({GOLDEN_MATCH_MAX_DISTANCE}; closest distance was {_min_dist:.3f}). "
            f"To grade more sessions, curate matching questions in {GOLDEN_QA_TABLE_ID} (or cautiously adjust DRIFT_THRESHOLD / GOLDEN_MATCH_MAX_DISTANCE, e.g. to 0.20–0.25; higher thresholds risk pairing unrelated questions)."
        )
    _g_eval_n = int(graded_df["verdict"].isin(["PASS", "FAIL"]).sum())
    _g_err_n = int((graded_df["verdict"] == "ERROR").sum())
    _ver_eval_n = int(verified_graded_df["verdict"].isin(["PASS", "FAIL"]).sum())
    _ver_err_n = int((verified_graded_df["verdict"] == "ERROR").sum())
    _tpl_eval_n = int(template_graded_df["verdict"].isin(["PASS", "FAIL"]).sum())
    _tpl_err_n = int((template_graded_df["verdict"] == "ERROR").sum())
    print(
        f"Graded {_g_eval_n}/{len(graded_df)} in-domain sessions (question_distance <= {GOLDEN_MATCH_MAX_DISTANCE}, mode={GOLDEN_GRADING_MODE}) against {GOLDEN_QA_TABLE_ID} "
        + (f"[{_g_err_n} ERROR/unavailable] " if _g_err_n else "")
        + f"(verified ground-truth evaluated: {_ver_eval_n}/{len(verified_graded_df)}"
        + (f" [+{_ver_err_n} ERROR]" if _ver_err_n else "")
        + f", unverified template smoke-test evaluated: {_tpl_eval_n}/{len(template_graded_df)}"
        + (f" [+{_tpl_err_n} ERROR]" if _tpl_err_n else "")
        + "; showing up to 25):"
    )
    display(graded_df.drop(columns=["is_self_seeded", "judge_status"], errors="ignore").head(25))

# Side-by-side inspection of any FAIL verdict
failed_grade_df = (
    graded_df[graded_df.verdict == "FAIL"]
    if (GOLDEN_GRADING_COMPLETED and isinstance(graded_df, pd.DataFrame) and not graded_df.empty)
    else pd.DataFrame()
)
if not failed_grade_df.empty:
    bad_sid = failed_grade_df.session_id.iloc[0]
    bad_match = matched_df_local.set_index("session_id").loc[bad_sid]
    bad_grade = graded_df.set_index("session_id").loc[bad_sid]
    side_by_side_df = pd.DataFrame({
        "Stage": [f"Golden Answer ({bad_match.golden_id})", f"Logged Agent Answer ({bad_sid})", f"Golden Q&A Grader Verdict ({GOLDEN_GRADING_MODE})"],
        "Content": [
            short(bad_match.golden_answer, 420),
            short(bad_match.prod_answer, 420),
            f"{bad_grade.verdict}: {bad_grade.justification}",
        ],
    })
    show("**Side-by-side audit of the failing session caught by Golden Q&A grading**\n\n" + md_table(side_by_side_df, max_chars=430))

### 6B. Reference-free SDK judges (`correctness` & `hallucination`), `GraderPipeline`, and multi-trial judge consistency (`TrialRunner`)

Below we run the SDK's built-in `LLMAsJudge.correctness()` and `LLMAsJudge.hallucination()`, combine them with a latency SLO in `GraderPipeline`, and provide the drop-in `BQCAPerformanceEvaluator` adapter so `TrialRunner` reads `AGENT_RESPONSE` on BQCA logs. Because `TrialRunner` re-evaluates a logged session across `k=3` independent judge trials, `pass@3` and `pass^3` measure **multi-trial LLM judge consistency** on that trace under two explicit gating modes (`trial_gate_mode`):

- **`verified_golden_answer` mode** *(when the session has an SME-verified `verified = TRUE`, non-self-seeded golden answer in `bqca_golden_qa`)*: gates directly on `llm_judge_final_answer_correct >= JUDGE_THRESHOLD` (setting composite `llm_judge_correctness`, `llm_judge_tool_usage_correct`, and `llm_judge_sound_reasoning` to `0.0` so unlogged `TOOL_*` events and intermediate reasoning trajectories in BQCA never cause false failures).
- **`reference_free_sentiment` mode** *(when `golden_ref is None` — the default on a freshly auto-seeded `verified = FALSE` table or when no golden question matches within `GOLDEN_MATCH_MAX_DISTANCE`)*: gates on `llm_judge_sentiment >= JUDGE_THRESHOLD` while keeping `llm_judge_hallucination` informational at `0.0` (since its rubric grades grounding in tool calls). This keeps a positive `JUDGE_THRESHOLD` gate active so a `0.0` score never passes, while noting that separating a fluent factual hallucination from an accurate answer in `TrialRunner` requires a `verified = TRUE` golden reference.

If any of the `k=3` trials cannot be evaluated (a Vertex AI `429`, a timeout, or an empty/invalid judge reply), `pass@3` and `pass^3` are reported as **unavailable** for that session: they are never recomputed over fewer trials and still labelled `pass@3` / `pass^3`, and `trial_status` shows the partial counts. A judge outage is never displayed as a measured failure.

In [ ]:
#@title 6B. Reference-Free Judges, GraderPipeline & Multi-Trial Consistency (TrialRunner)
with timed(f"LLMAsJudge.correctness & LLMAsJudge.hallucination on {COMPAT_VIEW_ID} ({JUDGE_MODEL})"):
    corr_report = client.evaluate(
        LLMAsJudge.correctness(threshold=JUDGE_THRESHOLD, model=JUDGE_MODEL),
        filters=EVAL_FILTER,
    )
    hall_report = client.evaluate(
        LLMAsJudge.hallucination(threshold=JUDGE_THRESHOLD, model=JUDGE_MODEL),
        filters=EVAL_FILTER,
    )
# Keep report-level counts on the evaluated denominator (judge errors -> unevaluated, not failed):
_recompute_eval_report_counts(corr_report)
_recompute_eval_report_counts(hall_report)

corr_df = score_frame(corr_report).set_index("session_id")
corr_df = corr_df[~corr_df.index.duplicated(keep="first")]
hall_df = score_frame(hall_report).set_index("session_id")
hall_df = hall_df[~hall_df.index.duplicated(keep="first")]
if "correctness" not in corr_df.columns:
    corr_df["correctness"] = pd.NA
if "faithfulness" not in hall_df.columns:
    hall_df["faithfulness"] = pd.NA
corr_mode = corr_report.details.get("execution_mode", "unknown")
hall_mode = hall_report.details.get("execution_mode", "unknown")
corr_not_evaluated = int(corr_df["correctness"].isna().sum()) if not corr_df.empty else 0
hall_not_evaluated = int(hall_df["faithfulness"].isna().sum()) if not hall_df.empty else 0
corr_evaluated = len(corr_df) - corr_not_evaluated
hall_evaluated = len(hall_df) - hall_not_evaluated
corr_passed_n = int(corr_df["passed"].sum())
hall_passed_n = int(hall_df["passed"].sum())
corr_pass_rate = (corr_passed_n / corr_evaluated) if corr_evaluated > 0 else None
hall_pass_rate = (hall_passed_n / hall_evaluated) if hall_evaluated > 0 else None
if len(corr_df) > 0 and corr_not_evaluated >= len(corr_df):
    print(f"LLMAsJudge.correctness: no live evaluations — all {len(corr_df)} judge calls failed (mode={corr_mode})")
elif corr_evaluated == 0:
    print(f"LLMAsJudge.correctness: no sessions matched EVAL_FILTER (mode={corr_mode})")
else:
    print(
        f"LLMAsJudge.correctness: {corr_passed_n}/{corr_evaluated} evaluated sessions passed ({corr_pass_rate:.0%} of evaluated, mode={corr_mode})"
        + (f" [{corr_not_evaluated} of {len(corr_df)} unevaluated: judge errors]" if corr_not_evaluated else "")
    )
if len(hall_df) > 0 and hall_not_evaluated >= len(hall_df):
    print(f"LLMAsJudge.hallucination: no live evaluations — all {len(hall_df)} judge calls failed (mode={hall_mode})")
elif hall_evaluated == 0:
    print(f"LLMAsJudge.hallucination: no sessions matched EVAL_FILTER (mode={hall_mode})")
else:
    print(
        f"LLMAsJudge.hallucination: {hall_passed_n}/{hall_evaluated} evaluated sessions passed ({hall_pass_rate:.0%} of evaluated, mode={hall_mode})"
        + (f" [{hall_not_evaluated} of {len(hall_df)} unevaluated: judge errors]" if hall_not_evaluated else "")
    )

judge_summary_df = pd.DataFrame({
    "persona": corr_df["persona"] if "persona" in corr_df.columns else pd.Series("unattributed", index=corr_df.index),
    "correctness": corr_df["correctness"],
    "correctness_passed": corr_df["passed"],
    "faithfulness": hall_df["faithfulness"],
    "faithfulness_passed": hall_df["passed"],
})

# Resolve optional upstream variables so Section 6B can also run standalone without Sections 5.2 & 6A:
ref_sid = globals().get("REFERENCE_SESSION", "session_lena_top_math_01")
hal_sid = globals().get("HALLUCINATED_SESSION", "session_hallucinated_retrieval_demo")
golden_max_dist = float(globals().get("GOLDEN_MATCH_MAX_DISTANCE", DRIFT_THRESHOLD))
turns_df_local = globals().get("turns_df", pd.DataFrame())
_raw_matched_df = globals().get("matched_df")
_s5_available = bool(
    isinstance(_raw_matched_df, pd.DataFrame)
    and ("s5" in RESULTS and RESULTS["s5"].get("available", True))
)
_raw_graded_df = globals().get("graded_df")
golden_grading_completed = bool(
    _s5_available
    and globals().get("GOLDEN_GRADING_COMPLETED", False)
    and isinstance(_raw_graded_df, pd.DataFrame)
)
golden_grading_available = golden_grading_completed
golden_grading_mode = str(globals().get("GOLDEN_GRADING_MODE", "ai_generate" if golden_grading_completed else "not_run"))
matched_df_local = _raw_matched_df if _s5_available else pd.DataFrame()
def _norm_golden_verdicts(df, status_map=None):
    if not isinstance(df, pd.DataFrame) or df.empty or "verdict" not in df.columns:
        return df if isinstance(df, pd.DataFrame) else pd.DataFrame(columns=["session_id", "verdict"])
    out = df.copy()
    v = out["verdict"].fillna("ERROR").astype("string").str.strip().str.upper()
    if "judge_status" in out.columns:
        has_err = out["judge_status"].fillna("").astype("string").str.strip() != ""
        v = v.where(~has_err, "ERROR")
    if status_map and "session_id" in out.columns:
        mapped_err = out["session_id"].map(status_map).fillna(False)
        v = v.where(~mapped_err, "ERROR")
    v = v.where(v.isin(["PASS", "FAIL", "ERROR"]), "ERROR")
    out["verdict"] = v
    return out


_has_graded_rows = bool(
    _s5_available and isinstance(_raw_graded_df, pd.DataFrame) and not _raw_graded_df.empty
)
_use_graded_frames = golden_grading_completed or _has_graded_rows
graded_df_local = (
    _norm_golden_verdicts(_raw_graded_df)
    if _use_graded_frames
    else pd.DataFrame(columns=["session_id", "verdict"])
)
_err_by_sid = (
    (graded_df_local.set_index("session_id")["verdict"] == "ERROR").to_dict()
    if (_use_graded_frames and not graded_df_local.empty and "session_id" in graded_df_local.columns)
    else {}
)
verified_graded_df_local = (
    _norm_golden_verdicts(globals().get("verified_graded_df", pd.DataFrame(columns=["session_id", "verdict"])), _err_by_sid)
    if _use_graded_frames
    else pd.DataFrame(columns=["session_id", "verdict"])
)
template_graded_df_local = (
    _norm_golden_verdicts(globals().get("template_graded_df", pd.DataFrame(columns=["session_id", "verdict"])), _err_by_sid)
    if _use_graded_frames
    else pd.DataFrame(columns=["session_id", "verdict"])
)

# Select target sessions for deep-dive comparison (uses the benchmark twin pair if present, else top-1 and bottom-1 sessions):
target_sids = [s for s in (ref_sid, hal_sid) if s in judge_summary_df.index]
if len(target_sids) < 2:
    sorted_sids = list(judge_summary_df.sort_values(["correctness", "faithfulness"], ascending=[False, False]).index)
    target_sids = [sorted_sids[0], sorted_sids[-1]] if len(sorted_sids) >= 2 else sorted_sids

print("Reference-free SDK judge summary on target comparison sessions:")
display(judge_summary_df.loc[target_sids])


class BQCAPerformanceEvaluator(PerformanceEvaluator):
    """Adapter for BQCA logs: populates SessionTrace.final_response from AGENT_RESPONSE."""

    async def get_session_trace(self, session_id, **kwargs):
        session_trace = await super().get_session_trace(session_id, **kwargs)
        session_trace.final_response = (
            session_trace.final_response or _extract_bqca_final_response(session_trace.events)
        )
        return session_trace


_DEFAULT_TRACE_EVENT_TYPES = getattr(
    PerformanceEvaluator,
    "_DEFAULT_EVENT_TYPES",
    (
        "USER_MESSAGE_RECEIVED",
        "AGENT_STARTING",
        "AGENT_COMPLETED",
        "TOOL_STARTING",
        "TOOL_COMPLETED",
        "TOOL_ERROR",
        "LLM_REQUEST",
        "LLM_RESPONSE",
        "LLM_ERROR",
        "INVOCATION_STARTING",
        "INVOCATION_COMPLETED",
        "STATE_DELTA",
        "HITL_CONFIRMATION_REQUEST",
        "HITL_CONFIRMATION_REQUEST_COMPLETED",
        "HITL_CREDENTIAL_REQUEST",
        "HITL_CREDENTIAL_REQUEST_COMPLETED",
        "HITL_INPUT_REQUEST",
        "HITL_INPUT_REQUEST_COMPLETED",
        "AGENT_TRANSFER",
        "EVENT_COMPACTION",
        "AGENT_STATE_CHECKPOINT",
        "TOOL_PAUSED",
        "WORKFLOW_NODE_STARTING",
        "WORKFLOW_NODE_COMPLETED",
    ),
)

bqca_evaluator = BQCAPerformanceEvaluator(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=COMPAT_VIEW_ID,
    client=bq,
    llm_judge_model=DIRECT_JUDGE_MODEL,
    include_event_types=list(_DEFAULT_TRACE_EVENT_TYPES) + ["AGENT_RESPONSE"],
)

# Run GraderPipeline (weighted latency + correctness + hallucination) and TrialRunner (3 judge consistency trials)
PIPELINE_WEIGHTS = {"latency_evaluator": 0.2, "correctness_judge": 0.5, "hallucination_judge": 0.3}
INFORMATIONAL_METRICS = [
    "response_match", "llm_judge_sentiment", "llm_judge_hallucination",
    "llm_judge_tool_usage_correct", "llm_judge_sound_reasoning", "llm_judge_efficiency",
]
# When a verified golden reference exists, gate directly on final_answer_correct >= JUDGE_THRESHOLD
# (setting composite llm_judge_correctness, tool_usage_correct, and sound_reasoning to 0.0 so unlogged BQCA tool/reasoning traces never cause false failures).
VERIFIED_TRIAL_THRESHOLDS = {
    **{m: 0.0 for m in INFORMATIONAL_METRICS},
    "llm_judge_correctness": 0.0,
    "llm_judge_final_answer_correct": float(JUDGE_THRESHOLD),
}
assert (
    VERIFIED_TRIAL_THRESHOLDS["llm_judge_final_answer_correct"] == float(JUDGE_THRESHOLD)
    and VERIFIED_TRIAL_THRESHOLDS["llm_judge_sound_reasoning"] == 0.0
)
# When no verified golden reference exists, keep a positive JUDGE_THRESHOLD gate on llm_judge_sentiment
# so a 0.0 score never passes while keeping tool-grounded llm_judge_hallucination informational at 0.0.
REFERENCE_FREE_TRIAL_THRESHOLDS = {
    "llm_judge_sentiment": float(JUDGE_THRESHOLD),
    "llm_judge_hallucination": 0.0,
}

# Only pass a golden_response reference to TrialRunner when the session has a verified, non-self-seeded in-domain golden match:
verified_golden_by_session = {}
trial_gate_reason_by_session = {}
if _s5_available and not matched_df_local.empty:
    for _row in matched_df_local.itertuples():
        _dist = getattr(_row, "question_distance", None)
        if pd.isna(_dist) or _dist > golden_max_dist:
            trial_gate_reason_by_session[_row.session_id] = "out_of_domain_distance"
        elif not bool(getattr(_row, "golden_verified", False)):
            trial_gate_reason_by_session[_row.session_id] = "unverified_golden_row"
        elif _row.session_id == getattr(_row, "source_session_id", None):
            trial_gate_reason_by_session[_row.session_id] = "self_seeded_golden_row"
        else:
            verified_golden_by_session[_row.session_id] = _row.golden_answer
            trial_gate_reason_by_session[_row.session_id] = "verified_golden_match"
graded_verdict_by_session = (
    graded_df_local.set_index("session_id")["verdict"].to_dict() if (golden_grading_completed and not graded_df_local.empty) else {}
)
runner = TrialRunner(bqca_evaluator, num_trials=3, concurrency=3)

twin_eval_rows = []
for sid in target_sids:
    try:
        summary_sql = SESSION_SUMMARY_QUERY.format(
            project=PROJECT_ID, dataset=DATASET_ID, table=COMPAT_VIEW_ID, where="session_id = @sid"
        )
        job_config = bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("sid", "STRING", sid),
                bigquery.ScalarQueryParameter("trace_limit", "INT64", 10),
            ],
            maximum_bytes_billed=int(MAX_BYTES_BILLED),
        )
        summary_rows = list(bq.query(summary_sql, job_config=job_config).result())
        if not summary_rows:
            continue
        summary = dict(summary_rows[0])
        trace = client.get_session_trace(sid)
        trace_text = "\n".join(
            f"{s.event_type}: {_extract_bqca_span_text(s) or getattr(s, 'summary', '') or ''}"
            for s in trace.spans
        )
        answer = trace.final_response or ""

        pipeline = GraderPipeline(WeightedStrategy(weights=PIPELINE_WEIGHTS, threshold=JUDGE_THRESHOLD))
        pipeline.add_system_grader(SystemEvaluator.latency(threshold_ms=10000))
        pipeline.add_llm_grader(LLMAsJudge.correctness(threshold=JUDGE_THRESHOLD, model=DIRECT_JUDGE_MODEL))
        pipeline.add_llm_grader(LLMAsJudge.hallucination(threshold=JUDGE_THRESHOLD, model=DIRECT_JUDGE_MODEL))
        pipe_verdict = await pipeline.evaluate(
            session_summary=summary, trace_text=trace_text, final_response=answer, session_id=sid
        )
        # An LLM grader whose judge call failed (429 / 503 / empty or malformed reply) comes back from the Cell 1.2
        # AggregateGrader._run_grader wrapper as a _BQCAGraderResult with EMPTY scores and a structured evaluation_error;
        # WeightedStrategy would still fold it in as 0.0 and print a misleading 0.20 latency-only composite. Report the
        # verdict as unavailable instead of a measured quality score.
        _pipe_llm_errors = [
            r.grader_name
            for r in (getattr(pipe_verdict, "grader_results", None) or [])
            if r.grader_name != "latency_evaluator" and (not r.scores or getattr(r, "evaluation_error", None))
        ]
        if _pipe_llm_errors:
            pipeline_score, pipeline_passed = None, None
            pipeline_status = f"unavailable (judge error: {', '.join(_pipe_llm_errors)})"
        else:
            pipeline_score = round(float(pipe_verdict.final_score), 2)
            pipeline_passed = bool(pipe_verdict.passed)
            pipeline_status = "evaluated"
        golden_ref = verified_golden_by_session.get(sid)
        if golden_ref is not None:
            trial_mode = "verified_golden_answer"
            trial_thresholds = VERIFIED_TRIAL_THRESHOLDS
        else:
            trial_mode = "reference_free_sentiment"
            trial_thresholds = REFERENCE_FREE_TRIAL_THRESHOLDS
        trial_rep = await runner.run_trials(
            sid,
            golden_response=golden_ref,
            use_llm_judge=True,
            task_description="Answer the user's analytics question accurately.",
            thresholds=trial_thresholds,
        )
        # A trial whose judge call failed carries details["errors"] and lacks the gated metric; stock pass@k / pass^k
        # would count it as a 0.0 failure. pass@3 / pass^3 are defined over exactly 3 judged trials, so when ANY trial is
        # unevaluated the k-trial metrics are UNAVAILABLE (never recomputed over fewer trials and still labelled
        # pass@3 / pass^3); trial_status keeps the partial counts for diagnosis.
        _req_trial_key = "llm_judge_final_answer_correct" if golden_ref is not None else "llm_judge_sentiment"
        _trial_results = list(getattr(trial_rep, "trial_results", None) or [])
        _valid_trials = [
            t
            for t in _trial_results
            if (getattr(t, "scores", None) or {}).get(_req_trial_key) is not None
            and not (getattr(t, "details", None) or {}).get("errors")
        ]
        _n_trials = len(_trial_results) or int(getattr(trial_rep, "num_trials", 3) or 3)
        _n_valid = len(_valid_trials)
        _n_passed = sum(1 for t in _valid_trials if bool(getattr(t, "passed", False)))
        if _n_valid == _n_trials:
            trial_pass_at_k = round(compute_pass_at_k(_n_valid, _n_passed), 2)
            trial_pass_pow_k = round(compute_pass_pow_k(_n_valid, _n_passed), 2)
            trial_status = f"evaluated ({_n_valid}/{_n_trials} trials)"
        elif _n_valid == 0:
            trial_pass_at_k, trial_pass_pow_k = None, None
            trial_status = f"unavailable (0/{_n_trials} trials evaluated: judge error)"
        else:
            trial_pass_at_k, trial_pass_pow_k = None, None
            trial_status = (
                f"unavailable ({_n_valid}/{_n_trials} trials evaluated, {_n_passed}/{_n_valid} passed: judge error — "
                f"all {_n_trials} trials required for pass@{_n_trials}/pass^{_n_trials})"
            )
        _corr_cell = judge_summary_df.loc[sid, "correctness"]
        _faith_cell = judge_summary_df.loc[sid, "faithfulness"]
        twin_eval_rows.append({
            "session_id": sid,
            "sql_golden_verdict": graded_verdict_by_session.get(sid, "UNGRADED"),
            "sdk_correctness": None if pd.isna(_corr_cell) else float(_corr_cell),
            "sdk_faithfulness": None if pd.isna(_faith_cell) else float(_faith_cell),
            "pipeline_score": pipeline_score,
            "pipeline_passed": pipeline_passed,
            "pipeline_status": pipeline_status,
            "trial_gate_mode": trial_mode,
            "trial_gate_reason": trial_gate_reason_by_session.get(sid, "no_golden_match"),
            "trial_pass@3": trial_pass_at_k,
            "trial_pass^3": trial_pass_pow_k,
            "trial_status": trial_status,
        })
    except Exception as exc:
        print(f"Skipping direct Vertex AI judge evaluation for {sid}: {exc}")

gc.collect()
await asyncio.sleep(0.05)

twin_eval_df = pd.DataFrame(twin_eval_rows)
print("Side-by-side comparison of all quality evaluators on target sessions:")
display(twin_eval_df.drop(columns=["trial_gate_reason"], errors="ignore"))

verdict_counts = graded_df_local.verdict.value_counts().to_dict() if (_use_graded_frames and not graded_df_local.empty) else {}
verified_counts = verified_graded_df_local.verdict.value_counts().to_dict() if (_use_graded_frames and not verified_graded_df_local.empty) else {}
template_counts = template_graded_df_local.verdict.value_counts().to_dict() if (_use_graded_frames and not template_graded_df_local.empty) else {}
_pass_n = int(verdict_counts.get("PASS", 0)) if golden_grading_completed else 0
_fail_n = int(verdict_counts.get("FAIL", 0)) if golden_grading_completed else 0
_err_n = int(verdict_counts.get("ERROR", 0))
_eval_n = _pass_n + _fail_n
_attempted_n = len(graded_df_local) if _use_graded_frames else 0
_ver_pass = int(verified_counts.get("PASS", 0)) if golden_grading_completed else 0
_ver_fail = int(verified_counts.get("FAIL", 0)) if golden_grading_completed else 0
_ver_err = int(verified_counts.get("ERROR", 0))
_ver_eval = _ver_pass + _ver_fail
_ver_attempted = len(verified_graded_df_local) if _use_graded_frames else 0
_tpl_pass = int(template_counts.get("PASS", 0)) if golden_grading_completed else 0
_tpl_fail = int(template_counts.get("FAIL", 0)) if golden_grading_completed else 0
_tpl_err = int(template_counts.get("ERROR", 0))
_tpl_eval = _tpl_pass + _tpl_fail
_tpl_attempted = len(template_graded_df_local) if _use_graded_frames else 0
twin_dict = twin_eval_df.set_index("session_id").to_dict(orient="index") if not twin_eval_df.empty else {}
emb_failed_local = int(globals().get("EMBEDDING_FAILED_COUNT", 0))
prod_emb_failed_local = int(globals().get("PROD_EMBEDDING_FAILED_COUNT", emb_failed_local))
gold_emb_failed_local = int(globals().get("GOLD_EMBEDDING_FAILED_COUNT", 0))
no_gold_emb_local = int(globals().get("no_gold_emb_count", 0))
# Compute in-domain and out-of-domain counts directly from matched_df_local["question_distance"]
# (never by subtracting len(graded_df_local) from len(matched_df_local), which would misreport
# all matched sessions as out-of-domain if Cell 6A failed or was cancelled):
if _s5_available and not matched_df_local.empty and "question_distance" in matched_df_local.columns:
    _valid_dist = matched_df_local["question_distance"].dropna()
    in_domain_matched_count = int((_valid_dist <= golden_max_dist).sum())
    out_of_domain_count = int((_valid_dist > golden_max_dist).sum())
else:
    in_domain_matched_count = 0
    out_of_domain_count = 0
ungraded_in_domain_count = int(max(0, in_domain_matched_count - _attempted_n))
unsampled_or_multiturn_count = (
    int(max(0, len(turns_df_local) - len(matched_df_local) - prod_emb_failed_local - no_gold_emb_local))
    if _s5_available
    else 0
)
RESULTS["s6"] = {
    "s5_available": _s5_available,
    "golden_grading_available": golden_grading_completed,
    "golden_grading_completed": golden_grading_completed,
    "golden_grading_mode": golden_grading_mode,
    "golden_max_dist": golden_max_dist,
    "in_domain_matched": in_domain_matched_count,
    "ungraded_in_domain": ungraded_in_domain_count,
    "attempted": _attempted_n,
    "evaluated": _eval_n,
    "unavailable": _err_n,
    "graded": _eval_n,
    "verified_attempted": _ver_attempted,
    "verified_graded": _ver_eval,
    "verified_pass": _ver_pass,
    "verified_fail": _ver_fail,
    "verified_error": _ver_err,
    "template_attempted": _tpl_attempted,
    "template_graded": _tpl_eval,
    "template_pass": _tpl_pass,
    "template_fail": _tpl_fail,
    "template_error": _tpl_err,
    "pass": _pass_n,
    "fail": _fail_n,
    "error": _err_n,
    "out_of_domain": out_of_domain_count,
    "embedding_failed": emb_failed_local,
    "prod_embedding_failed": prod_emb_failed_local,
    "golden_embedding_failed": gold_emb_failed_local,
    "unsampled_or_multiturn": unsampled_or_multiturn_count,
    "failed_sessions": graded_df_local[graded_df_local.verdict == "FAIL"].session_id.tolist() if (golden_grading_completed and not graded_df_local.empty) else [],
    "corr_passed": corr_passed_n,
    "hall_passed": hall_passed_n,
    "corr_pass_rate": corr_pass_rate,
    "hall_pass_rate": hall_pass_rate,
    "corr_evaluated": corr_evaluated,
    "hall_evaluated": hall_evaluated,
    "corr_not_evaluated": corr_not_evaluated,
    "hall_not_evaluated": hall_not_evaluated,
    "corr_mode": corr_mode,
    "hall_mode": hall_mode,
    "twin_df": twin_dict,
}
def _fmt_rf_judge(label, rate, passed_c, eval_c, not_eval_c, mode_s):
    if eval_c > 0 and rate is not None:
        return (
            f"`{label}` pass rate **{rate:.0%}** (**{passed_c}/{eval_c}**, `{mode_s}`)"
            + (f" (**{not_eval_c}** unevaluated: judge error)" if not_eval_c > 0 else "")
        )
    if not_eval_c > 0:
        return f"`{label}` **unavailable** (**{not_eval_c}** session(s) unevaluated: judge error, `{mode_s}`)"
    return f"`{label}` **unavailable** (0 sessions evaluated, `{mode_s}`)"


_s6c = RESULTS["s6"]
if _s6c["corr_evaluated"] == 0 and _s6c["hall_evaluated"] == 0:
    if _s6c["corr_not_evaluated"] > 0 or _s6c["hall_not_evaluated"] > 0:
        _corr_part = _fmt_rf_judge(
            "correctness", None, 0, 0, _s6c["corr_not_evaluated"], _s6c["corr_mode"]
        )
        _hall_part = _fmt_rf_judge(
            "hallucination", None, 0, 0, _s6c["hall_not_evaluated"], _s6c["hall_mode"]
        )
        rf_judges_summary_line = (
            f"- **Reference-free SDK judges**: no live reference-free judge evaluations — all judge calls failed "
            f"({_corr_part}; {_hall_part})."
        )
    else:
        rf_judges_summary_line = (
            "- **Reference-free SDK judges**: no sessions matched `EVAL_FILTER` for reference-free SDK judges "
            f"(`correctness` **unavailable**, 0 sessions evaluated, `{_s6c['corr_mode']}`; "
            f"`hallucination` **unavailable**, 0 sessions evaluated, `{_s6c['hall_mode']}`)."
        )
else:
    _corr_part = _fmt_rf_judge(
        "correctness",
        _s6c.get("corr_pass_rate"),
        _s6c.get("corr_passed", 0),
        _s6c["corr_evaluated"],
        _s6c["corr_not_evaluated"],
        _s6c["corr_mode"],
    )
    _hall_part = _fmt_rf_judge(
        "hallucination",
        _s6c.get("hall_pass_rate"),
        _s6c.get("hall_passed", 0),
        _s6c["hall_evaluated"],
        _s6c["hall_not_evaluated"],
        _s6c["hall_mode"],
    )
    rf_judges_summary_line = f"- **Reference-free SDK judges**: {_corr_part}; {_hall_part}."

if ref_sid in twin_dict and hal_sid in twin_dict:
    ref_row = twin_dict[ref_sid]
    hal_row = twin_dict[hal_sid]
    # None / NaN scores mean the direct Vertex AI judge call failed (see pipeline_status / trial_status);
    # never compare or render them as 0.20 / 0.00 measured scores.
    pipe_ok = bool(pd.notna(ref_row.get("pipeline_score")) and pd.notna(hal_row.get("pipeline_score")))
    trial_ok = bool(pd.notna(ref_row.get("trial_pass@3")) and pd.notna(hal_row.get("trial_pass@3")))
    pipeline_separated = bool(pipe_ok and float(ref_row["pipeline_score"]) > float(hal_row["pipeline_score"]))
    trial_separated = bool(trial_ok and float(ref_row["trial_pass@3"]) > float(hal_row["trial_pass@3"]))
    _unavailable_str = "**unavailable (judge error)** — not a measured quality/consistency failure"
    _trial_unavailable_str = (
        "**unavailable (judge error)** — pass@3 / pass^3 need all 3 trials evaluated; not a measured quality/consistency failure"
    )
    pipe_str = (
        f"**{ref_row['pipeline_score']:.2f}** vs. **{hal_row['pipeline_score']:.2f}**" if pipe_ok else _unavailable_str
    )
    trial_str = (
        f"**{ref_row['trial_pass@3']:.2f}** vs. **{hal_row['trial_pass@3']:.2f}**" if trial_ok else _trial_unavailable_str
    )
    _unavailable_surfaces = [
        (name, key) for name, key, ok in (("`GraderPipeline`", "pipeline_status", pipe_ok), ("`TrialRunner`", "trial_status", trial_ok)) if not ok
    ]
    both_verified_mode = (
        ref_row.get("trial_gate_mode") == "verified_golden_answer"
        and hal_row.get("trial_gate_mode") == "verified_golden_answer"
    )
    if _unavailable_surfaces:
        _unavailable_detail = "; ".join(
            f"{name}: ref=`{ref_row.get(key, 'n/a')}`, target=`{hal_row.get(key, 'n/a')}`" for name, key in _unavailable_surfaces
        )
        if pipe_ok:
            _measured_note = (
                f" `GraderPipeline` {'separated' if pipeline_separated else 'did not separate'} the sessions on its measured scores."
            )
        elif trial_ok:
            _measured_note = (
                f" `TrialRunner` {'separated' if trial_separated else 'did not separate'} the sessions on its evaluated trials."
            )
        else:
            _measured_note = ""
        outcome_clause = (
            f"could not be compared on {' and '.join(name for name, _ in _unavailable_surfaces)} because the direct "
            f"Vertex AI judge calls failed ({_unavailable_detail}) — a judge availability gap, not a measured quality or "
            f"consistency failure; re-run Section 6B once Vertex AI quota recovers.{_measured_note}"
        )
    elif pipeline_separated and trial_separated:
        outcome_clause = "separated the accurate reference session from the lower-scoring comparison session."
    elif not both_verified_mode:
        pipe_note = (
            "`GraderPipeline` separated the sessions"
            if pipeline_separated
            else "`GraderPipeline` did not separate the sessions"
        )
        both_ref_free = (
            ref_row.get("trial_gate_mode") == "reference_free_sentiment"
            and hal_row.get("trial_gate_mode") == "reference_free_sentiment"
        )
        mode_note = (
            "`TrialRunner` used `reference_free_sentiment` mode on both sessions"
            if both_ref_free
            else "`TrialRunner` used `reference_free_sentiment` mode on one of the two sessions"
        )
        reasons = f"ref={ref_row.get('trial_gate_reason', 'n/a')}, target={hal_row.get('trial_gate_reason', 'n/a')}"
        outcome_clause = (
            f"recorded the comparison scores above ({pipe_note}; {mode_note} "
            f"[{reasons}] — verify in-domain golden rows in `bqca_golden_qa` with `verified = TRUE` to gate "
            "`TrialRunner` on `final_answer_correct`)."
        )
    else:
        pipe_note = (
            "`GraderPipeline` separated the sessions"
            if pipeline_separated
            else "`GraderPipeline` did not separate the sessions"
        )
        outcome_clause = f"recorded the comparison scores above ({pipe_note}; check Vertex AI direct judge logs if `TrialRunner` did not separate the sessions)."
    twin_summary_line = (
        f"- **Twin session comparison (`{ref_sid}` vs. `{hal_sid}`)**: "
        f"Golden SQL grading (`{ref_row['sql_golden_verdict']}` vs. `{hal_row['sql_golden_verdict']}`), "
        f"`GraderPipeline` ({pipe_str}), "
        f"and `TrialRunner pass@3` (`mode={hal_row.get('trial_gate_mode', 'n/a')}`, {trial_str}) "
        f"{outcome_clause}\n"
        f"{rf_judges_summary_line}"
    )
else:
    twin_summary_line = rf_judges_summary_line

emb_failed_note = f"; **{RESULTS['s6']['embedding_failed']}** session embedding failures" if RESULTS["s6"]["embedding_failed"] > 0 else ""
_failed_s6_list = RESULTS["s6"]["failed_sessions"]
_failed_s6_str = (
    ", ".join(f"`{s}`" for s in _failed_s6_list[:5])
    + (f", ... (+{len(_failed_s6_list) - 5} more)" if len(_failed_s6_list) > 5 else "")
) or "none"
_s6_mode_label = (
    "`AI.GENERATE` over Section 5.2 matches"
    if RESULTS["s6"].get("golden_grading_mode") != "api_fallback"
    else "`api_fallback` direct Vertex AI over Section 5.2 matches"
)
if not RESULTS["s6"]["s5_available"]:
    golden_takeaway_line = (
        "- **Golden Q&A SQL Grading (`AI.GENERATE` over Section 5.2 matches)**: "
        "unavailable (upstream Section 5 Golden Question Bank did not complete); "
        "reference-free SDK judges and direct evaluators below still ran."
    )
elif not RESULTS["s6"]["golden_grading_completed"]:
    _att_err_detail = (
        f"**0/{RESULTS['s6']['attempted']}** evaluated, **{RESULTS['s6']['error']} ERROR/unavailable** across attempted "
        f"in-domain single-turn session(s) (verified ground-truth: **0/{RESULTS['s6'].get('verified_attempted', 0)}** evaluated, "
        f"**{RESULTS['s6'].get('verified_error', 0)} ERROR**; unverified template: "
        f"**0/{RESULTS['s6'].get('template_attempted', 0)}** evaluated, **{RESULTS['s6'].get('template_error', 0)} ERROR**); "
        if RESULTS["s6"].get("attempted", 0) > 0
        else (
            f"**{RESULTS['s6']['in_domain_matched']}** in-domain single-turn session(s) within cosine distance `{golden_max_dist}` were not graded; "
        )
    )
    golden_takeaway_line = (
        "- **Golden Q&A SQL Grading (`AI.GENERATE` over Section 5.2 matches)**: "
        f"unavailable (Section 6A grading job did not complete — {_att_err_detail}"
        f"**{RESULTS['s6']['out_of_domain']}** out-of-domain single-turn session(s) above `{golden_max_dist}` "
        f"and **{RESULTS['s6']['unsampled_or_multiturn']}** multi-turn or unsampled session(s){emb_failed_note}); "
        "reference-free SDK judges and direct evaluators below still ran."
    )
else:
    _s6_fail_label = (
        "Mismatched vs. unverified template (pipeline smoke test, not an accuracy KPI)"
        if (RESULTS["s6"]["verified_graded"] == 0 and RESULTS["s6"]["template_graded"] > 0)
        else "Failing session(s)"
    )
    _ungraded_in_domain_note = (
        f"; **{RESULTS['s6']['ungraded_in_domain']}** in-domain single-turn session(s) ungraded"
        if RESULTS["s6"].get("ungraded_in_domain", 0) > 0
        else ""
    )
    _s6_eval_c = RESULTS["s6"].get("evaluated", RESULTS["s6"]["graded"])
    _s6_att_c = RESULTS["s6"].get("attempted", _s6_eval_c + RESULTS["s6"]["error"])
    _s6_head_verb = (
        f"Evaluated **{_s6_eval_c}** of **{_s6_att_c}** attempted in-domain single-turn sessions"
        if RESULTS["s6"]["error"] > 0
        else f"Graded **{_s6_eval_c}** in-domain single-turn sessions"
    )
    _ver_err_note = (
        f" [+{RESULTS['s6']['verified_error']} ERROR/unavailable]"
        if RESULTS["s6"].get("verified_error", 0) > 0
        else ""
    )
    _tpl_err_note = (
        f" [+{RESULTS['s6']['template_error']} ERROR/unavailable]"
        if RESULTS["s6"].get("template_error", 0) > 0
        else ""
    )
    golden_takeaway_line = (
        f"- **Golden Q&A SQL Grading ({_s6_mode_label})**: {_s6_head_verb} "
        f"(**{RESULTS['s6']['pass']} PASS**, **{RESULTS['s6']['fail']} FAIL**, **{RESULTS['s6']['error']} ERROR**; "
        f"verified ground-truth subset: **{RESULTS['s6']['verified_pass']} PASS / {RESULTS['s6']['verified_fail']} FAIL** across **{RESULTS['s6']['verified_graded']}** evaluated sessions{_ver_err_note}; "
        f"unverified template smoke-test subset: **{RESULTS['s6']['template_pass']} PASS / {RESULTS['s6']['template_fail']} FAIL** across **{RESULTS['s6']['template_graded']}** evaluated sessions{_tpl_err_note}; "
        f"**{RESULTS['s6']['out_of_domain']}** out-of-domain single-turn sessions skipped{_ungraded_in_domain_note}; "
        f"**{RESULTS['s6']['unsampled_or_multiturn']}** multi-turn or unsampled sessions excluded{emb_failed_note}). {_s6_fail_label}: {_failed_s6_str}."
    )
show(f"""**Section 6 Takeaway (live)**

{golden_takeaway_line}
{twin_summary_line}""")

## 7. FinOps, latency SLOs, thinking token accounting & Verified Queries fast-path ROI

BQCA logs per-call `usage_metadata` (`prompt_token_count`, `candidates_token_count`, `thoughts_token_count`, `cached_content_token_count`, `total_token_count`) and latency (`total_ms`, `time_to_first_token_ms`). This section runs two FinOps workflows:

1. **Automated SLO Budget Checks (`SystemEvaluator`)**: Evaluates `latency`, `token_efficiency`, `cost_per_session`, `context_cache_hit_rate`, `turn_count`, and `ttft` via pure BigQuery SQL.
2. **Token & Thinking-Token Cost Estimation + Fast-Path ROI**: `SystemEvaluator.cost_per_session()` prices only `prompt_token_count` and `candidates_token_count`. Because Gemini reasoning models also generate `thoughts_token_count` (billed at the output token rate under token-based pricing), the query below computes a token-equivalent cost estimate at your configured `COST_RATES` (defined in Cell `1.1`; default values are illustrative placeholder token rates, not Gemini Flash list prices) including thinking tokens, and compares **Verified Queries Fast-Path (`is_fast_path = TRUE`)** against standard NL2SQL turns.

In [ ]:
#@title 7.1 Automated Session SLO Checks (SystemEvaluator)
# Default illustrative SLO budgets (replace BUDGETS with your own production SLO targets):
BUDGETS = {
    "latency_ms": 3000,
    "total_tokens": 8000,
    "cost_usd": 0.02,
    "cache_hit_rate": 0.5,
    "turns": 3,
    "ttft_ms": 1000,
}

SYSTEM_SUITE = [
    ("latency (avg event ms)", "latency", SystemEvaluator.latency(threshold_ms=BUDGETS["latency_ms"]), BUDGETS["latency_ms"]),
    ("token_efficiency (max tokens)", "token_efficiency", SystemEvaluator.token_efficiency(max_tokens=BUDGETS["total_tokens"]), BUDGETS["total_tokens"]),
    ("cost_per_session (max USD)", "cost", SystemEvaluator.cost_per_session(
        max_cost_usd=BUDGETS["cost_usd"], input_cost_per_1k=COST_RATES["input_per_1k"], output_cost_per_1k=COST_RATES["output_per_1k"]
    ), BUDGETS["cost_usd"]),
    ("context_cache_hit_rate (min)", "context_cache_hit_rate", SystemEvaluator.context_cache_hit_rate(min_hit_rate=BUDGETS["cache_hit_rate"]), BUDGETS["cache_hit_rate"]),
    ("turn_count (max turns)", "turn_count", SystemEvaluator.turn_count(max_turns=BUDGETS["turns"]), BUDGETS["turns"]),
    ("ttft (avg ms)", "ttft", SystemEvaluator.ttft(threshold_ms=BUDGETS["ttft_ms"]), BUDGETS["ttft_ms"]),
]

sys_rows = []
with timed("SystemEvaluator SLO suite via client.evaluate"):
    for label, metric, evaluator, budget in SYSTEM_SUITE:
        rep = client.evaluate(evaluator, filters=EVAL_FILTER)
        failing = sorted(s.session_id for s in rep.session_scores if not s.passed)
        failing_preview = (
            ", ".join(failing[:5]) + (f", ... (+{len(failing) - 5} more)" if len(failing) > 5 else "")
        ) or "-"
        sys_rows.append({
            "metric": metric,
            "_failing_list": failing,
            "evaluator": label,
            "budget": f"{budget:g}",
            "sessions": len(rep.session_scores),
            "passed": len(rep.session_scores) - len(failing),
            "failed": len(failing),
            "failing sessions": failing_preview,
        })
sys_df = pd.DataFrame(sys_rows)
display(sys_df.drop(columns=["metric", "_failing_list"]))

In [ ]:
#@title 7.2 Thinking-Token Cost Accounting & Verified Queries Fast-Path ROI
finops_df = turns_df.copy()
rate_in, rate_out = COST_RATES["input_per_1k"], COST_RATES["output_per_1k"]
finops_df["cost_sdk_usd"] = finops_df.prompt_tokens / 1000.0 * rate_in + finops_df.candidate_tokens / 1000.0 * rate_out
finops_df["cost_incl_thinking_usd"] = finops_df.cost_sdk_usd + finops_df.thinking_tokens / 1000.0 * rate_out
finops_df["execution_path"] = _classify_session_execution_path(finops_df)

llm_turns_df = finops_df[finops_df.llm_calls > 0]
token_identity_ok = (
    llm_turns_df.prompt_tokens + llm_turns_df.candidate_tokens + llm_turns_df.thinking_tokens == llm_turns_df.total_tokens
)
print(
    f"Token accounting identity (prompt + candidates + thinking == total) holds on "
    f"{int(token_identity_ok.sum())}/{len(llm_turns_df)} LLM sessions "
    "(sessions with tool-use or system-prompt overhead tokens can have a small positive residual in total_token_count)."
)

finops_by_persona_df = (
    finops_df.groupby("persona")
    .agg(
        sessions=("session_id", "count"),
        llm_calls=("llm_calls", "sum"),
        prompt_tokens=("prompt_tokens", "sum"),
        candidate_tokens=("candidate_tokens", "sum"),
        thinking_tokens=("thinking_tokens", "sum"),
        cached_tokens=("cached_tokens", "sum"),
        total_tokens=("total_tokens", "sum"),
        cost_sdk_usd=("cost_sdk_usd", "sum"),
        cost_incl_thinking_usd=("cost_incl_thinking_usd", "sum"),
    )
    .round({"cost_sdk_usd": 4, "cost_incl_thinking_usd": 4})
    .reset_index()
)
denom_out = (finops_by_persona_df.candidate_tokens + finops_by_persona_df.thinking_tokens).where(
    (finops_by_persona_df.candidate_tokens + finops_by_persona_df.thinking_tokens) > 0
)
finops_by_persona_df["thinking_share_of_output"] = (finops_by_persona_df.thinking_tokens / denom_out).fillna(0.0).round(2)
print("Token and estimated cost breakdown by user cohort (comparing SDK cost vs. cost including thinking tokens):")
display(finops_by_persona_df)

INVOCATION_FINOPS_SQL = f"""
WITH inv_base AS (
  SELECT
    session_id,
    IFNULL(NULLIF(TRIM(invocation_id), ''), CAST(timestamp AS STRING)) AS inv_key,
    event_type,
    timestamp,
    attributes,
    latency_ms,
    IF(
      event_type = 'INVOCATION_COMPLETED',
      ROW_NUMBER() OVER (
        PARTITION BY
          session_id,
          event_type = 'INVOCATION_COMPLETED',
          IFNULL(NULLIF(TRIM(invocation_id), ''), CAST(timestamp AS STRING))
        ORDER BY
          (event_type = 'INVOCATION_COMPLETED') DESC,
          SAFE_CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64) DESC NULLS LAST,
          timestamp DESC
      ),
      NULL
    ) AS _inv_complete_rn
  FROM {T(COMPAT_VIEW_ID)}
  WHERE event_type IN ('INVOCATION_STARTING', 'USER_MESSAGE_RECEIVED', 'AGENT_RESPONSE', 'LLM_RESPONSE', 'INVOCATION_COMPLETED')
)
SELECT
  session_id,
  inv_key AS invocation_id,
  LOGICAL_OR(COALESCE(JSON_VALUE(attributes, '$.fast_path') = 'true', FALSE)) AS is_fast_path,
  SUM(IF(event_type = 'INVOCATION_COMPLETED' AND _inv_complete_rn = 1, SAFE_CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), NULL)) AS e2e_latency_ms,
  COUNTIF(event_type = 'LLM_RESPONSE') AS llm_calls,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.prompt_token_count') AS INT64), 0)), 0) AS prompt_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64), 0)), 0) AS candidate_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.thoughts_token_count') AS INT64), 0)), 0) AS thinking_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.cached_content_token_count') AS INT64), 0)), 0) AS cached_tokens,
  COALESCE(SUM(IF(event_type = 'LLM_RESPONSE', SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.total_token_count') AS INT64), 0)), 0) AS total_tokens
FROM inv_base
GROUP BY session_id, inv_key
HAVING COUNTIF(event_type IN ('USER_MESSAGE_RECEIVED', 'AGENT_RESPONSE', 'INVOCATION_COMPLETED', 'LLM_RESPONSE')) > 0
"""
inv_finops_df = globals().get("inv_finops_df")
if inv_finops_df is None and "run_sql" in globals():
    try:
        _cand_inv_df = run_sql(INVOCATION_FINOPS_SQL)
        if isinstance(_cand_inv_df, pd.DataFrame) and {"session_id", "invocation_id", "is_fast_path"}.issubset(_cand_inv_df.columns):
            if not finops_df.empty and "session_id" in finops_df.columns:
                _cand_inv_df = _cand_inv_df[_cand_inv_df["session_id"].isin(set(finops_df["session_id"]))].reset_index(drop=True)
            if not _cand_inv_df.empty:
                inv_finops_df = _cand_inv_df
    except Exception:
        inv_finops_df = None

roi_turns_df = _prepare_fast_path_roi_df(finops_df, inv_df=inv_finops_df, cost_rates=COST_RATES)
fast_path_cmp_df = (
    roi_turns_df.groupby("execution_path")
    .agg(
        sessions=("session_id", "count"),
        llm_calls=("llm_calls", "sum"),
        e2e_min_ms=("e2e_latency_ms", "min"),
        e2e_p50_ms=("e2e_latency_ms", "median"),
        e2e_p90_ms=("e2e_latency_ms", lambda s: s.quantile(0.9)),
        median_total_tokens=("total_tokens", "median"),
        median_cost_incl_thinking_usd=("cost_incl_thinking_usd", "median"),
    )
    .round({"e2e_min_ms": 0, "e2e_p50_ms": 0, "e2e_p90_ms": 0, "median_total_tokens": 0, "median_cost_incl_thinking_usd": 4})
    .reset_index()
)
print("Verified Queries Fast-Path vs. Standard NL2SQL comparison:")
display(fast_path_cmp_df)

sdk_total_cost = float(finops_df.cost_sdk_usd.sum())
true_total_cost = float(finops_df.cost_incl_thinking_usd.sum())
thinking_add_pct = 100.0 * (true_total_cost - sdk_total_cost) / max(sdk_total_cost, 1e-9)
prompt_sum = float(llm_turns_df.prompt_tokens.sum())
weighted_cache_hit = float(llm_turns_df.cached_tokens.sum() / prompt_sum) if prompt_sum > 0 else 0.0
fc_idx = fast_path_cmp_df.set_index("execution_path")
fast_n = int(fc_idx.loc["Fast Path (Verified Queries)", "sessions"]) if "Fast Path (Verified Queries)" in fc_idx.index else 0
std_n = int(fc_idx.loc["Standard NL2SQL", "sessions"]) if "Standard NL2SQL" in fc_idx.index else 0
mixed_n = (
    int(fc_idx.loc["Mixed Session (Fast Path + Standard NL2SQL)", "sessions"])
    if "Mixed Session (Fast Path + Standard NL2SQL)" in fc_idx.index
    else (
        int((finops_df["execution_path"] == "Mixed Session (Fast Path + Standard NL2SQL)").sum())
        if "finops_df" in locals() and isinstance(finops_df, pd.DataFrame) and "execution_path" in finops_df.columns
        else 0
    )
)
_fast_p50_raw = (
    fc_idx.loc["Fast Path (Verified Queries)", "e2e_p50_ms"]
    if (fast_n > 0 and "Fast Path (Verified Queries)" in fc_idx.index)
    else None
)
_std_p50_raw = (
    fc_idx.loc["Standard NL2SQL", "e2e_p50_ms"]
    if (std_n > 0 and "Standard NL2SQL" in fc_idx.index)
    else None
)
fast_p50_ms = float(_fast_p50_raw) if (_fast_p50_raw is not None and pd.notna(_fast_p50_raw)) else None
std_p50_ms = float(_std_p50_raw) if (_std_p50_raw is not None and pd.notna(_std_p50_raw)) else None
_fast_med_tokens_raw = (
    fc_idx.loc["Fast Path (Verified Queries)", "median_total_tokens"]
    if (fast_n > 0 and "Fast Path (Verified Queries)" in fc_idx.index)
    else None
)
fast_med_tokens = int(round(float(_fast_med_tokens_raw))) if (_fast_med_tokens_raw is not None and pd.notna(_fast_med_tokens_raw)) else None

sys_pass_map = {r.metric: (int(r.passed), int(r.sessions)) for r in sys_df.itertuples()}
failing_any_slo = sorted({s for sids in sys_df["_failing_list"] for s in sids})

RESULTS["s7"] = {
    "sys_pass": sys_pass_map,
    "failing_any": failing_any_slo,
    "identity_ok": int(token_identity_ok.sum()),
    "llm_sessions": len(llm_turns_df),
    "sdk_cost": sdk_total_cost,
    "true_cost": true_total_cost,
    "thinking_add_pct": thinking_add_pct,
    "weighted_cache_hit": weighted_cache_hit,
    "fast_n": fast_n,
    "std_n": std_n,
    "mixed_n": mixed_n,
    "fast_p50_ms": fast_p50_ms,
    "std_p50_ms": std_p50_ms,
    "fast_med_tokens": fast_med_tokens,
}
_s7 = RESULTS["s7"]
_s7_roi_line, _ = _format_fast_path_roi(_s7)
show(f"""**Section 7 Takeaway (live)**

- **SLO pass rates**: {'; '.join(f"`{k}` **{p}/{n}**" for k, (p, n) in RESULTS['s7']['sys_pass'].items())}. Sessions exceeding at least one budget: {(', '.join(f"`{s}`" for s in RESULTS['s7']['failing_any'][:5]) + (f' (+{len(RESULTS["s7"]["failing_any"]) - 5} more)' if len(RESULTS['s7']['failing_any']) > 5 else '')) or 'none'}.
- **Thinking-token cost accounting**: `prompt + candidates + thinking == total_token_count` holds exactly on **{RESULTS['s7']['identity_ok']}/{RESULTS['s7']['llm_sessions']}** LLM sessions (sessions with tool-use or system-prompt overhead tokens can have a small positive residual in `total_token_count`). Including `thoughts_token_count` raises estimated token-equivalent spend from **${RESULTS['s7']['sdk_cost']:.3f}** to **${RESULTS['s7']['true_cost']:.3f}** (**+{RESULTS['s7']['thinking_add_pct']:.0f}%** over the `cost_per_session` estimate); token-weighted context cache hit rate is **{RESULTS['s7']['weighted_cache_hit']:.2f}**.
{_s7_roi_line}""")

## 8. Operational health monitor (`finish_reason`, sanitised `*_ERROR` rows & polite refusals) + Executive Summary

In BQCA production logs, operational issues appear across three distinct signals:

1. **Model generation cut-offs (`LLM_RESPONSE.attributes.finish_reason`)**: Detects responses truncated by `MAX_TOKENS`, blocked by `SAFETY`, or missing `finish_reason` telemetry (`'(missing)'`) even when no `*_ERROR` event was emitted.
2. **Sanitised error events (`INVOCATION_ERROR`, `AGENT_ERROR`, `LLM_ERROR`)**: Query these directly with `bqaa.ERROR_SQL_PREDICATE` (`status = 'ERROR' OR error_message IS NOT NULL OR ENDS_WITH(event_type, '_ERROR')`). Do **not** rely on `SystemEvaluator.error_rate()` on BQCA logs, because `SystemEvaluator.error_rate()` only counts `TOOL_ERROR` events (which BQCA omits by design).
3. **Polite out-of-scope refusals (`AGENT_RESPONSE`)**: When a user asks an out-of-domain or policy-restricted question, the agent answers politely with `status = 'OK'` and `finish_reason = 'STOP'`. Use `evaluate_categorical()` to separate substantive answers from polite refusals, clarifying questions, and unanswered error turns.

In [ ]:
#@title 8.1 Model Finish Reasons & Sanitised Error Events (bqaa.ERROR_SQL_PREDICATE)
row_user_cohort_sql = (
    "COALESCE(IF(NULLIF(TRIM(user_id), '') IS NOT NULL, CONCAT('user_', SUBSTR(TO_HEX(SHA256(CAST(TRIM(user_id) AS BYTES))), 1, 8)), NULL), 'unattributed')"
    if ANONYMIZE_USER_ID
    else "COALESCE(NULLIF(SPLIT(TRIM(user_id), '@')[SAFE_OFFSET(0)], ''), 'unattributed')"
)
finish_df = run_sql(f"""
SELECT
  {row_user_cohort_sql} AS user_cohort,
  session_id,
  COALESCE(JSON_VALUE(attributes, '$.finish_reason'), '(missing)') AS finish_reason,
  SAFE_CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64) AS candidate_tokens
FROM {T(COMPAT_VIEW_ID)}
WHERE event_type = 'LLM_RESPONSE'
""")
finish_df["persona"] = finish_df["session_id"].map(PERSONA_BY_SESSION).fillna(finish_df["user_cohort"])
finish_df = finish_df[["persona", "session_id", "finish_reason", "candidate_tokens"]]

print("1. Model finish_reason distribution across all LLM_RESPONSE events:")
display(
    finish_df.groupby("finish_reason")
    .agg(llm_responses=("session_id", "count"), sessions=("session_id", "nunique"))
    .reset_index()
    .sort_values("llm_responses", ascending=False)
    .reset_index(drop=True)
)
non_stop_df = finish_df[
    (finish_df.finish_reason != "STOP") & (finish_df.finish_reason != "(missing)")
].sort_values(["finish_reason", "session_id"]).reset_index(drop=True)
missing_finish_df = finish_df[finish_df.finish_reason == "(missing)"].sort_values("session_id").reset_index(drop=True)
print(f"Sessions with a non-STOP model finish_reason ({len(non_stop_df)} total, showing up to 25):")
display(non_stop_df.head(25))
if not missing_finish_df.empty:
    print(
        f"Sessions with a missing finish_reason ({len(missing_finish_df)} total, showing up to 25; "
        "telemetry gap — not a model cut-off; excluded from the Section 8.2 attention queue):"
    )
    display(missing_finish_df.head(25))

errors_df = run_sql(f"""
SELECT
  {row_user_cohort_sql} AS user_cohort,
  session_id,
  event_type,
  status,
  content IS NULL AS content_is_null,
  error_message
FROM {T(COMPAT_VIEW_ID)}
WHERE {bqaa.ERROR_SQL_PREDICATE}
ORDER BY session_id, timestamp
""")
errors_df["persona"] = errors_df["session_id"].map(PERSONA_BY_SESSION).fillna(errors_df["user_cohort"])
errors_df = errors_df[["persona", "session_id", "event_type", "status", "content_is_null", "error_message"]]


def classify_error_message(msg):
    if msg is None or msg is pd.NA or (isinstance(msg, float) and pd.isna(msg)):
        msg = ""
    m = str(msg).lower()
    if "resource_exhausted" in m or "429" in m:
        return "Quota / Rate Limit (429)"
    if "query failed" in m:
        return "SQL Query Execution Error"
    return "Other Runtime Error"


errors_df["error_class"] = errors_df.error_message.fillna("").map(classify_error_message)
print(
    f"\n2. Sanitised error rows matched by bqaa.ERROR_SQL_PREDICATE "
    f"({len(errors_df)} rows across {errors_df.session_id.nunique()} session(s), showing up to 25):"
)
display(errors_df.head(25))

In [ ]:
#@title 8.2 Delivery Outcome Classification & Operational Attention Queue
outcome_metric = CategoricalMetricDefinition(
    name="delivery_outcome",
    definition="What the agent delivered to the user in this session.",
    categories=[
        CategoricalMetricCategory(
            name="Answered",
            definition="The agent delivered a substantive data answer to the user's question.",
        ),
        CategoricalMetricCategory(
            name="Refused_Out_of_Scope",
            definition="The agent politely declined because the request is outside the available dataset or data-governance policy.",
        ),
        CategoricalMetricCategory(
            name="Clarification_Requested",
            definition="The agent asked the user a clarifying question to disambiguate the request.",
        ),
        CategoricalMetricCategory(
            name="No_Answer_Delivered",
            definition="The session terminated with an error before any answer was delivered.",
        ),
    ],
)
outcome_cfg = CategoricalEvaluationConfig(
    metrics=[outcome_metric],
    endpoint=CATEGORICAL_MODEL,
    connection_id=VERTEX_CONNECTION_ID,
    include_justification=True,
)
with timed(f"evaluate_categorical: delivery_outcome ({CATEGORICAL_MODEL})"):
    outcome_report = client.evaluate_categorical(outcome_cfg, filters=EVAL_FILTER)
print("Execution mode:", outcome_report.details.get("execution_mode"))
outcome_df = cat_frame(outcome_report)

# Combine structured failure signals (has_error, non-STOP finish_reason) with text classification
non_stop_by_session = non_stop_df.groupby("session_id").finish_reason.agg(lambda s: ", ".join(sorted(set(s))))
ops_df = (
    turns_df[["session_id", "persona", "has_error", "error_messages"]]
    .merge(outcome_df[["session_id", "category", "justification"]], on="session_id", how="left")
)
_sampled_sids = set(EVAL_SESSION_IDS)
_valid_by_sid = outcome_df.set_index("session_id")["valid"] if "valid" in outcome_df.columns else pd.Series(dtype=bool)
ops_df["category"] = ops_df["category"].where(
    ops_df["category"].notna(),
    ops_df["session_id"].map(lambda s: "Not_Evaluated" if s in _sampled_sids else "Unsampled"),
)
ops_df["justification"] = ops_df["justification"].fillna("-")
outcome_mode = outcome_report.details.get("execution_mode", "unknown")
outcomes_unsampled = int((ops_df["category"] == "Unsampled").sum())
outcomes_not_evaluated = int(
    ((ops_df["category"] == "Not_Evaluated") | (ops_df["session_id"].map(_valid_by_sid) == False)).sum()
)
outcomes_sampled = len(ops_df) - outcomes_unsampled
print(
    f"Delivery outcomes evaluated on {outcomes_sampled - outcomes_not_evaluated}/{outcomes_sampled} sampled sessions"
    + (f" ({outcomes_not_evaluated} Not_Evaluated: judge errors)" if outcomes_not_evaluated else "")
    + (f"; {outcomes_unsampled} additional explorer sessions outside MAX_EVAL_SESSIONS={MAX_EVAL_SESSIONS} marked Unsampled" if outcomes_unsampled else "")
    + "."
)
ops_df["non_stop_finish"] = ops_df["session_id"].map(non_stop_by_session).fillna("-")
ops_df["has_structured_alert"] = ops_df["has_error"].astype(bool) | (ops_df["non_stop_finish"] != "-")

print("Delivery outcome cross-tabulated by user cohort:")
display(pd.crosstab(ops_df["persona"], ops_df["category"]))

attention_mask = (
    ops_df["category"].isin(["Refused_Out_of_Scope", "Clarification_Requested", "No_Answer_Delivered"])
    | ops_df["has_structured_alert"]
)
_attn_df = (
    ops_df[attention_mask][
        ["persona", "session_id", "category", "has_error", "non_stop_finish", "justification"]
    ]
    .sort_values(["category", "persona", "session_id"])
    .reset_index(drop=True)
)
print(
    "Operational attention queue — sessions with an error row, non-STOP finish_reason cut-off, refusal, or clarification "
    f"({len(_attn_df)} total, showing up to 25):"
)
display(_attn_df.head(25))

RESULTS["s8"] = {
    "finish_counts": finish_df.finish_reason.value_counts().to_dict(),
    "non_stop_sessions": dict(zip(non_stop_df.session_id, non_stop_df.finish_reason)),
    "missing_finish_sessions": dict(zip(missing_finish_df.session_id, missing_finish_df.finish_reason)),
    "error_rows": len(errors_df),
    "error_sessions": int(errors_df.session_id.nunique()),
    "all_error_content_null": bool(errors_df.content_is_null.all()) if not errors_df.empty else True,
    "outcomes": (
        outcome_df[outcome_df["valid"].fillna(False)].category.value_counts().to_dict()
        if not outcome_df.empty and "valid" in outcome_df.columns
        else outcome_df.category.value_counts().to_dict()
    ),
    "refusals": int(
        (
            outcome_df[outcome_df["valid"].fillna(False)].category == "Refused_Out_of_Scope"
            if not outcome_df.empty and "valid" in outcome_df.columns
            else (outcome_df.category == "Refused_Out_of_Scope")
        ).sum()
    ),
    "outcome_sessions": outcomes_sampled,
    "outcomes_not_evaluated": outcomes_not_evaluated,
    "outcomes_unsampled": outcomes_unsampled,
    "outcome_mode": outcome_mode,
}

### 8.3 Customer Production Health & Analytics Summary Table (Live)

The summary table below is computed dynamically from the live BigQuery queries and SDK evaluations executed in Sections 1–8.

In [ ]:
#@title 8.3 Customer Production Health & Analytics Summary Table
R = RESULTS
_ref_sid = globals().get("REFERENCE_SESSION", "session_lena_top_math_01")
_hal_sid = globals().get("HALLUCINATED_SESSION", "session_hallucinated_retrieval_demo")

# Row 1: Views & Schema Health
_s1 = R.get("s1")
_s2 = R.get("s2")
if not _s1:
    _s1_row_cell = "Unavailable (Section 1 did not complete)."
else:
    if _s2 and "sessions" in _s2:
        _s1_total_sess = int(R.get("s1_total_sessions", _s2["sessions"]))
        _s1_sess_str = (
            f"**{_s1_total_sess:,}** total sessions (**{_s2['sessions']}** loaded in explorer)"
            if _s1_total_sess > _s2["sessions"]
            else f"**{_s2['sessions']}** sessions"
        )
    else:
        _s1_sess_str = f"**{int(R.get('s1_total_sessions', 0)):,}** sessions"
    _s1_row_cell = (
        f"Schema `{_s1['schema_status']}` ({_s1['columns']} cols); **{_s1['total_events']:,}** events across {_s1_sess_str} "
        f"and **{_s1['active_event_types']}** BQCA event types; **{_s1['scalar_responses']}/{_s1['agent_response_rows']}** "
        f"`AGENT_RESPONSE` rows normalized; ping `{_s1['ping_verdict']}`."
    )

# Row 2: Conversation Explorer
if not _s2:
    _s2_row_cell = "Unavailable (Section 2 did not complete)."
else:
    _s2_row_cell = (
        f"**{_s2['answered']}/{_s2['sessions']}** sessions delivered an answer (including **{_s2['clarifying']}** clarifying question(s)); "
        f"**{_s2['fast_path']}** hit Verified Queries fast path; **{_s2['errors']}** ended in an error."
    )

# Row 3: Sentiment & UX Triage (the sentiment judge and the UX-tone rubric are rendered independently: one can succeed while the other fails)
_s3 = R.get("s3")
if not _s3:
    _s3_row_cell = "Unavailable (Section 3 did not complete)."
else:
    _s3_sessions = _s3["sessions"]
    _s3_sent_un = _s3.get("sent_not_evaluated", 0)
    _s3_sent_eval = _s3.get("sent_evaluated", _s3_sessions - _s3_sent_un)
    _s3_tone_total = _s3.get("tone_sessions", _s3_sessions)
    _s3_tone_un = _s3.get("tone_not_evaluated", 0)
    _s3_tone_eval = _s3.get("tone_evaluated", _s3_tone_total - _s3_tone_un)
    _s3_frust = _s3["frustrated_sessions"]
    _s3_frust_str = (
        ", ".join(_s3_frust[:5]) + (f", ... (+{len(_s3_frust) - 5} more)" if len(_s3_frust) > 5 else "")
    ) or "none"
    if _s3_sessions == 0 and _s3_tone_total == 0:
        _s3_row_cell = "No sessions matched `EVAL_FILTER` for sentiment/tone evaluation."
    else:
        if _s3_sent_eval > 0:
            _s3_sent_part = (
                f"Sentiment pass: **{_s3['passed']}/{_s3_sent_eval}** (**{_s3['passed'] / _s3_sent_eval:.0%}**) "
                f"at the `{JUDGE_THRESHOLD}` bar"
                + (f" (**{_s3_sent_un}** unevaluated: judge errors)" if _s3_sent_un else "")
            )
        elif _s3_sessions == 0:
            _s3_sent_part = "Sentiment: no sessions matched `EVAL_FILTER`"
        else:
            _s3_sent_part = (
                f"Sentiment: **unavailable** (**{_s3_sent_un}/{_s3_sessions}** judge call(s) failed, "
                f"mode=`{_s3.get('sent_mode', 'n/a')}`)"
            )
        if _s3_tone_eval > 0:
            _s3_tone_part = (
                f"UX rubric flagged **{len(_s3_frust)}/{_s3_tone_eval}** `Frustrated_Impatient` (`{_s3_frust_str}`) "
                f"and **{len(_s3['clarification_sessions'])}/{_s3_tone_eval}** `Clarification_Needed`"
                + (f" (**{_s3_tone_un}** unevaluated: judge errors)" if _s3_tone_un else "")
            )
        elif _s3_tone_total == 0:
            _s3_tone_part = "UX tone: no sessions matched `EVAL_FILTER`"
        else:
            _s3_tone_part = (
                f"UX tone: **unavailable** (**{_s3_tone_un}/{_s3_tone_total}** call(s) failed, "
                f"mode=`{_s3.get('tone_mode', 'n/a')}`)"
            )
        _s3_row_cell = f"{_s3_sent_part}; {_s3_tone_part}."

# Row 4: Question Topic Taxonomy
_s4 = R.get("s4")
if not _s4:
    _s4_row_cell = "Unavailable (Section 4 did not complete)."
else:
    s4_modes_ok = (_s4.get("classify_mode") == "ai_classify" and _s4.get("generate_mode") == "ai_generate")
    s4_pair_label = (
        "`AI.CLASSIFY` and `AI.GENERATE`"
        if s4_modes_ok
        else f"fast mode (`{_s4.get('classify_mode', 'n/a')}`) and rationale mode (`{_s4.get('generate_mode', 'n/a')}`)"
    )
    if _s4["sessions"] > 0 and _s4.get("evaluated", _s4["sessions"]) == 0:
        _s4_row_cell = (
            "no live topic evaluations — every session failed on at least one engine "
            f"(fast mode=`{_s4.get('classify_mode', 'n/a')}`, rationale mode=`{_s4.get('generate_mode', 'n/a')}`)."
        )
    else:
        _s4_row_cell = (
            f"{s4_pair_label} agreed on **{_s4['agree']}/{_s4.get('evaluated', _s4['sessions'])}** evaluated sessions"
            + (f" (**{_s4.get('not_evaluated', 0)}** unevaluated: judge errors)" if _s4.get("not_evaluated", 0) else "")
            + f" ({', '.join(f'{k}: {v}' for k, v in _s4['distribution'].items())})."
        )

# Row 5: Semantic Drift & Coverage
_s5 = R.get("s5")
if not _s5 or not _s5.get("available", True):
    _s5_row_cell = "Unavailable (Section 5 Golden Question Bank / drift analysis did not complete)."
else:
    _s51_failed_sem = _s5.get("s51_failed_semantic_rows")
    if _s5.get("coverage_is_semantic", True):
        _s51_excl_note = (
            f", **{_s51_failed_sem}** NULL/invalid distance row(s) excluded"
            if _s51_failed_sem
            else ""
        )
        _s5_coverage_cell = (
            f"**{_s5['coverage_pct']:.1f}%** Golden Question Bank coverage (**{_s5['covered_golden']}/{_s5['total_golden']}** covered{_s51_excl_note})"
        )
    else:
        _s51_fb_note = (
            f"SDK 5.1 failed semantic rows={_s51_failed_sem}"
            if _s51_failed_sem is not None
            else "SDK 5.1 per-row failed embedding count unknown (no `.status` in `feedback.py`)"
        )
        _s5_coverage_cell = (
            f"Golden Question Bank semantic coverage **unavailable** (keyword-fallback coverage **{_s5['coverage_pct']:.1f}%**, "
            f"**{_s5['covered_golden']}/{_s5['total_golden']}** covered by keyword overlap, method=`{_s5.get('coverage_method', 'unknown')}` "
            f"— not a cosine-distance measurement; {_s51_fb_note})"
        )
    if _s5.get("semantic_distances_ok", True):
        if _s5["top_drift_median"] == 0.0:
            _s5_drift_cell = (
                f"**{_s5['drifted_over_015']}** production sessions drifted beyond cosine distance `{DRIFT_THRESHOLD}` "
                "(all cohort medians `0.000`)"
            )
        else:
            _s5_drift_cell = (
                f"**{_s5['drifted_over_015']}** production sessions drifted beyond cosine distance `{DRIFT_THRESHOLD}` "
                f"(top drifted cohort: `{_s5['top_drift_persona']}`, median `{_s5['top_drift_median']:.3f}`)"
            )
    else:
        _s5_drift_cell = "no valid semantic distances (embedding failures); drift ranking unavailable"
    _s5_status_cell = (
        f"; ⚠️ status: `{_s5['drift_status']}` (prod_emb_failed={_s5.get('prod_emb_failed', 0)}, gold_emb_failed={_s5.get('gold_emb_failed', 0)})"
        if _s5.get("drift_status", "COMPLETE") != "COMPLETE"
        else ""
    )
    _s5_row_cell = _s5_coverage_cell + "; " + _s5_drift_cell + _s5_status_cell + "."

# Row 6: Response Quality & Hallucination Catch
_s6 = R.get("s6")
if not _s6:
    _s6_row_cell = "Unavailable (Section 6 did not complete)."
else:
    if _ref_sid in _s6.get("twin_df", {}) and _hal_sid in _s6.get("twin_df", {}):
        _ref = _s6["twin_df"][_ref_sid]
        _hal = _s6["twin_df"][_hal_sid]
        # None / NaN = direct Vertex AI judge call failed (pipeline_status / trial_status); never render as 0.20 / 0.00.
        _pipe_ok = bool(pd.notna(_ref.get("pipeline_score")) and pd.notna(_hal.get("pipeline_score")))
        _trial_ok = bool(pd.notna(_ref.get("trial_pass@3")) and pd.notna(_hal.get("trial_pass@3")))
        _pipe_sep = bool(_pipe_ok and float(_ref["pipeline_score"]) > float(_hal["pipeline_score"]))
        _trial_sep = bool(_trial_ok and float(_ref["trial_pass@3"]) > float(_hal["trial_pass@3"]))
        _unavail_str = "**unavailable (judge error)** — not a measured quality/consistency failure"
        _trial_unavail_str = (
            "**unavailable (judge error)** — pass@3 / pass^3 need all 3 trials evaluated; not a measured quality/consistency failure"
        )
        _pipe_str = f"**{_ref['pipeline_score']:.2f}** vs. **{_hal['pipeline_score']:.2f}**" if _pipe_ok else _unavail_str
        _trial_str = f"**{_ref['trial_pass@3']:.2f}** vs. **{_hal['trial_pass@3']:.2f}**" if _trial_ok else _trial_unavail_str
        _unavail_surfaces = [
            (n, k) for n, k, ok in (("`GraderPipeline`", "pipeline_status", _pipe_ok), ("`TrialRunner`", "trial_status", _trial_ok)) if not ok
        ]
        _both_ver = (
            _ref.get("trial_gate_mode") == "verified_golden_answer"
            and _hal.get("trial_gate_mode") == "verified_golden_answer"
        )
        if _unavail_surfaces:
            _unavail_detail = "; ".join(
                f"{n}: ref=`{_ref.get(k, 'n/a')}`, target=`{_hal.get(k, 'n/a')}`" for n, k in _unavail_surfaces
            )
            _sep_verb = (
                f"could not compare the twin sessions on {' and '.join(n for n, _ in _unavail_surfaces)} because the direct "
                f"Vertex AI judge calls failed ({_unavail_detail}) — a judge availability gap, not a measured quality/consistency failure"
            )
        elif _pipe_sep and _trial_sep:
            _sep_verb = "separated the twin sessions"
        elif not _both_ver:
            _both_rf = (
                _ref.get("trial_gate_mode") == "reference_free_sentiment"
                and _hal.get("trial_gate_mode") == "reference_free_sentiment"
            )
            _mode_str = "in `reference_free_sentiment` mode" if _both_rf else "with mixed gate modes"
            _sep_verb = (
                f"evaluated the twin sessions (`GraderPipeline` {'separated' if _pipe_sep else 'did not separate'} the sessions; "
                f"`TrialRunner` {_mode_str} [{_ref.get('trial_gate_reason', 'n/a')} / {_hal.get('trial_gate_reason', 'n/a')}])"
            )
        else:
            _sep_verb = "evaluated the twin sessions"
        def _fmt_s6_rf(label, rate, passed_c, eval_c, not_eval_c, mode_s):
            if eval_c > 0 and rate is not None:
                _p_str = f" (**{passed_c}/{eval_c}**)" if passed_c is not None else ""
                return (
                    f"`{label}` pass rate **{rate:.0%}**{_p_str}"
                    + (f" (**{not_eval_c}** unevaluated: judge error)" if not_eval_c > 0 else "")
                )
            if not_eval_c > 0:
                _m_str = f", mode=`{mode_s}`" if mode_s else ""
                return f"`{label}` **unavailable** (**{not_eval_c}** session(s) unevaluated: judge error{_m_str})"
            return f"`{label}` **unavailable** (0 sessions evaluated)"

        _c_eval = _s6.get("corr_evaluated", 1 if _s6.get("corr_pass_rate") is not None else 0)
        _h_eval = _s6.get("hall_evaluated", 1 if _s6.get("hall_pass_rate") is not None else 0)
        _c_part = _fmt_s6_rf(
            "correctness",
            _s6.get("corr_pass_rate") if _c_eval > 0 else None,
            _s6.get("corr_passed"),
            _c_eval,
            _s6.get("corr_not_evaluated", 0),
            _s6.get("corr_mode"),
        )
        _h_part = _fmt_s6_rf(
            "hallucination",
            _s6.get("hall_pass_rate") if _h_eval > 0 else None,
            _s6.get("hall_passed"),
            _h_eval,
            _s6.get("hall_not_evaluated", 0),
            _s6.get("hall_mode"),
        )
        twin_summary_cell = (
            f"; `GraderPipeline` ({_pipe_str}) "
            f"and `TrialRunner pass@3` ({_trial_str}) {_sep_verb}; SDK {_c_part}, {_h_part}."
        )
    else:
        if _s6.get("corr_evaluated", 1) == 0 and _s6.get("hall_evaluated", 1) == 0:
            if _s6.get("corr_not_evaluated", 0) > 0 or _s6.get("hall_not_evaluated", 0) > 0:
                twin_summary_cell = (
                    "; no live reference-free judge evaluations — all judge calls failed "
                    f"(`correctness` **unavailable**, **{_s6.get('corr_not_evaluated', 0)}** unevaluated: judge error, mode=`{_s6.get('corr_mode', 'n/a')}`; "
                    f"`hallucination` **unavailable**, **{_s6.get('hall_not_evaluated', 0)}** unevaluated: judge error, mode=`{_s6.get('hall_mode', 'n/a')}`)."
                )
            else:
                twin_summary_cell = "; no sessions matched `EVAL_FILTER` for reference-free SDK judges (`correctness` **unavailable**, `hallucination` **unavailable**)."
        else:
            def _fmt_s6_rf(label, rate, passed_c, eval_c, not_eval_c, mode_s):
                if eval_c > 0 and rate is not None:
                    _p_str = f" (**{passed_c}/{eval_c}**)" if passed_c is not None else ""
                    return (
                        f"`{label}` pass rate **{rate:.0%}**{_p_str}"
                        + (f" (**{not_eval_c}** unevaluated: judge error)" if not_eval_c > 0 else "")
                    )
                if not_eval_c > 0:
                    _m_str = f", mode=`{mode_s}`" if mode_s else ""
                    return f"`{label}` **unavailable** (**{not_eval_c}** session(s) unevaluated: judge error{_m_str})"
                return f"`{label}` **unavailable** (0 sessions evaluated)"

            _c_eval = _s6.get("corr_evaluated", 1 if _s6.get("corr_pass_rate") is not None else 0)
            _h_eval = _s6.get("hall_evaluated", 1 if _s6.get("hall_pass_rate") is not None else 0)
            _c_part = _fmt_s6_rf(
                "correctness",
                _s6.get("corr_pass_rate") if _c_eval > 0 else None,
                _s6.get("corr_passed"),
                _c_eval,
                _s6.get("corr_not_evaluated", 0),
                _s6.get("corr_mode"),
            )
            _h_part = _fmt_s6_rf(
                "hallucination",
                _s6.get("hall_pass_rate") if _h_eval > 0 else None,
                _s6.get("hall_passed"),
                _h_eval,
                _s6.get("hall_not_evaluated", 0),
                _s6.get("hall_mode"),
            )
            twin_summary_cell = f"; SDK {_c_part}, {_h_part}."
    s6_emb_note = f", **{_s6['embedding_failed']}** session embedding failed" if _s6.get("embedding_failed", 0) > 0 else ""
    if not _s5 or not _s5.get("available", True) or not _s6.get("s5_available", True):
        _s6_row_cell = (
            f"Golden Q&A grading: unavailable (Section 5 Golden Question Bank did not complete){twin_summary_cell}"
        )
    elif not _s6.get("golden_grading_completed", _s6.get("golden_grading_available", True)):
        _s6_max_dist = _s6.get("golden_max_dist", globals().get("GOLDEN_MATCH_MAX_DISTANCE", DRIFT_THRESHOLD))
        _s6_att_err_clause = (
            f"**0/{_s6['attempted']}** evaluated, **{_s6.get('error', 0)} ERROR/unavailable** across attempted in-domain session(s) "
            f"(verified: **0/{_s6.get('verified_attempted', 0)}** evaluated, **{_s6.get('verified_error', 0)} ERROR**), "
            if _s6.get("attempted", 0) > 0
            else f"**{_s6.get('in_domain_matched', 0)}** in-domain single-turn session(s) unevaluated, "
        )
        _s6_row_cell = (
            "Golden Q&A grading: unavailable (Section 6A grading job did not complete — "
            f"{_s6_att_err_clause}"
            f"**{_s6.get('out_of_domain', 0)}** out-of-domain single-turn session(s) above distance `{_s6_max_dist}`, "
            f"**{_s6.get('unsampled_or_multiturn', 0)}** multi-turn/unsampled excluded{s6_emb_note}){twin_summary_cell}"
        )
    else:
        _s6_err_n = int(_s6.get("error", 0) or 0)
        _s6_eval_n = int(_s6.get("evaluated", _s6["graded"]))
        _s6_att_n = int(_s6.get("attempted", _s6_eval_n + _s6_err_n))
        _s6_err_note = f", **{_s6_err_n} ERROR**" if _s6_err_n else ""
        _s6_ungraded_note = (
            f", **{_s6['ungraded_in_domain']}** in-domain ungraded"
            if _s6.get("ungraded_in_domain", 0) > 0
            else ""
        )
        _s6_mode_note = " via `api_fallback`" if _s6.get("golden_grading_mode") == "api_fallback" else ""
        _s6_failed_list = _s6.get("failed_sessions", [])
        _s6_failed_str = (
            ", ".join(_s6_failed_list[:5])
            + (f", ... (+{len(_s6_failed_list) - 5} more)" if len(_s6_failed_list) > 5 else "")
        ) or "none"
        _ver_err_n = int(_s6.get("verified_error", 0) or 0)
        _tpl_err_n = int(_s6.get("template_error", 0) or 0)
        if _s6["verified_graded"] == 0 and (_s6["template_graded"] > 0 or _tpl_err_n > 0) and _ver_err_n == 0:
            _tpl_err_sfx = f", **{_tpl_err_n} ERROR**" if _tpl_err_n else ""
            golden_subset_note = (
                f"unverified template smoke-test — not an accuracy KPI: "
                f"**{_s6['template_pass']} PASS / {_s6['template_fail']} FAIL**{_tpl_err_sfx}; "
                f"mismatched vs. template: `{_s6_failed_str}`"
            )
            _s6_catch_suffix = ""
        elif _s6["template_graded"] > 0 or _tpl_err_n > 0:
            _ver_err_sfx = f" [+{_ver_err_n} ERROR]" if _ver_err_n else ""
            _tpl_err_sfx = f" [+{_tpl_err_n} ERROR]" if _tpl_err_n else ""
            golden_subset_note = (
                f"verified ground-truth: **{_s6['verified_pass']} PASS / {_s6['verified_fail']} FAIL**{_ver_err_sfx}; "
                f"unverified template smoke-test: **{_s6['template_pass']} PASS / {_s6['template_fail']} FAIL**{_tpl_err_sfx}"
            )
            _s6_catch_suffix = f"; caught `{_s6_failed_str}`"
        elif _ver_err_n > 0:
            _ver_att_n = int(_s6.get("verified_attempted", _s6["verified_graded"] + _ver_err_n))
            golden_subset_note = (
                f"**{_s6['verified_graded']}** evaluated of **{_ver_att_n}** verified ground-truth "
                f"(**{_ver_err_n} ERROR/unavailable**)"
            )
            _s6_catch_suffix = f"; caught `{_s6_failed_str}`"
        else:
            golden_subset_note = f"all **{_s6['verified_graded']}** verified ground-truth"
            _s6_catch_suffix = f"; caught `{_s6_failed_str}`"
        _s6_denom_str = (
            f"**{_s6_eval_n}** evaluated (of **{_s6_att_n}** attempted) in-domain single-turn sessions"
            if _s6_err_n > 0
            else f"**{_s6_eval_n}** in-domain single-turn sessions"
        )
        _s6_row_cell = (
            f"Golden Q&A grading{_s6_mode_note}: **{_s6['pass']} PASS** and **{_s6['fail']} FAIL**{_s6_err_note} across "
            f"{_s6_denom_str} (**{_s6['out_of_domain']}** out-of-domain skipped{_s6_ungraded_note}, "
            f"**{_s6['unsampled_or_multiturn']}** multi-turn/unsampled excluded{s6_emb_note}; "
            f"{golden_subset_note}{_s6_catch_suffix}){twin_summary_cell}"
        )

# Row 7: FinOps, Thinking Tokens & Fast Path
_s7 = R.get("s7")
if not _s7:
    _s7_row_cell = "Unavailable (Section 7 did not complete)."
else:
    _, _s7_fast_str = _format_fast_path_roi(_s7)
    _s7_row_cell = (
        f"Latency SLO **{_s7['sys_pass']['latency'][0]}/{_s7['sys_pass']['latency'][1]}**, "
        f"token SLO **{_s7['sys_pass']['token_efficiency'][0]}/{_s7['sys_pass']['token_efficiency'][1]}**, "
        f"cache hit rate **{_s7['weighted_cache_hit']:.2f}**; including `thoughts_token_count` adds "
        f"**+{_s7['thinking_add_pct']:.0f}%** to SDK cost (**${_s7['sdk_cost']:.3f}** -> **${_s7['true_cost']:.3f}**); {_s7_fast_str}."
    )

# Row 8: Operational Health Monitor
_s8 = R.get("s8")
if not _s8:
    _s8_row_cell = "Unavailable (Section 8 did not complete)."
else:
    _non_stop_items = list(_s8["non_stop_sessions"].items())
    non_stop_str = (
        ", ".join(f"{k}: {v}" for k, v in _non_stop_items[:5])
        + (f", ... (+{len(_non_stop_items) - 5} more)" if len(_non_stop_items) > 5 else "")
    ) or "none"
    _missing_finish_keys = list(_s8.get("missing_finish_sessions", {}).keys())
    missing_finish_str = (
        ", ".join(_missing_finish_keys[:5])
        + (f", ... (+{len(_missing_finish_keys) - 5} more)" if len(_missing_finish_keys) > 5 else "")
    ) or "none"
    _s8_row_cell = (
        f"**{_s8['error_rows']}** sanitised error rows in **{_s8['error_sessions']}** session(s) "
        f"(`content IS NULL`: `{_s8['all_error_content_null']}`); **{len(_s8['non_stop_sessions'])}** non-STOP `finish_reason` "
        f"cut-off session(s) (`{non_stop_str}`); **{len(_s8.get('missing_finish_sessions', {}))}** session(s) missing `finish_reason` "
        f"telemetry (`{missing_finish_str}`); delivery outcomes evaluated on "
        f"**{_s8.get('outcome_sessions', 0) - _s8.get('outcomes_not_evaluated', 0)}/{_s8.get('outcome_sessions', 0)}** session(s); "
        f"**{_s8['refusals']}** polite out-of-scope refusal(s)."
    )

summary_rows = [
    ("1. Views & Schema Health", f"`{COMPAT_VIEW_ID}`, `{TURNS_VIEW_ID}`, `client.doctor()`", _s1_row_cell),
    ("2. Conversation Explorer", f"`{TURNS_VIEW_ID}` SQL", _s2_row_cell),
    ("3. Sentiment & UX Triage", f"`LLMAsJudge.sentiment()`, `evaluate_categorical()` (`{JUDGE_MODEL}`)", _s3_row_cell),
    ("4. Question Topic Taxonomy", f"`evaluate_categorical()` (`AI.CLASSIFY` & `AI.GENERATE`, `{CATEGORICAL_MODEL}`)", _s4_row_cell),
    ("5. Semantic Drift & Coverage", "`client.drift_detection()`, `AI.EMBED` + `ML.DISTANCE`", _s5_row_cell),
    ("6. Response Quality & Hallucination Catch", f"Golden Q&A `AI.GENERATE` grader (`{JUDGE_MODEL}`), `LLMAsJudge`, `GraderPipeline`, `TrialRunner`", _s6_row_cell),
    ("7. FinOps, Thinking Tokens & Fast Path", f"`SystemEvaluator.*`, `{TURNS_VIEW_ID}` token accounting", _s7_row_cell),
    ("8. Operational Health Monitor", "`finish_reason`, `bqaa.ERROR_SQL_PREDICATE`, `evaluate_categorical()`", _s8_row_cell),
]

show(
    "| Section & Workflow | SDK / BigQuery Surface | Live Findings on This Dataset |\n|---|---|---|\n"
    + "\n".join(f"| **{a}** | {b} | {c} |" for a, b, c in summary_rows)
)